<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/20_Final_Comparative_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# NOTEBOOK 20 — FINAL COMPARATIVE ANALYSIS
# SPP-GAN: Statistical-Guided Privacy-Preserving GAN
# SECTION 1 — HEADER & SCOPE
# ============================================================

# ------------------------------------------------------------
# 1. GOOGLE DRIVE AND IMPORTS
# ------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive")

import os
import re
import gc
import json
import hashlib
import warnings

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 2. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

RESULTS_ROOT = (
    PROJECT_ROOT / "results" / "notebooks"
)

NOTEBOOK_ID = "notebook_20"

NOTEBOOK_NAME = "Final Comparative Analysis"

NOTEBOOK_ROOT = RESULTS_ROOT / NOTEBOOK_ID

RESULTS_DIR = NOTEBOOK_ROOT / "results"
TABLES_DIR = NOTEBOOK_ROOT / "tables"
FIGURES_DIR = NOTEBOOK_ROOT / "figures"
METADATA_DIR = NOTEBOOK_ROOT / "metadata"
VALIDATION_DIR = NOTEBOOK_ROOT / "validation"
MANIFESTS_DIR = NOTEBOOK_ROOT / "manifests"

OUTPUT_DIRECTORIES = [
    RESULTS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    METADATA_DIR,
    VALIDATION_DIR,
    MANIFESTS_DIR
]

for directory in OUTPUT_DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. EXPERIMENTAL DESIGN
# ------------------------------------------------------------

DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

# Four SPP-GAN ablation variants.
ABLATION_VARIANTS = [
    "gan_only",
    "statistical_guidance_gan",
    "gan_dp",
    "full_sppgan"
]

# Comparison methods specified in the methodology.
BASELINE_MODELS = [
    "statistical_baseline",
    "tvae",
    "ctgan",
    "dp_ctgan",
    "spp_gan"
]

REPETITIONS = [1, 2, 3, 4, 5]

MASTER_SEED = 2025

REPETITION_SEEDS = {
    repetition: MASTER_SEED + 1000 + repetition
    for repetition in REPETITIONS
}

# The 60-record grid refers specifically to the
# four-variant ablation experiment.
EXPECTED_ABLATION_EXPERIMENTS = (
    len(DATASETS)
    * len(ABLATION_VARIANTS)
    * len(REPETITIONS)
)

KEY_COLUMNS = [
    "dataset",
    "model",
    "repetition"
]

ABLATION_KEY_COLUMNS = [
    "dataset",
    "variant",
    "repetition"
]

# ------------------------------------------------------------
# 4. COMPARISON FRAMEWORK
# ------------------------------------------------------------

COMPARISON_METHODS = (
    BASELINE_MODELS
    + ABLATION_VARIANTS
)

# Do not assume that all baselines have the same
# experiment grid as the four ablation variants.

EXPERIMENT_DESIGN = {

    "datasets": DATASETS,

    "ablation_variants": ABLATION_VARIANTS,

    "baseline_models": BASELINE_MODELS,

    "repetitions": REPETITIONS,

    "expected_ablation_experiments": (
        EXPECTED_ABLATION_EXPERIMENTS
    ),

    "ablation_experiment_formula": (
        "3 datasets x 4 variants x 5 repetitions"
    ),

    "comparison_methods": COMPARISON_METHODS

}

# ------------------------------------------------------------
# 5. METRIC GROUPS
# ------------------------------------------------------------

METRIC_GROUPS = {

    "fidelity": [
        "fidelity_score",
        "wasserstein_distance",
        "js_divergence",
        "mmd",
        "correlation_error",
        "marginal_error"
    ],

    "utility": [
        "utility_score",
        "accuracy",
        "macro_f1",
        "weighted_f1",
        "precision",
        "recall",
        "auc",
        "rmse",
        "mae"
    ],

    "empirical_privacy": [
        "privacy_score",
        "nearest_neighbor_disclosure",
        "reidentification_risk",
        "membership_inference_risk"
    ],

    "formal_dp": [
        "achieved_epsilon",
        "achieved_delta"
    ],

    "computational": [
        "runtime_seconds",
        "training_time_seconds",
        "inference_time_seconds",
        "memory_mb",
        "peak_memory_mb"
    ]

}

# ------------------------------------------------------------
# 6. METRIC DIRECTIONS
# ------------------------------------------------------------

# Methodology-defined distance/error measures:
# lower values indicate smaller discrepancies.

LOWER_IS_BETTER = [

    "wasserstein_distance",
    "js_divergence",
    "mmd",
    "correlation_error",
    "marginal_error",

    "achieved_epsilon",

    "nearest_neighbor_disclosure",
    "reidentification_risk",
    "membership_inference_risk",

    "runtime_seconds",
    "training_time_seconds",
    "inference_time_seconds",
    "memory_mb",
    "peak_memory_mb",

    "rmse",
    "mae"

]

# Utility metrics are higher-is-better.
HIGHER_IS_BETTER = [

    "utility_score",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "precision",
    "recall",
    "auc"

]

# Aggregate fidelity_score and privacy_score require
# confirmation of their mathematical definitions.
# Their directions are intentionally not assumed.

UNCONFIRMED_METRIC_DIRECTIONS = [
    "fidelity_score",
    "privacy_score"
]

METRIC_DIRECTIONS = {
    metric: "higher"
    for metric in HIGHER_IS_BETTER
}

METRIC_DIRECTIONS.update({
    metric: "lower"
    for metric in LOWER_IS_BETTER
})

# ------------------------------------------------------------
# 7. STATISTICAL CONFIGURATION
# ------------------------------------------------------------

ALPHA = 0.05

CONFIDENCE_LEVEL = 0.95

# These are reporting parameters, not a declaration
# that statistical tests have been successfully executed.

STATISTICAL_CONFIG = {

    "alpha": ALPHA,

    "confidence_level": CONFIDENCE_LEVEL,

    "significance_source": "notebook_19",

    "multiple_comparison_method": "Holm",

    "do_not_infer_missing_significance": True

}

# ------------------------------------------------------------
# 8. PRIVACY REPORTING BOUNDARY
# ------------------------------------------------------------

PRIVACY_REPORTING_CONFIG = {

    "formal_dp_source": "privacy_accountant_evidence",

    "empirical_privacy_source": "disclosure_attack_evaluation",

    "require_achieved_epsilon": True,

    "require_achieved_delta": True,

    "require_explicit_formal_dp_verification": True,

    "configured_dp_is_not_formal_proof": True,

    "empirical_privacy_is_not_formal_dp": True

}

# ------------------------------------------------------------
# 9. REPRODUCIBILITY CONFIGURATION
# ------------------------------------------------------------

ANALYSIS_CONFIG = {

    "notebook": NOTEBOOK_ID,

    "title": NOTEBOOK_NAME,

    "project_root": str(PROJECT_ROOT),

    "notebook_root": str(NOTEBOOK_ROOT),

    "datasets": DATASETS,

    "ablation_variants": ABLATION_VARIANTS,

    "baseline_models": BASELINE_MODELS,

    "comparison_methods": COMPARISON_METHODS,

    "repetitions": REPETITIONS,

    "master_seed": MASTER_SEED,

    "repetition_seeds": REPETITION_SEEDS,

    "expected_ablation_experiments": (
        EXPECTED_ABLATION_EXPERIMENTS
    ),

    "key_columns": KEY_COLUMNS,

    "ablation_key_columns": ABLATION_KEY_COLUMNS,

    "metric_groups": METRIC_GROUPS,

    "metric_directions": METRIC_DIRECTIONS,

    "unconfirmed_metric_directions": (
        UNCONFIRMED_METRIC_DIRECTIONS
    ),

    "statistical_config": STATISTICAL_CONFIG,

    "privacy_reporting_config": (
        PRIVACY_REPORTING_CONFIG
    ),

    "methodology_reference": (
        "SPP-GAN Methodology, Sections 3 and 4"
    ),

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat()

}

# ------------------------------------------------------------
# 10. SAVE CONFIGURATION
# ------------------------------------------------------------

CONFIG_PATH = (
    METADATA_DIR / "notebook_20_configuration.json"
)

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        ANALYSIS_CONFIG,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 11. SECTION 1 VALIDATION
# ------------------------------------------------------------

section_01_checks = {

    "project_root_exists": (
        PROJECT_ROOT.is_dir()
    ),

    "results_root_exists": (
        RESULTS_ROOT.is_dir()
    ),

    "notebook_root_exists": (
        NOTEBOOK_ROOT.is_dir()
    ),

    "all_output_directories_exist": all(
        directory.is_dir()
        for directory in OUTPUT_DIRECTORIES
    ),

    "three_datasets_registered": (
        len(DATASETS) == 3
    ),

    "four_ablation_variants_registered": (
        len(ABLATION_VARIANTS) == 4
    ),

    "five_repetitions_registered": (
        len(REPETITIONS) == 5
    ),

    "expected_ablation_grid_is_60": (
        EXPECTED_ABLATION_EXPERIMENTS == 60
    ),

    "repetition_seeds_valid": (
        list(REPETITION_SEEDS.values())
        == [3026, 3027, 3028, 3029, 3030]
    ),

    "configuration_saved": (
        CONFIG_PATH.is_file()
    )

}

section_01_validation_df = pd.DataFrame([

    {
        "section": "section_01",

        "check": check_name,

        "passed": bool(check_value),

        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat()

    }

    for check_name, check_value in section_01_checks.items()

])

SECTION_01_VALIDATION_PATH = (
    VALIDATION_DIR / "section_01_validation.csv"
)

section_01_validation_df.to_csv(
    SECTION_01_VALIDATION_PATH,
    index=False
)

section_01_passed = int(
    section_01_validation_df["passed"].sum()
)

section_01_failed = int(
    (~section_01_validation_df["passed"]).sum()
)

SECTION_01_STATUS = (
    "PASS"
    if section_01_failed == 0
    else "FAIL"
)

# ------------------------------------------------------------
# 12. SECTION 1 SUMMARY
# ------------------------------------------------------------

section_01_summary = {

    "section": "section_01",

    "section_name": "Header & Scope",

    "status": SECTION_01_STATUS,

    "datasets": len(DATASETS),

    "ablation_variants": len(ABLATION_VARIANTS),

    "baseline_models": len(BASELINE_MODELS),

    "repetitions": len(REPETITIONS),

    "expected_ablation_experiments": (
        EXPECTED_ABLATION_EXPERIMENTS
    ),

    "master_seed": MASTER_SEED,

    "repetition_seeds": list(
        REPETITION_SEEDS.values()
    ),

    "validation_passed": section_01_passed,

    "validation_failed": section_01_failed,

    "configuration_path": str(CONFIG_PATH),

    "validation_path": str(
        SECTION_01_VALIDATION_PATH
    ),

    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat()

}

SECTION_01_SUMMARY_PATH = (
    METADATA_DIR / "section_01_summary.json"
)

with open(
    SECTION_01_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_01_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 13. MEMORY CLEANUP
# ------------------------------------------------------------

gc.collect()

# ------------------------------------------------------------
# 14. COMPLETION REPORT
# ------------------------------------------------------------

print("=" * 75)

print("NOTEBOOK 20 — SECTION 1: HEADER & SCOPE")

print("=" * 75)

print("Project root:", PROJECT_ROOT)

print("Datasets:", len(DATASETS))

print("Ablation variants:", len(ABLATION_VARIANTS))

print("Baseline models:", len(BASELINE_MODELS))

print("Repetitions:", len(REPETITIONS))

print(
    "Expected ablation experiments:",
    EXPECTED_ABLATION_EXPERIMENTS
)

print("Master seed:", MASTER_SEED)

print(
    "Repetition seeds:",
    list(REPETITION_SEEDS.values())
)

print("-" * 75)

print("Validation passed:", section_01_passed)

print("Validation failed:", section_01_failed)

print("Section 1 status:", SECTION_01_STATUS)

print("-" * 75)

print("Configuration:", CONFIG_PATH)

print("Validation:", SECTION_01_VALIDATION_PATH)

print("Summary:", SECTION_01_SUMMARY_PATH)

print("=" * 75)

assert SECTION_01_STATUS == "PASS", (
    "Section 1 validation failed. Review the validation report."
)

Mounted at /content/drive
NOTEBOOK 20 — SECTION 1: HEADER & SCOPE
Project root: /content/drive/MyDrive/SPP_GAN_Research
Datasets: 3
Ablation variants: 4
Baseline models: 5
Repetitions: 5
Expected ablation experiments: 60
Master seed: 2025
Repetition seeds: [3026, 3027, 3028, 3029, 3030]
---------------------------------------------------------------------------
Validation passed: 10
Validation failed: 0
Section 1 status: PASS
---------------------------------------------------------------------------
Configuration: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/metadata/notebook_20_configuration.json
Validation: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_01_validation.csv
Summary: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/metadata/section_01_summary.json


In [2]:
# ============================================================
# SECTION 2 — LOAD ALL VALIDATED RESULTS
# ============================================================

# ------------------------------------------------------------
# 1. INITIALIZE ARTIFACT REGISTRIES
# ------------------------------------------------------------

artifact_records = []

artifact_inventory_records = []

artifact_exception_records = []

EXCLUDED_NOTEBOOKS = {
    "notebook_20"
}

# Avoid loading unrelated raw datasets and temporary files.
# Retain result-bearing CSV files from previous notebooks.

ALLOWED_RESULT_DIRECTORIES = {
    "results",
    "tables",
    "evaluation",
    "outputs",
    "artifacts"
}

# ------------------------------------------------------------
# 2. DISCOVER PREVIOUS NOTEBOOK CSV FILES
# ------------------------------------------------------------

csv_paths = sorted(
    RESULTS_ROOT.rglob("*.csv")
)

for csv_path in csv_paths:

    relative_parts = csv_path.relative_to(
        RESULTS_ROOT
    ).parts

    # Exclude Notebook 20 outputs.
    if any(
        part in EXCLUDED_NOTEBOOKS
        for part in relative_parts
    ):
        continue

    source_notebook = next(
        (
            part for part in relative_parts
            if re.fullmatch(r"notebook_\d+", part)
        ),
        "unknown"
    )

    # Identify the directory containing the CSV.
    parent_names = {
        part.lower()
        for part in relative_parts[:-1]
    }

    if not (
        parent_names & ALLOWED_RESULT_DIRECTORIES
    ):
        continue

    try:

        # ----------------------------------------------------
        # 3. FILE INTEGRITY CHECK
        # ----------------------------------------------------

        if csv_path.stat().st_size == 0:

            artifact_exception_records.append({

                "source_path": str(csv_path),

                "source_notebook": source_notebook,

                "filename": csv_path.name,

                "exception_type": "EMPTY_FILE",

                "exception_message": (
                    "CSV file contains no data."
                )

            })

            continue

        # ----------------------------------------------------
        # 4. LOAD CSV
        # ----------------------------------------------------

        frame = pd.read_csv(
            csv_path,
            low_memory=False
        )

        # Empty result tables are recorded, not treated
        # as valid empirical evidence.

        if frame.empty:

            artifact_exception_records.append({

                "source_path": str(csv_path),

                "source_notebook": source_notebook,

                "filename": csv_path.name,

                "exception_type": "EMPTY_DATAFRAME",

                "exception_message": (
                    "CSV contains headers but no records."
                )

            })

            continue

        # ----------------------------------------------------
        # 5. REGISTER SUCCESSFULLY LOADED ARTIFACT
        # ----------------------------------------------------

        artifact_records.append({

            "source_path": str(csv_path),

            "source_notebook": source_notebook,

            "filename": csv_path.name,

            "data": frame

        })

        artifact_inventory_records.append({

            "source_notebook": source_notebook,

            "filename": csv_path.name,

            "source_path": str(csv_path),

            "rows": len(frame),

            "columns": len(frame.columns),

            "size_bytes": csv_path.stat().st_size,

            "load_status": "LOADED"

        })

    except Exception as error:

        artifact_exception_records.append({

            "source_path": str(csv_path),

            "source_notebook": source_notebook,

            "filename": csv_path.name,

            "exception_type": type(error).__name__,

            "exception_message": str(error)[:500]

        })

# ------------------------------------------------------------
# 6. CREATE ARTIFACT INVENTORY
# ------------------------------------------------------------

artifact_inventory = pd.DataFrame(
    artifact_inventory_records,
    columns=[
        "source_notebook",
        "filename",
        "source_path",
        "rows",
        "columns",
        "size_bytes",
        "load_status"
    ]
)

artifact_inventory_path = (
    TABLES_DIR / "source_artifact_inventory.csv"
)

artifact_inventory.to_csv(
    artifact_inventory_path,
    index=False
)

# ------------------------------------------------------------
# 7. SAVE LOAD EXCEPTION REPORT
# ------------------------------------------------------------

artifact_exceptions_df = pd.DataFrame(
    artifact_exception_records,
    columns=[
        "source_path",
        "source_notebook",
        "filename",
        "exception_type",
        "exception_message"
    ]
)

artifact_exceptions_path = (
    VALIDATION_DIR / "section_02_load_exceptions.csv"
)

artifact_exceptions_df.to_csv(
    artifact_exceptions_path,
    index=False
)

# ------------------------------------------------------------
# 8. SECTION 2 VALIDATION
# ------------------------------------------------------------

section_02_checks = {

    "artifact_inventory_created": (
        artifact_inventory_path.is_file()
    ),

    "load_exception_report_created": (
        artifact_exceptions_path.is_file()
    ),

    "notebook_20_excluded": (
        not any(
            "notebook_20" in Path(
                item["source_path"]
            ).parts
            for item in artifact_records
        )
    ),

    "all_loaded_artifacts_have_data": all(
        not item["data"].empty
        for item in artifact_records
    ),

    "inventory_matches_loaded_artifacts": (
        len(artifact_inventory) == len(artifact_records)
    ),

    "source_paths_preserved": (
        artifact_inventory["source_path"].notna().all()
        if not artifact_inventory.empty
        else True
    )

}

section_02_validation_df = pd.DataFrame([

    {
        "section": "section_02",

        "check": check_name,

        "passed": bool(check_value),

        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat()

    }

    for check_name, check_value in section_02_checks.items()

])

section_02_validation_path = (
    VALIDATION_DIR / "section_02_validation.csv"
)

section_02_validation_df.to_csv(
    section_02_validation_path,
    index=False
)

section_02_passed = int(
    section_02_validation_df["passed"].sum()
)

section_02_failed = int(
    (~section_02_validation_df["passed"]).sum()
)

SECTION_02_STATUS = (
    "PASS"
    if section_02_failed == 0
    else "FAIL"
)

# ------------------------------------------------------------
# 9. SECTION 2 SUMMARY
# ------------------------------------------------------------

section_02_summary = {

    "section": "section_02",

    "section_name": "Load All Validated Results",

    "status": SECTION_02_STATUS,

    "csv_files_discovered": len(csv_paths),

    "loaded_artifacts": len(artifact_records),

    "empty_or_unreadable_artifacts": len(
        artifact_exceptions_df
    ),

    "source_notebooks": sorted(
        artifact_inventory["source_notebook"].unique().tolist()
    ) if not artifact_inventory.empty else [],

    "loaded_rows": int(
        artifact_inventory["rows"].sum()
    ) if not artifact_inventory.empty else 0,

    "validation_passed": section_02_passed,

    "validation_failed": section_02_failed,

    "inventory_path": str(artifact_inventory_path),

    "exceptions_path": str(artifact_exceptions_path),

    "validation_path": str(section_02_validation_path),

    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat()

}

section_02_summary_path = (
    METADATA_DIR / "section_02_summary.json"
)

with open(
    section_02_summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_02_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 10. MEMORY CLEANUP
# ------------------------------------------------------------

gc.collect()

# ------------------------------------------------------------
# 11. COMPLETION REPORT
# ------------------------------------------------------------

print("=" * 75)

print("NOTEBOOK 20 — SECTION 2: LOAD ALL VALIDATED RESULTS")

print("=" * 75)

print("CSV files discovered:", len(csv_paths))

print("Artifacts loaded:", len(artifact_records))

print(
    "Empty/unreadable artifacts:",
    len(artifact_exceptions_df)
)

print(
    "Source notebooks:",
    section_02_summary["source_notebooks"]
)

print(
    "Total loaded rows:",
    section_02_summary["loaded_rows"]
)

print("-" * 75)

print("Validation passed:", section_02_passed)

print("Validation failed:", section_02_failed)

print("Section 2 status:", SECTION_02_STATUS)

print("-" * 75)

print("Inventory:", artifact_inventory_path)

print("Exceptions:", artifact_exceptions_path)

print("Validation:", section_02_validation_path)

print("Summary:", section_02_summary_path)

print("=" * 75)

assert SECTION_02_STATUS == "PASS", (
    "Section 2 validation failed."
)


NOTEBOOK 20 — SECTION 2: LOAD ALL VALIDATED RESULTS
CSV files discovered: 431
Artifacts loaded: 50
Empty/unreadable artifacts: 18
Source notebooks: ['notebook_15', 'notebook_16', 'notebook_17', 'notebook_18', 'notebook_19']
Total loaded rows: 1924
---------------------------------------------------------------------------
Validation passed: 6
Validation failed: 0
Section 2 status: PASS
---------------------------------------------------------------------------
Inventory: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/source_artifact_inventory.csv
Exceptions: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_02_load_exceptions.csv
Validation: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_02_validation.csv
Summary: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/metadata/section_02_summary.json


In [7]:

# ============================================================
# SECTION 3 — VALIDATE RESULT SCHEMAS
# ============================================================

# ------------------------------------------------------------
# 3.1 COLUMN NORMALIZATION
# ------------------------------------------------------------

def normalize_column_name(value):
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def normalize_dataframe(frame):
    result = frame.copy()
    result.columns = [
        normalize_column_name(column)
        for column in result.columns
    ]

    # Prevent duplicate column names after normalization.
    if result.columns.duplicated().any():
        duplicated = result.columns[
            result.columns.duplicated()
        ].tolist()
        raise ValueError(
            f"Duplicate columns after normalization: {duplicated}"
        )

    return result


# ------------------------------------------------------------
# 3.2 COLUMN ALIASES
# Keep model and ablation variant separate.
# ------------------------------------------------------------

COLUMN_ALIASES = {
    "dataset": [
        "dataset",
        "dataset_name"
    ],
    "model": [
        "model",
        "method",
        "generator",
        "algorithm",
        "model_name",
        "baseline_model"
    ],
    "variant": [
        "variant",
        "ablation_variant",
        "experiment_variant"
    ],
    "repetition": [
        "repetition",
        "repetition_id",
        "run",
        "run_id",
        "repeat",
        "seed_id"
    ],
    "achieved_epsilon": [
        "achieved_epsilon",
        "epsilon_achieved",
        "actual_epsilon"
    ],
    "achieved_delta": [
        "achieved_delta",
        "delta_achieved",
        "actual_delta"
    ],
    "runtime_seconds": [
        "runtime_seconds",
        "training_time_seconds",
        "execution_time_seconds",
        "runtime"
    ],
    "metric": [
        "metric",
        "metric_name",
        "measure"
    ],
    "value": [
        "value",
        "metric_value",
        "score"
    ],
    "status": [
        "status",
        "experiment_status",
        "validation_status"
    ]
}


def apply_aliases(frame):
    result = normalize_dataframe(frame)

    for canonical, aliases in COLUMN_ALIASES.items():

        if canonical in result.columns:
            continue

        matching_aliases = [
            alias for alias in aliases
            if alias in result.columns
        ]

        if matching_aliases:
            result = result.rename(
                columns={matching_aliases[0]: canonical}
            )

    return result


# ------------------------------------------------------------
# 3.3 SCHEMA CLASSIFICATION
# ------------------------------------------------------------

def classify_schema(columns):
    columns = set(columns)

    has_dataset = "dataset" in columns
    has_model = "model" in columns
    has_variant = "variant" in columns
    has_repetition = "repetition" in columns

    # Experiment-level records.
    if (
        has_dataset
        and has_variant
        and has_repetition
    ):
        return "ablation_experiment"

    if (
        has_dataset
        and has_model
        and has_repetition
    ):
        return "model_experiment"

    # Metric records.
    if (
        "metric" in columns
        and "value" in columns
    ):
        return "metric_value"

    # Privacy metadata.
    if (
        "achieved_epsilon" in columns
        or "achieved_delta" in columns
    ):
        return "privacy_metadata"

    # Dataset-level and supporting records.
    if has_dataset:
        return "dataset_level"

    return "supporting_or_unclassified"


# ------------------------------------------------------------
# 3.4 NORMALIZE AND INSPECT ALL ARTIFACTS
# ------------------------------------------------------------

schema_records = []

for item in artifact_records:

    frame = apply_aliases(item["data"])

    # Preserve normalized dataframe for later sections.
    item["normalized_data"] = frame

    columns = set(frame.columns)

    has_dataset = "dataset" in columns
    has_model = "model" in columns
    has_variant = "variant" in columns
    has_repetition = "repetition" in columns

    schema_type = classify_schema(columns)

    schema_records.append({

        "source_path": item.get("source_path"),
        "source_notebook": item.get("source_notebook"),
        "filename": item.get("filename"),

        "rows": int(len(frame)),
        "column_count": int(len(frame.columns)),

        "has_dataset": has_dataset,
        "has_model": has_model,
        "has_variant": has_variant,
        "has_repetition": has_repetition,

        "has_model_experiment_keys": (
            has_dataset
            and has_model
            and has_repetition
        ),

        "has_ablation_experiment_keys": (
            has_dataset
            and has_variant
            and has_repetition
        ),

        "has_metric_value": (
            "metric" in columns
            and "value" in columns
        ),

        "has_achieved_epsilon": (
            "achieved_epsilon" in columns
        ),

        "has_achieved_delta": (
            "achieved_delta" in columns
        ),

        "schema_type": schema_type,

        "columns": "|".join(frame.columns)

    })


schema_validation_df = pd.DataFrame(schema_records)


# ------------------------------------------------------------
# 3.5 SCHEMA CLASSIFICATION SUMMARY
# ------------------------------------------------------------

schema_type_summary = (
    schema_validation_df
    .groupby("schema_type", dropna=False)
    .size()
    .reset_index(name="artifact_count")
)


# ------------------------------------------------------------
# 3.6 EXPERIMENT SCHEMA COVERAGE
# ------------------------------------------------------------

model_experiment_count = int(
    schema_validation_df[
        "has_model_experiment_keys"
    ].sum()
)

ablation_experiment_count = int(
    schema_validation_df[
        "has_ablation_experiment_keys"
    ].sum()
)

dataset_level_count = int(
    (
        schema_validation_df["schema_type"]
        == "dataset_level"
    ).sum()
)

metric_value_count = int(
    (
        schema_validation_df["schema_type"]
        == "metric_value"
    ).sum()
)

privacy_metadata_count = int(
    (
        schema_validation_df["schema_type"]
        == "privacy_metadata"
    ).sum()
)

supporting_count = int(
    (
        schema_validation_df["schema_type"]
        == "supporting_or_unclassified"
    ).sum()
)


# ------------------------------------------------------------
# 3.7 MANUAL SCHEMA REVIEW
# ------------------------------------------------------------

schema_review_df = schema_validation_df[
    schema_validation_df["schema_type"]
    == "supporting_or_unclassified"
].copy()

# Review is informational. Supporting artifacts are not
# automatically considered invalid.


# ------------------------------------------------------------
# 3.8 SAVE SCHEMA REPORTS
# ------------------------------------------------------------

schema_validation_path = (
    VALIDATION_DIR
    / "section_03_schema_validation.csv"
)

schema_summary_path = (
    VALIDATION_DIR
    / "section_03_schema_summary.csv"
)

schema_review_path = (
    VALIDATION_DIR
    / "section_03_schema_review.csv"
)

schema_validation_df.to_csv(
    schema_validation_path,
    index=False
)

schema_type_summary.to_csv(
    schema_summary_path,
    index=False
)

schema_review_df.to_csv(
    schema_review_path,
    index=False
)


# ------------------------------------------------------------
# 3.9 VALIDATION CHECKS
# ------------------------------------------------------------

section_03_checks = [

    (
        "All loaded artifacts inspected",
        len(schema_validation_df) == len(artifact_records)
    ),

    (
        "Normalized data retained for every artifact",
        all(
            "normalized_data" in item
            for item in artifact_records
        )
    ),

    (
        "Source paths preserved",
        (
            schema_validation_df["source_path"].notna().all()
            if len(schema_validation_df) > 0
            else len(artifact_records) == 0
        )
    ),

    (
        "Schema classifications assigned",
        (
            schema_validation_df["schema_type"].notna().all()
            if len(schema_validation_df) > 0
            else len(artifact_records) == 0
        )
    ),

    (
        "Model and variant dimensions tracked separately",
        (
            "has_model" in schema_validation_df.columns
            and "has_variant" in schema_validation_df.columns
            and "model" in COLUMN_ALIASES
            and "variant" in COLUMN_ALIASES
        )
    ),

    (
        "Schema validation report saved",
        schema_validation_path.exists()
    ),

    (
        "Schema summary report saved",
        schema_summary_path.exists()
    ),

    (
        "Schema review report saved",
        schema_review_path.exists()
    ),

    (
        "No empty normalized dataframes",
        all(
            not item["normalized_data"].empty
            for item in artifact_records
        )
    ),

    (
        "Schema classification counts reconcile",
        int(schema_type_summary["artifact_count"].sum())
        == len(schema_validation_df)
    ),

    (
        "Experiment key flags are consistent",
        (
            (
                schema_validation_df[
                    "has_model_experiment_keys"
                ]
                ==
                (
                    schema_validation_df["has_dataset"]
                    & schema_validation_df["has_model"]
                    & schema_validation_df["has_repetition"]
                )
            ).all()
            and
            (
                schema_validation_df[
                    "has_ablation_experiment_keys"
                ]
                ==
                (
                    schema_validation_df["has_dataset"]
                    & schema_validation_df["has_variant"]
                    & schema_validation_df["has_repetition"]
                )
            ).all()
        )
    )

]


section_03_validation_df = pd.DataFrame(
    section_03_checks,
    columns=["check", "passed"]
)

section_03_validation_df["passed"] = (
    section_03_validation_df["passed"].astype(bool)
)

section_03_validation_df["status"] = (
    section_03_validation_df["passed"]
    .map({
        True: "PASS",
        False: "FAIL"
    })
)

section_03_validation_path = (
    VALIDATION_DIR / "section_03_validation.csv"
)

section_03_validation_df.to_csv(
    section_03_validation_path,
    index=False
)

passed_checks = int(
    section_03_validation_df["passed"].sum()
)

failed_checks = (
    len(section_03_validation_df) - passed_checks
)


# ------------------------------------------------------------
# 3.10 SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_03_summary = {

    "section": 3,

    "section_name": "Validate Result Schemas",

    "artifacts_checked": int(
        len(schema_validation_df)
    ),

    "model_experiment_artifacts": (
        model_experiment_count
    ),

    "ablation_experiment_artifacts": (
        ablation_experiment_count
    ),

    "dataset_level_artifacts": (
        dataset_level_count
    ),

    "metric_value_artifacts": (
        metric_value_count
    ),

    "privacy_metadata_artifacts": (
        privacy_metadata_count
    ),

    "supporting_or_unclassified_artifacts": (
        supporting_count
    ),

    "artifacts_requiring_manual_review": (
        int(len(schema_review_df))
    ),

    "validation_passed": passed_checks,

    "validation_failed": failed_checks,

    "status": (
        "PASS"
        if failed_checks == 0
        else "FAILED"
    )
}

section_03_summary_path = (
    METADATA_DIR / "section_03_summary.json"
)

with open(section_03_summary_path, "w") as f:
    json.dump(
        section_03_summary,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 3.11 FINAL OUTPUT
# ------------------------------------------------------------

print("=" * 65)
print("NOTEBOOK 20 — SECTION 3: VALIDATE RESULT SCHEMAS")
print("=" * 65)

print(
    "Artifacts checked:",
    len(schema_validation_df)
)

print("\nSchema classification:")

print(
    schema_type_summary.to_string(index=False)
)

print(
    "\nArtifacts with model experiment keys:",
    model_experiment_count
)

print(
    "Artifacts with ablation experiment keys:",
    ablation_experiment_count
)

print(
    "Dataset-level artifacts:",
    dataset_level_count
)

print(
    "Metric-value artifacts:",
    metric_value_count
)

print(
    "Privacy metadata artifacts:",
    privacy_metadata_count
)

print(
    "Artifacts requiring manual review:",
    len(schema_review_df)
)

print("\nValidation passed:", passed_checks)
print("Validation failed:", failed_checks)

print(
    "Section 3 status:",
    section_03_summary["status"]
)

print("\nSaved artifacts:")

print(schema_validation_path)
print(schema_summary_path)
print(schema_review_path)
print(section_03_validation_path)
print(section_03_summary_path)

assert failed_checks == 0, (
    "Section 3 validation failed. "
    "Inspect section_03_validation.csv."
)


NOTEBOOK 20 — SECTION 3: VALIDATE RESULT SCHEMAS
Artifacts checked: 50

Schema classification:
               schema_type  artifact_count
       ablation_experiment               8
             dataset_level              29
supporting_or_unclassified              13

Artifacts with model experiment keys: 0
Artifacts with ablation experiment keys: 8
Dataset-level artifacts: 29
Metric-value artifacts: 0
Privacy metadata artifacts: 0
Artifacts requiring manual review: 13

Validation passed: 11
Validation failed: 0
Section 3 status: PASS

Saved artifacts:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_03_schema_validation.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_03_schema_summary.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_03_schema_review.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_03_validation.csv


In [13]:

# ============================================================
# SECTION 4 — INTEGRATE FIDELITY RESULTS
# ============================================================

# ------------------------------------------------------------
# 4.1 FIDELITY METRIC REGISTRY
# ------------------------------------------------------------

FIDELITY_METRIC_ALIASES = {
    "wasserstein": "wasserstein_distance",
    "wasserstein_distance": "wasserstein_distance",
    "norm_wasserstein_distance": "norm_wasserstein_distance",

    "js": "js_divergence",
    "js_divergence": "js_divergence",
    "norm_js_divergence": "norm_js_divergence",

    "mmd": "mmd",
    "norm_mmd": "norm_mmd",

    "correlation_error": "correlation_error",
    "norm_correlation_error": "norm_correlation_error",

    "ks_statistic": "ks_statistic",
    "norm_ks_statistic": "norm_ks_statistic",

    "ks_distance": "ks_distance",
    "norm_ks_distance": "norm_ks_distance",

    "marginal_distance": "marginal_distance",
    "dependency_error": "dependency_error",
    "categorical_divergence": "categorical_divergence",

    "fidelity_score": "fidelity_score",
    "norm_fidelity_score": "norm_fidelity_score",

    "ks_similarity": "ks_similarity",
    "norm_ks_similarity": "norm_ks_similarity",

    "categorical_similarity": "categorical_similarity",
    "norm_categorical_similarity": "norm_categorical_similarity"
}

FIDELITY_OUTPUT_COLUMNS = [
    "dataset",
    "model",
    "variant",
    "repetition",
    "metric",
    "value",
    "metric_group",
    "experiment_type",
    "source_path",
    "source_notebook",
    "source_filename"
]


# ------------------------------------------------------------
# 4.2 IDENTIFY PRIMARY FIDELITY SOURCES
# ------------------------------------------------------------

def is_primary_fidelity_source(item):

    path_text = str(
        item.get("source_path", "")
    ).replace("\\", "/").lower()

    filename = str(
        item.get("filename", "")
    ).lower()

    # Primary empirical evidence is expected from Notebook 18.
    # Notebook 19 contains downstream statistical outputs,
    # not primary fidelity measurements.
    return (
        "notebook_18" in path_text
        and "/results/" in path_text
        and any(
            token in filename
            for token in [
                "aligned_experimental_conditions",
                "normalized_metrics",
                "tradeoff_measures"
            ]
        )
    )


# ------------------------------------------------------------
# 4.3 NORMALIZE EXPERIMENT KEYS
# ------------------------------------------------------------

def prepare_fidelity_frame(frame):

    result = apply_aliases(frame)

    for column in ["dataset", "model", "variant"]:

        if column in result.columns:

            result[column] = (
                result[column]
                .astype("string")
                .str.strip()
            )

    if "repetition" in result.columns:

        result["repetition"] = pd.to_numeric(
            result["repetition"],
            errors="coerce"
        )

    return result


# ------------------------------------------------------------
# 4.4 DETECT FIDELITY METRICS
# ------------------------------------------------------------

def get_fidelity_columns(frame):

    metric_map = {}

    for column in frame.columns:

        normalized = normalize_column_name(column)

        if normalized in FIDELITY_METRIC_ALIASES:

            metric_map[column] = (
                FIDELITY_METRIC_ALIASES[normalized]
            )

    return metric_map


# ------------------------------------------------------------
# 4.5 DISCOVER PRIMARY FIDELITY ARTIFACTS
# ------------------------------------------------------------

fidelity_source_items = []

fidelity_source_inventory = []

for item in artifact_records:

    if not is_primary_fidelity_source(item):
        continue

    frame = item.get("normalized_data")

    if frame is None or frame.empty:
        continue

    metric_map = get_fidelity_columns(frame)

    if not metric_map:
        continue

    fidelity_source_items.append(item)

    fidelity_source_inventory.append({

        "source_path": item.get("source_path"),
        "source_notebook": item.get("source_notebook"),
        "filename": item.get("filename"),

        "rows": int(len(frame)),

        "fidelity_metrics_detected": "|".join(
            metric_map.values()
        ),

        "metric_columns_detected": int(
            len(metric_map)
        )

    })


fidelity_source_inventory_df = pd.DataFrame(
    fidelity_source_inventory
)


# ------------------------------------------------------------
# 4.6 EXTRACT VALID MEASUREMENTS
# ------------------------------------------------------------

fidelity_measurement_records = []
fidelity_missing_records = []

for item in fidelity_source_items:

    frame = prepare_fidelity_frame(
        item["normalized_data"]
    )

    metric_map = get_fidelity_columns(frame)

    for source_column, canonical_metric in metric_map.items():

        numeric_values = pd.to_numeric(
            frame[source_column],
            errors="coerce"
        )

        numeric_values = numeric_values.replace(
            [np.inf, -np.inf],
            np.nan
        )

        valid_mask = numeric_values.notna()

        missing_count = int(
            (~valid_mask).sum()
        )

        valid_count = int(
            valid_mask.sum()
        )

        fidelity_missing_records.append({

            "source_path": item.get("source_path"),
            "filename": item.get("filename"),

            "metric": canonical_metric,

            "source_rows": int(len(frame)),

            "valid_measurements": valid_count,

            "missing_measurements": missing_count,

            "evidence_status": (
                "AVAILABLE"
                if valid_count > 0
                else "MISSING_EMPIRICAL_EVIDENCE"
            )

        })

        if valid_count == 0:
            continue

        required_keys = [
            column
            for column in [
                "dataset",
                "model",
                "variant",
                "repetition"
            ]
            if column in frame.columns
        ]

        part = frame.loc[
            valid_mask,
            required_keys
        ].copy()

        part["value"] = numeric_values.loc[
            valid_mask
        ].values

        part["metric"] = canonical_metric
        part["metric_group"] = "fidelity"

        part["experiment_type"] = np.where(
            part.get(
                "variant",
                pd.Series(
                    pd.NA,
                    index=part.index
                )
            ).notna(),
            "ablation",
            np.where(
                part.get(
                    "model",
                    pd.Series(
                        pd.NA,
                        index=part.index
                    )
                ).notna(),
                "model",
                "unidentified"
            )
        )

        part["source_path"] = item.get("source_path")
        part["source_notebook"] = item.get("source_notebook")
        part["source_filename"] = item.get("filename")

        fidelity_measurement_records.append(part)


# ------------------------------------------------------------
# 4.7 CREATE INTEGRATED FIDELITY DATAFRAME
# ------------------------------------------------------------

if fidelity_measurement_records:

    fidelity_results = pd.concat(
        fidelity_measurement_records,
        ignore_index=True
    )

else:

    fidelity_results = pd.DataFrame(
        columns=FIDELITY_OUTPUT_COLUMNS
    )


for column in FIDELITY_OUTPUT_COLUMNS:

    if column not in fidelity_results.columns:
        fidelity_results[column] = pd.NA

fidelity_results = fidelity_results[
    FIDELITY_OUTPUT_COLUMNS
].copy()


# ------------------------------------------------------------
# 4.8 MISSING EVIDENCE REPORT
# ------------------------------------------------------------

fidelity_missing_evidence_df = pd.DataFrame(
    fidelity_missing_records
)

if not fidelity_missing_evidence_df.empty:

    fidelity_missing_evidence_df = (
        fidelity_missing_evidence_df
        .drop_duplicates()
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 4.9 METRIC COVERAGE SUMMARY
# ------------------------------------------------------------

if not fidelity_results.empty:

    fidelity_metric_coverage = (
        fidelity_results
        .groupby(
            ["metric", "experiment_type"],
            dropna=False
        )
        .size()
        .reset_index(name="measurement_count")
    )

else:

    fidelity_metric_coverage = pd.DataFrame(
        columns=[
            "metric",
            "experiment_type",
            "measurement_count"
        ]
    )


# ------------------------------------------------------------
# 4.10 SAVE OUTPUT ARTIFACTS
# ------------------------------------------------------------

fidelity_results_path = (
    TABLES_DIR / "integrated_fidelity_results.csv"
)

fidelity_inventory_path = (
    VALIDATION_DIR / "section_04_fidelity_sources.csv"
)

fidelity_missing_path = (
    VALIDATION_DIR / "section_04_missing_evidence.csv"
)

fidelity_coverage_path = (
    TABLES_DIR / "section_04_fidelity_metric_coverage.csv"
)

fidelity_results.to_csv(
    fidelity_results_path,
    index=False
)

fidelity_source_inventory_df.to_csv(
    fidelity_inventory_path,
    index=False
)

fidelity_missing_evidence_df.to_csv(
    fidelity_missing_path,
    index=False
)

fidelity_metric_coverage.to_csv(
    fidelity_coverage_path,
    index=False
)


# ------------------------------------------------------------
# 4.11 STRUCTURAL VALIDATION
# ------------------------------------------------------------

section_04_checks = [

    (
        "Fidelity result dataframe created",
        isinstance(fidelity_results, pd.DataFrame)
    ),

    (
        "Required output columns present",
        set(FIDELITY_OUTPUT_COLUMNS).issubset(
            fidelity_results.columns
        )
    ),

    (
        "Source inventory created",
        isinstance(
            fidelity_source_inventory_df,
            pd.DataFrame
        )
    ),

    (
        "Missing evidence report created",
        isinstance(
            fidelity_missing_evidence_df,
            pd.DataFrame
        )
    ),

    (
        "Metric coverage report created",
        isinstance(
            fidelity_metric_coverage,
            pd.DataFrame
        )
    ),

    (
        "Integrated values are numeric",
        (
            fidelity_results.empty
            or pd.to_numeric(
                fidelity_results["value"],
                errors="coerce"
            ).notna().all()
        )
    ),

    (
        "Dataset keys preserved",
        (
            fidelity_results.empty
            or fidelity_results["dataset"].notna().all()
        )
    ),

    (
        "Model and variant columns remain separate",
        (
            "model" in fidelity_results.columns
            and "variant" in fidelity_results.columns
        )
    ),

    (
        "Fidelity result file saved",
        fidelity_results_path.exists()
    ),

    (
        "Source inventory file saved",
        fidelity_inventory_path.exists()
    ),

    (
        "Missing evidence file saved",
        fidelity_missing_path.exists()
    ),

    (
        "Metric coverage file saved",
        fidelity_coverage_path.exists()
    )

]


section_04_validation_df = pd.DataFrame(
    section_04_checks,
    columns=["check", "passed"]
)

section_04_validation_df["status"] = (
    section_04_validation_df["passed"]
    .map({
        True: "PASS",
        False: "FAIL"
    })
)

section_04_validation_df.to_csv(
    VALIDATION_DIR / "section_04_validation.csv",
    index=False
)

section_04_passed = int(
    section_04_validation_df["passed"].sum()
)

section_04_failed = (
    len(section_04_validation_df)
    - section_04_passed
)


# ------------------------------------------------------------
# 4.12 EMPIRICAL EVIDENCE STATUS
# ------------------------------------------------------------

fidelity_measurement_count = int(
    len(fidelity_results)
)

fidelity_source_count = int(
    fidelity_results["source_path"].nunique()
)

fidelity_candidate_count = int(
    len(fidelity_source_inventory_df)
)

if section_04_failed > 0:

    section_04_status = "FAILED"

elif fidelity_measurement_count == 0:

    section_04_status = "COMPLETED_WITH_MISSING_EVIDENCE"

else:

    section_04_status = "COMPLETED_WITH_EMPIRICAL_EVIDENCE"


# ------------------------------------------------------------
# 4.13 SECTION SUMMARY
# ------------------------------------------------------------

section_04_summary = {

    "section": 4,

    "section_name": "Integrate Fidelity Results",

    "primary_fidelity_source_artifacts": (
        fidelity_candidate_count
    ),

    "fidelity_measurements": (
        fidelity_measurement_count
    ),

    "fidelity_source_artifacts_with_valid_values": (
        fidelity_source_count
    ),

    "unique_datasets": int(
        fidelity_results["dataset"].nunique()
    ),

    "unique_models": int(
        fidelity_results["model"].nunique()
    ),

    "unique_ablation_variants": int(
        fidelity_results["variant"].nunique()
    ),

    "unique_fidelity_metrics": int(
        fidelity_results["metric"].nunique()
    ),

    "missing_metric_records": int(
        len(fidelity_missing_evidence_df)
    ),

    "validation_passed": section_04_passed,

    "validation_failed": section_04_failed,

    "structural_validation_status": (
        "PASS"
        if section_04_failed == 0
        else "FAILED"
    ),

    "empirical_evidence_status": (
        "AVAILABLE"
        if fidelity_measurement_count > 0
        else "MISSING"
    ),

    "status": section_04_status
}


with open(
    METADATA_DIR / "section_04_summary.json",
    "w"
) as f:

    json.dump(
        section_04_summary,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 4.14 FINAL OUTPUT
# ------------------------------------------------------------

print("=" * 65)
print("NOTEBOOK 20 — SECTION 4: INTEGRATE FIDELITY RESULTS")
print("=" * 65)

print(
    "Primary fidelity source artifacts:",
    fidelity_candidate_count
)

print(
    "Valid fidelity measurements:",
    fidelity_measurement_count
)

print(
    "Source artifacts with valid measurements:",
    fidelity_source_count
)

print(
    "Unique datasets:",
    fidelity_results["dataset"].nunique()
)

print(
    "Unique models:",
    fidelity_results["model"].nunique()
)

print(
    "Unique ablation variants:",
    fidelity_results["variant"].nunique()
)

print(
    "Unique fidelity metrics:",
    fidelity_results["metric"].nunique()
)

print(
    "Missing metric records:",
    len(fidelity_missing_evidence_df)
)

print("\nMetric evidence status:")

if not fidelity_missing_evidence_df.empty:

    print(
        fidelity_missing_evidence_df[
            [
                "filename",
                "metric",
                "source_rows",
                "valid_measurements",
                "missing_measurements",
                "evidence_status"
            ]
        ].to_string(index=False)
    )

print("\nStructural validation passed:", section_04_passed)
print("Structural validation failed:", section_04_failed)

print(
    "Structural validation status:",
    section_04_summary["structural_validation_status"]
)

print(
    "Empirical evidence status:",
    section_04_summary["empirical_evidence_status"]
)

print("Section 4 status:", section_04_status)

print("\nSaved:")
print(fidelity_results_path)
print(fidelity_inventory_path)
print(fidelity_missing_path)
print(fidelity_coverage_path)
print(VALIDATION_DIR / "section_04_validation.csv")
print(METADATA_DIR / "section_04_summary.json")

assert section_04_failed == 0, (
    "Section 4 structural validation failed."
)


NOTEBOOK 20 — SECTION 4: INTEGRATE FIDELITY RESULTS
Primary fidelity source artifacts: 3
Valid fidelity measurements: 0
Source artifacts with valid measurements: 0
Unique datasets: 0
Unique models: 0
Unique ablation variants: 0
Unique fidelity metrics: 0
Missing metric records: 34

Metric evidence status:
                           filename                      metric  source_rows  valid_measurements  missing_measurements            evidence_status
aligned_experimental_conditions.csv        wasserstein_distance           60                   0                    60 MISSING_EMPIRICAL_EVIDENCE
aligned_experimental_conditions.csv               js_divergence           60                   0                    60 MISSING_EMPIRICAL_EVIDENCE
aligned_experimental_conditions.csv           correlation_error           60                   0                    60 MISSING_EMPIRICAL_EVIDENCE
aligned_experimental_conditions.csv                         mmd           60                   0             

In [15]:
# ============================================================
# SECTION 5 — INTEGRATE ML UTILITY RESULTS
# ============================================================

UTILITY_METRIC_ALIASES = {
    "accuracy": "accuracy",
    "macro_f1": "macro_f1",
    "weighted_f1": "weighted_f1",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "f1_score": "f1",
    "auc": "auc",
    "roc_auc": "auc",
    "rmse": "rmse",
    "mae": "mae",
    "r2": "r2",
    "r_squared": "r2",
    "tstr_accuracy": "tstr_accuracy",
    "tstr_macro_f1": "tstr_macro_f1",
    "tstr_weighted_f1": "tstr_weighted_f1",
}

UTILITY_SOURCE_FILENAMES = {
    "normalized_metrics.csv",
    "aligned_experimental_conditions.csv",
    "tradeoff_measures.csv",
}

UTILITY_SOURCE_NOTEBOOKS = {
    "notebook_18",
    "notebook_19",
}

UTILITY_KEY_COLUMNS = [
    "dataset",
    "model",
    "variant",
    "repetition",
]

utility_records = []
utility_evidence_records = []
utility_source_records = []

for item in artifact_records:
    filename = str(item.get("filename", "")).lower()
    source_path = str(item.get("path", ""))
    source_notebook = str(item.get("source_notebook", "")).lower()

    if filename not in UTILITY_SOURCE_FILENAMES:
        continue

    if source_notebook not in UTILITY_SOURCE_NOTEBOOKS:
        continue

    frame = item.get("normalized_data", item.get("data"))

    if not isinstance(frame, pd.DataFrame) or frame.empty:
        continue

    frame = apply_aliases(frame.copy())

    normalized_columns = {
        column: normalize_column_name(column)
        for column in frame.columns
    }
    frame = frame.rename(columns=normalized_columns)

    for key in UTILITY_KEY_COLUMNS:
        if key not in frame.columns:
            frame[key] = np.nan

    frame["dataset"] = frame["dataset"].astype("string").str.strip().str.lower()
    frame["model"] = frame["model"].astype("string").str.strip().str.lower()
    frame["variant"] = frame["variant"].astype("string").str.strip().str.lower()
    frame["repetition"] = pd.to_numeric(
        frame["repetition"], errors="coerce"
    )

    utility_columns = []

    for column in frame.columns:
        metric_name = UTILITY_METRIC_ALIASES.get(
            normalize_column_name(column)
        )
        if metric_name:
            utility_columns.append((column, metric_name))

    for column, metric_name in utility_columns:
        values = pd.to_numeric(frame[column], errors="coerce")
        valid_mask = values.notna() & np.isfinite(values)

        utility_evidence_records.append({
            "source_path": source_path,
            "source_notebook": source_notebook,
            "filename": filename,
            "metric": metric_name,
            "source_rows": len(frame),
            "valid_measurements": int(valid_mask.sum()),
            "missing_measurements": int((~valid_mask).sum()),
            "evidence_status": (
                "AVAILABLE"
                if valid_mask.any()
                else "MISSING_EMPIRICAL_EVIDENCE"
            )
        })

        if not valid_mask.any():
            continue

        part = frame.loc[valid_mask, UTILITY_KEY_COLUMNS].copy()
        part["metric"] = metric_name
        part["value"] = values.loc[valid_mask].astype(float)
        part["metric_group"] = "utility"
        part["source_path"] = source_path
        part["source_notebook"] = source_notebook
        part["source_filename"] = filename

        # Preserve whether the source identifies a baseline model
        # or an ablation variant.
        part["experiment_type"] = np.where(
            part["variant"].notna(),
            "ablation",
            np.where(part["model"].notna(), "baseline", "unclassified")
        )

        utility_records.extend(part.to_dict("records"))

    utility_source_records.append({
        "source_path": source_path,
        "source_notebook": source_notebook,
        "filename": filename,
        "rows": len(frame),
        "utility_columns_found": len(utility_columns),
    })

UTILITY_RESULT_COLUMNS = [
    "dataset", "model", "variant", "repetition",
    "metric", "value", "metric_group", "experiment_type",
    "source_path", "source_notebook", "source_filename"
]

utility_results = pd.DataFrame(
    utility_records,
    columns=UTILITY_RESULT_COLUMNS
)

utility_evidence_df = pd.DataFrame(utility_evidence_records)
utility_sources_df = pd.DataFrame(utility_source_records)

utility_results.to_csv(
    TABLES_DIR / "integrated_utility_results.csv",
    index=False
)

utility_sources_df.to_csv(
    VALIDATION_DIR / "section_05_utility_sources.csv",
    index=False
)

utility_evidence_df.to_csv(
    VALIDATION_DIR / "section_05_missing_evidence.csv",
    index=False
)

# ------------------------------------------------------------
# STRUCTURAL VALIDATION — EMPIRICAL EVIDENCE IS SEPARATE
# ------------------------------------------------------------

section_05_checks = {
    "integrated_results_schema_valid":
        list(utility_results.columns) == UTILITY_RESULT_COLUMNS,

    "output_file_exists":
        (TABLES_DIR / "integrated_utility_results.csv").exists(),

    "source_inventory_saved":
        (VALIDATION_DIR / "section_05_utility_sources.csv").exists(),

    "evidence_inventory_saved":
        (VALIDATION_DIR / "section_05_missing_evidence.csv").exists(),

    "no_non_finite_measurements":
        (
            utility_results.empty
            or np.isfinite(utility_results["value"]).all()
        ),

    "source_provenance_preserved":
        (
            utility_results.empty
            or utility_results[
                ["source_path", "source_notebook", "source_filename"]
            ].notna().all().all()
        ),

    "no_duplicate_measurement_records":
        (
            utility_results.empty
            or not utility_results.duplicated(
                UTILITY_KEY_COLUMNS + ["metric", "source_path"]
            ).any()
        ),
}

section_05_validation = pd.DataFrame([
    {"check": name, "status": "PASS" if passed else "FAIL"}
    for name, passed in section_05_checks.items()
])

section_05_validation.to_csv(
    VALIDATION_DIR / "section_05_validation.csv",
    index=False
)

structural_passed = int(
    (section_05_validation["status"] == "PASS").sum()
)
structural_failed = int(
    (section_05_validation["status"] == "FAIL").sum()
)

empirical_status = (
    "AVAILABLE"
    if len(utility_results) > 0
    else "MISSING_EMPIRICAL_EVIDENCE"
)

section_05_status = (
    "COMPLETED_WITH_EMPIRICAL_EVIDENCE"
    if structural_failed == 0 and len(utility_results) > 0
    else (
        "COMPLETED_WITH_MISSING_EVIDENCE"
        if structural_failed == 0
        else "FAILED"
    )
)

section_05_summary = {
    "section": 5,
    "section_name": "Integrate ML Utility Results",
    "source_artifacts": int(len(utility_sources_df)),
    "utility_measurements": int(len(utility_results)),
    "unique_metrics": int(utility_results["metric"].nunique()),
    "unique_datasets": int(utility_results["dataset"].nunique()),
    "empirical_evidence_status": empirical_status,
    "structural_checks_passed": structural_passed,
    "structural_checks_failed": structural_failed,
    "section_status": section_05_status,
}

with open(
    METADATA_DIR / "section_05_summary.json",
    "w"
) as f:
    json.dump(section_05_summary, f, indent=4)

print("=" * 65)
print("NOTEBOOK 20 — SECTION 5: INTEGRATE ML UTILITY RESULTS")
print("=" * 65)
print("Utility source artifacts:", len(utility_sources_df))
print("Valid utility measurements:", len(utility_results))
print("Unique utility metrics:", utility_results["metric"].nunique())
print("Unique datasets:", utility_results["dataset"].nunique())
print("Missing metric records:", len(utility_evidence_df))
print()
print("Structural validation passed:", structural_passed)
print("Structural validation failed:", structural_failed)
print("Empirical evidence status:", empirical_status)
print("Section 5 status:", section_05_status)

print("\nSaved:")
print(TABLES_DIR / "integrated_utility_results.csv")
print(VALIDATION_DIR / "section_05_utility_sources.csv")
print(VALIDATION_DIR / "section_05_missing_evidence.csv")
print(VALIDATION_DIR / "section_05_validation.csv")
print(METADATA_DIR / "section_05_summary.json")

assert structural_failed == 0, (
    "Section 5 structural validation failed. "
    "Inspect section_05_validation.csv."
)

NOTEBOOK 20 — SECTION 5: INTEGRATE ML UTILITY RESULTS
Utility source artifacts: 3
Valid utility measurements: 0
Unique utility metrics: 0
Unique datasets: 0
Missing metric records: 24

Structural validation passed: 7
Structural validation failed: 0
Empirical evidence status: MISSING_EMPIRICAL_EVIDENCE
Section 5 status: COMPLETED_WITH_MISSING_EVIDENCE

Saved:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/integrated_utility_results.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_05_utility_sources.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_05_missing_evidence.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_05_validation.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/metadata/section_05_summary.json


In [17]:
# ============================================================
# SECTION 6 — INTEGRATE EMPIRICAL PRIVACY RESULTS
# ============================================================

PRIVACY_METRIC_ALIASES = {
    "nearest_neighbor_distance": "nearest_neighbor_distance",
    "distance_to_closest_record": "nearest_neighbor_distance",
    "dcr": "dcr",
    "nndr": "nndr",
    "nearest_neighbor_disclosure": "nearest_neighbor_disclosure",
    "disclosure_risk": "disclosure_risk",
    "reidentification_risk": "reidentification_risk",
    "re_identification_risk": "reidentification_risk",
    "membership_inference_auc": "membership_inference_auc",
    "mia_auc": "membership_inference_auc",
    "membership_inference_advantage": "membership_inference_advantage",
    "membership_inference_risk": "membership_inference_risk",
    "attribute_inference_risk": "attribute_inference_risk",
    "attribute_disclosure_risk": "attribute_disclosure_risk",
    "singling_out_risk": "singling_out_risk",
}

PRIVACY_SOURCE_FILENAMES = {
    "normalized_metrics.csv",
    "aligned_experimental_conditions.csv",
    "tradeoff_measures.csv",
}

PRIVACY_SOURCE_NOTEBOOKS = {"notebook_18", "notebook_19"}

privacy_records = []
privacy_evidence_records = []
privacy_source_records = []

for item in artifact_records:
    filename = str(item.get("filename", "")).lower()
    source_path = str(item.get("path", ""))
    source_notebook = str(item.get("source_notebook", "")).lower()

    if filename not in PRIVACY_SOURCE_FILENAMES:
        continue

    if source_notebook not in PRIVACY_SOURCE_NOTEBOOKS:
        continue

    frame = item.get("normalized_data", item.get("data"))

    if not isinstance(frame, pd.DataFrame) or frame.empty:
        continue

    frame = apply_aliases(frame.copy())

    frame = frame.rename(columns={
        column: normalize_column_name(column)
        for column in frame.columns
    })

    for key in ["dataset", "model", "variant", "repetition"]:
        if key not in frame.columns:
            frame[key] = np.nan

    frame["dataset"] = frame["dataset"].astype("string").str.strip().str.lower()
    frame["model"] = frame["model"].astype("string").str.strip().str.lower()
    frame["variant"] = frame["variant"].astype("string").str.strip().str.lower()
    frame["repetition"] = pd.to_numeric(
        frame["repetition"], errors="coerce"
    )

    privacy_columns = []

    for column in frame.columns:
        metric_name = PRIVACY_METRIC_ALIASES.get(
            normalize_column_name(column)
        )
        if metric_name:
            privacy_columns.append((column, metric_name))

    for column, metric_name in privacy_columns:
        values = pd.to_numeric(frame[column], errors="coerce")
        valid_mask = values.notna() & np.isfinite(values)

        privacy_evidence_records.append({
            "source_path": source_path,
            "source_notebook": source_notebook,
            "filename": filename,
            "metric": metric_name,
            "source_rows": len(frame),
            "valid_measurements": int(valid_mask.sum()),
            "missing_measurements": int((~valid_mask).sum()),
            "evidence_status": (
                "AVAILABLE"
                if valid_mask.any()
                else "MISSING_EMPIRICAL_EVIDENCE"
            )
        })

        if not valid_mask.any():
            continue

        part = frame.loc[
            valid_mask, ["dataset", "model", "variant", "repetition"]
        ].copy()

        part["metric"] = metric_name
        part["value"] = values.loc[valid_mask].astype(float)
        part["metric_group"] = "empirical_privacy"
        part["experiment_type"] = np.where(
            part["variant"].notna(),
            "ablation",
            np.where(part["model"].notna(), "baseline", "unclassified")
        )
        part["source_path"] = source_path
        part["source_notebook"] = source_notebook
        part["source_filename"] = filename

        privacy_records.extend(part.to_dict("records"))

    privacy_source_records.append({
        "source_path": source_path,
        "source_notebook": source_notebook,
        "filename": filename,
        "rows": len(frame),
        "privacy_columns_found": len(privacy_columns),
    })

PRIVACY_RESULT_COLUMNS = [
    "dataset", "model", "variant", "repetition",
    "metric", "value", "metric_group", "experiment_type",
    "source_path", "source_notebook", "source_filename"
]

privacy_results = pd.DataFrame(
    privacy_records,
    columns=PRIVACY_RESULT_COLUMNS
)

privacy_evidence_df = pd.DataFrame(privacy_evidence_records)
privacy_sources_df = pd.DataFrame(privacy_source_records)

privacy_results.to_csv(
    TABLES_DIR / "integrated_empirical_privacy_results.csv",
    index=False
)

privacy_sources_df.to_csv(
    VALIDATION_DIR / "section_06_privacy_sources.csv",
    index=False
)

privacy_evidence_df.to_csv(
    VALIDATION_DIR / "section_06_missing_evidence.csv",
    index=False
)

# ------------------------------------------------------------
# STRUCTURAL VALIDATION
# ------------------------------------------------------------

section_06_checks = {
    "integrated_schema_valid":
        list(privacy_results.columns) == PRIVACY_RESULT_COLUMNS,

    "results_file_exists":
        (TABLES_DIR / "integrated_empirical_privacy_results.csv").exists(),

    "source_inventory_saved":
        (VALIDATION_DIR / "section_06_privacy_sources.csv").exists(),

    "missing_evidence_inventory_saved":
        (VALIDATION_DIR / "section_06_missing_evidence.csv").exists(),

    "all_values_finite":
        (
            privacy_results.empty
            or np.isfinite(privacy_results["value"]).all()
        ),

    "source_provenance_preserved":
        (
            privacy_results.empty
            or privacy_results[
                ["source_path", "source_notebook", "source_filename"]
            ].notna().all().all()
        ),

    "no_duplicate_measurements":
        (
            privacy_results.empty
            or not privacy_results.duplicated(
                [
                    "dataset", "model", "variant", "repetition",
                    "metric", "source_path"
                ]
            ).any()
        ),
}

section_06_validation = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "FAIL"
    }
    for name, passed in section_06_checks.items()
])

section_06_validation.to_csv(
    VALIDATION_DIR / "section_06_validation.csv",
    index=False
)

structural_passed = int(
    (section_06_validation["status"] == "PASS").sum()
)
structural_failed = int(
    (section_06_validation["status"] == "FAIL").sum()
)

empirical_status = (
    "AVAILABLE"
    if len(privacy_results) > 0
    else "MISSING_EMPIRICAL_EVIDENCE"
)

section_06_status = (
    "COMPLETED_WITH_EMPIRICAL_EVIDENCE"
    if structural_failed == 0 and len(privacy_results) > 0
    else (
        "COMPLETED_WITH_MISSING_EVIDENCE"
        if structural_failed == 0
        else "FAILED"
    )
)

section_06_summary = {
    "section": 6,
    "section_name": "Integrate Empirical Privacy Results",
    "source_artifacts": int(len(privacy_sources_df)),
    "empirical_privacy_measurements": int(len(privacy_results)),
    "unique_metrics": int(privacy_results["metric"].nunique()),
    "unique_datasets": int(privacy_results["dataset"].nunique()),
    "empirical_evidence_status": empirical_status,
    "structural_checks_passed": structural_passed,
    "structural_checks_failed": structural_failed,
    "section_status": section_06_status,
}

with open(
    METADATA_DIR / "section_06_summary.json",
    "w"
) as f:
    json.dump(section_06_summary, f, indent=4)

print("=" * 65)
print("NOTEBOOK 20 — SECTION 6: INTEGRATE EMPIRICAL PRIVACY RESULTS")
print("=" * 65)
print("Privacy source artifacts:", len(privacy_sources_df))
print("Empirical privacy measurements:", len(privacy_results))
print("Unique privacy metrics:", privacy_results["metric"].nunique())
print("Unique datasets:", privacy_results["dataset"].nunique())
print("Missing metric records:", len(privacy_evidence_df))
print()
print("Structural validation passed:", structural_passed)
print("Structural validation failed:", structural_failed)
print("Empirical evidence status:", empirical_status)
print("Section 6 status:", section_06_status)

print("\nSaved:")
print(TABLES_DIR / "integrated_empirical_privacy_results.csv")
print(VALIDATION_DIR / "section_06_privacy_sources.csv")
print(VALIDATION_DIR / "section_06_missing_evidence.csv")
print(VALIDATION_DIR / "section_06_validation.csv")
print(METADATA_DIR / "section_06_summary.json")

assert structural_failed == 0, (
    "Section 6 structural validation failed. "
    "Inspect section_06_validation.csv."
)

NOTEBOOK 20 — SECTION 6: INTEGRATE EMPIRICAL PRIVACY RESULTS
Privacy source artifacts: 3
Empirical privacy measurements: 0
Unique privacy metrics: 0
Unique datasets: 0
Missing metric records: 12

Structural validation passed: 7
Structural validation failed: 0
Empirical evidence status: MISSING_EMPIRICAL_EVIDENCE
Section 6 status: COMPLETED_WITH_MISSING_EVIDENCE

Saved:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/integrated_empirical_privacy_results.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_06_privacy_sources.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_06_missing_evidence.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_06_validation.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/metadata/section_06_summary.json


In [19]:
# ============================================================
# SECTION 7 — INTEGRATE FORMAL DIFFERENTIAL PRIVACY GUARANTEES
# ============================================================

DP_SOURCE_FILENAMES = {
    "dp_evidence.csv",
    "dp_accountant_results.csv",
    "privacy_accountant_results.csv",
    "formal_dp_results.csv",
    "dp_validation_results.csv",
    "dp_guarantee_results.csv",
}

DP_RESULT_COLUMNS = [
    "dataset",
    "model",
    "variant",
    "repetition",
    "achieved_epsilon",
    "achieved_delta",
    "formal_dp_verified",
    "accountant_evidence_present",
    "verified_dp_evidence",
    "source_path",
    "source_notebook",
    "source_filename",
]

dp_records = []
dp_source_inventory = []
dp_exceptions = []

def parse_explicit_boolean(series):
    """Parse explicit boolean evidence without treating missing values as False."""
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "1": True,
            "yes": True,
            "false": False,
            "0": False,
            "no": False,
        })
    )

for item in artifact_records:
    filename = str(item.get("filename", "")).lower()
    source_path = str(item.get("path", item.get("source_path", "")))
    source_notebook = str(item.get("source_notebook", "")).lower()

    # Do not infer formal DP from generic privacy, ablation,
    # trade-off, or configuration artifacts.
    if filename not in DP_SOURCE_FILENAMES:
        continue

    frame = item.get("normalized_data", item.get("data"))

    if not isinstance(frame, pd.DataFrame) or frame.empty:
        dp_exceptions.append({
            "source_path": source_path,
            "filename": filename,
            "reason": "EMPTY_OR_UNREADABLE_SOURCE",
        })
        continue

    frame = apply_aliases(frame.copy())
    frame = frame.rename(columns={
        column: normalize_column_name(column)
        for column in frame.columns
    })

    for key in ["dataset", "model", "variant", "repetition"]:
        if key not in frame.columns:
            frame[key] = np.nan

    for column in ["achieved_epsilon", "achieved_delta"]:
        if column in frame.columns:
            frame[column] = pd.to_numeric(
                frame[column], errors="coerce"
            )
        else:
            frame[column] = np.nan

    if "formal_dp_verified" in frame.columns:
        frame["formal_dp_verified"] = parse_explicit_boolean(
            frame["formal_dp_verified"]
        )
    else:
        frame["formal_dp_verified"] = pd.Series(
            pd.NA, index=frame.index, dtype="boolean"
        )

    # Require explicit accountant evidence, not merely configured/applied flags.
    accountant_columns = [
        column for column in [
            "accountant_type",
            "accountant_name",
            "accountant_evidence_path",
            "rdp_evidence_path",
            "privacy_accountant_status",
            "accountant_verified",
        ]
        if column in frame.columns
    ]

    if accountant_columns:
        accountant_present = pd.Series(
            False, index=frame.index, dtype=bool
        )
        for column in accountant_columns:
            values = frame[column].astype("string").str.strip()
            present = values.notna() & ~values.str.lower().isin(
                ["", "none", "nan", "null", "false", "0", "not_available"]
            )
            accountant_present |= present.fillna(False)
    else:
        accountant_present = pd.Series(
            False, index=frame.index, dtype=bool
        )

    epsilon_valid = (
        frame["achieved_epsilon"].notna()
        & np.isfinite(frame["achieved_epsilon"])
        & (frame["achieved_epsilon"] > 0)
    )

    delta_valid = (
        frame["achieved_delta"].notna()
        & np.isfinite(frame["achieved_delta"])
        & (frame["achieved_delta"] > 0)
        & (frame["achieved_delta"] < 1)
    )

    explicit_verification = (
        frame["formal_dp_verified"].fillna(False).astype(bool)
    )

    frame["accountant_evidence_present"] = accountant_present

    frame["verified_dp_evidence"] = (
        explicit_verification
        & epsilon_valid
        & delta_valid
        & accountant_present
    )

    frame["source_path"] = source_path
    frame["source_notebook"] = source_notebook
    frame["source_filename"] = filename

    dp_records.extend(
        frame[DP_RESULT_COLUMNS].to_dict("records")
    )

    dp_source_inventory.append({
        "source_path": source_path,
        "source_notebook": source_notebook,
        "filename": filename,
        "source_rows": len(frame),
        "verified_records": int(frame["verified_dp_evidence"].sum()),
        "accountant_evidence_records": int(accountant_present.sum()),
    })

dp_results = pd.DataFrame(
    dp_records,
    columns=DP_RESULT_COLUMNS
)

dp_sources_df = pd.DataFrame(dp_source_inventory)
dp_exceptions_df = pd.DataFrame(dp_exceptions)

dp_results.to_csv(
    TABLES_DIR / "integrated_formal_dp_results.csv",
    index=False
)

dp_sources_df.to_csv(
    VALIDATION_DIR / "section_07_dp_sources.csv",
    index=False
)

dp_exceptions_df.to_csv(
    VALIDATION_DIR / "section_07_dp_exceptions.csv",
    index=False
)

verified_dp_count = int(
    dp_results["verified_dp_evidence"].sum()
) if not dp_results.empty else 0

configured_or_unverified_count = (
    len(dp_results) - verified_dp_count
)

section_07_checks = {
    "output_schema_valid":
        list(dp_results.columns) == DP_RESULT_COLUMNS,

    "results_file_exists":
        (TABLES_DIR / "integrated_formal_dp_results.csv").exists(),

    "source_inventory_saved":
        (VALIDATION_DIR / "section_07_dp_sources.csv").exists(),

    "exceptions_saved":
        (VALIDATION_DIR / "section_07_dp_exceptions.csv").exists(),

    "verified_records_have_epsilon_delta":
        (
            not dp_results["verified_dp_evidence"].any()
            or (
                dp_results.loc[
                    dp_results["verified_dp_evidence"],
                    ["achieved_epsilon", "achieved_delta"]
                ].notna().all().all()
            )
        ),

    "verified_records_have_accountant_evidence":
        (
            not dp_results["verified_dp_evidence"].any()
            or dp_results.loc[
                dp_results["verified_dp_evidence"],
                "accountant_evidence_present"
            ].all()
        ),

    "no_duplicate_source_records":
        (
            dp_results.empty
            or not dp_results.duplicated(
                [
                    "dataset", "model", "variant", "repetition",
                    "source_path"
                ]
            ).any()
        ),
}

section_07_validation = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "FAIL"
    }
    for name, passed in section_07_checks.items()
])

section_07_validation.to_csv(
    VALIDATION_DIR / "section_07_validation.csv",
    index=False
)

structural_passed = int(
    (section_07_validation["status"] == "PASS").sum()
)
structural_failed = int(
    (section_07_validation["status"] == "FAIL").sum()
)

dp_evidence_status = (
    "VERIFIED_FORMAL_DP_EVIDENCE_AVAILABLE"
    if verified_dp_count > 0
    else "MISSING_FORMAL_DP_EVIDENCE"
)

section_07_status = (
    "COMPLETED_WITH_VERIFIED_DP_EVIDENCE"
    if structural_failed == 0 and verified_dp_count > 0
    else (
        "COMPLETED_WITH_MISSING_EVIDENCE"
        if structural_failed == 0
        else "FAILED"
    )
)

section_07_summary = {
    "section": 7,
    "section_name": "Integrate Formal Differential Privacy Guarantees",
    "dp_source_artifacts": int(len(dp_sources_df)),
    "dp_records": int(len(dp_results)),
    "verified_formal_dp_records": verified_dp_count,
    "unverified_or_configuration_only_records": int(
        configured_or_unverified_count
    ),
    "formal_dp_evidence_status": dp_evidence_status,
    "structural_checks_passed": structural_passed,
    "structural_checks_failed": structural_failed,
    "section_status": section_07_status,
}

with open(
    METADATA_DIR / "section_07_summary.json",
    "w"
) as f:
    json.dump(section_07_summary, f, indent=4)

print("=" * 65)
print("NOTEBOOK 20 — SECTION 7: INTEGRATE DP GUARANTEES")
print("=" * 65)
print("DP source artifacts:", len(dp_sources_df))
print("DP records:", len(dp_results))
print("Verified formal DP records:", verified_dp_count)
print(
    "Unverified/configuration-only records:",
    configured_or_unverified_count
)
print("Formal DP evidence status:", dp_evidence_status)
print("Structural validation passed:", structural_passed)
print("Structural validation failed:", structural_failed)
print("Section 7 status:", section_07_status)

print("\nSaved:")
print(TABLES_DIR / "integrated_formal_dp_results.csv")
print(VALIDATION_DIR / "section_07_dp_sources.csv")
print(VALIDATION_DIR / "section_07_dp_exceptions.csv")
print(VALIDATION_DIR / "section_07_validation.csv")
print(METADATA_DIR / "section_07_summary.json")

assert structural_failed == 0, (
    "Section 7 structural validation failed. "
    "Inspect section_07_validation.csv."
)

NOTEBOOK 20 — SECTION 7: INTEGRATE DP GUARANTEES
DP source artifacts: 0
DP records: 0
Verified formal DP records: 0
Unverified/configuration-only records: 0
Formal DP evidence status: MISSING_FORMAL_DP_EVIDENCE
Structural validation passed: 7
Structural validation failed: 0
Section 7 status: COMPLETED_WITH_MISSING_EVIDENCE

Saved:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/integrated_formal_dp_results.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_07_dp_sources.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_07_dp_exceptions.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_07_validation.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/metadata/section_07_summary.json


In [21]:
# ============================================================
# SECTION 8 — INTEGRATE ABLATION RESULTS
# ============================================================

ABLATION_VARIANTS = {
    "gan_only",
    "statistical_guidance_gan",
    "gan_dp",
    "full_sppgan",
}

ABLATION_KEY_COLUMNS = [
    "dataset",
    "variant",
    "repetition",
]

ABLATION_METRIC_GROUPS = [
    "fidelity",
    "utility",
    "empirical_privacy",
]

ablation_frames = []
ablation_source_inventory = []
ablation_evidence_inventory = []

# ------------------------------------------------------------
# COLLECT VALIDATED ABLATION MEASUREMENTS
# ------------------------------------------------------------

for metric_group, metric_aliases in [
    ("fidelity", FIDELITY_METRIC_ALIASES),
    ("utility", UTILITY_METRIC_ALIASES),
    ("empirical_privacy", PRIVACY_METRIC_ALIASES),
]:

    for item in artifact_records:

        filename = str(item.get("filename", "")).lower()
        source_notebook = str(
            item.get("source_notebook", "")
        ).lower()
        source_path = str(
            item.get("path", item.get("source_path", ""))
        )

        if filename not in {
            "aligned_experimental_conditions.csv",
            "normalized_metrics.csv",
            "tradeoff_measures.csv",
        }:
            continue

        if source_notebook not in {"notebook_18", "notebook_19"}:
            continue

        frame = item.get(
            "normalized_data",
            item.get("data")
        )

        if not isinstance(frame, pd.DataFrame) or frame.empty:
            continue

        frame = apply_aliases(frame.copy())

        frame = frame.rename(columns={
            column: normalize_column_name(column)
            for column in frame.columns
        })

        # Require an explicit ablation variant.
        if "variant" not in frame.columns:
            continue

        frame["variant"] = (
            frame["variant"]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        frame = frame[
            frame["variant"].isin(ABLATION_VARIANTS)
        ].copy()

        if frame.empty:
            continue

        for key in ABLATION_KEY_COLUMNS:
            if key not in frame.columns:
                frame[key] = np.nan

        frame["dataset"] = (
            frame["dataset"]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        frame["repetition"] = pd.to_numeric(
            frame["repetition"],
            errors="coerce"
        )

        metric_columns = []

        for column in frame.columns:
            metric_name = metric_aliases.get(
                normalize_column_name(column)
            )

            if metric_name:
                metric_columns.append((column, metric_name))

        valid_count_for_source = 0

        for column, metric_name in metric_columns:

            values = pd.to_numeric(
                frame[column],
                errors="coerce"
            )

            valid_mask = (
                values.notna()
                & np.isfinite(values)
            )

            ablation_evidence_inventory.append({
                "source_path": source_path,
                "source_notebook": source_notebook,
                "filename": filename,
                "metric_group": metric_group,
                "metric": metric_name,
                "source_rows": len(frame),
                "valid_measurements": int(valid_mask.sum()),
                "missing_measurements": int((~valid_mask).sum()),
                "evidence_status": (
                    "AVAILABLE"
                    if valid_mask.any()
                    else "MISSING_EMPIRICAL_EVIDENCE"
                ),
            })

            if not valid_mask.any():
                continue

            part = frame.loc[
                valid_mask,
                ABLATION_KEY_COLUMNS
            ].copy()

            part["metric"] = metric_name
            part["value"] = values.loc[
                valid_mask
            ].astype(float)

            part["metric_group"] = metric_group
            part["experiment_type"] = "ablation"
            part["source_path"] = source_path
            part["source_notebook"] = source_notebook
            part["source_filename"] = filename

            ablation_frames.append(part)
            valid_count_for_source += int(valid_mask.sum())

        ablation_source_inventory.append({
            "source_path": source_path,
            "source_notebook": source_notebook,
            "filename": filename,
            "ablation_rows": len(frame),
            "metric_columns_found": len(metric_columns),
            "valid_measurements": valid_count_for_source,
        })

# ------------------------------------------------------------
# BUILD MASTER ABLATION RESULTS
# ------------------------------------------------------------

ABLATION_RESULT_COLUMNS = [
    "dataset",
    "variant",
    "repetition",
    "metric",
    "value",
    "metric_group",
    "experiment_type",
    "source_path",
    "source_notebook",
    "source_filename",
]

if ablation_frames:

    ablation_results = pd.concat(
        ablation_frames,
        ignore_index=True
    )

    ablation_results = ablation_results[
        ABLATION_RESULT_COLUMNS
    ]

else:

    ablation_results = pd.DataFrame(
        columns=ABLATION_RESULT_COLUMNS
    )

ablation_sources_df = pd.DataFrame(
    ablation_source_inventory
)

ablation_evidence_df = pd.DataFrame(
    ablation_evidence_inventory
)

# ------------------------------------------------------------
# SAVE ARTIFACTS
# ------------------------------------------------------------

ablation_results.to_csv(
    TABLES_DIR / "integrated_ablation_results.csv",
    index=False
)

ablation_sources_df.to_csv(
    VALIDATION_DIR / "section_08_ablation_sources.csv",
    index=False
)

ablation_evidence_df.to_csv(
    VALIDATION_DIR / "section_08_ablation_evidence.csv",
    index=False
)

# ------------------------------------------------------------
# STRUCTURAL VALIDATION
# ------------------------------------------------------------

section_08_checks = {
    "output_schema_valid":
        list(ablation_results.columns) == ABLATION_RESULT_COLUMNS,

    "results_file_exists":
        (TABLES_DIR / "integrated_ablation_results.csv").exists(),

    "source_inventory_saved":
        (VALIDATION_DIR / "section_08_ablation_sources.csv").exists(),

    "evidence_inventory_saved":
        (VALIDATION_DIR / "section_08_ablation_evidence.csv").exists(),

    "only_registered_variants":
        (
            ablation_results.empty
            or set(ablation_results["variant"].dropna().unique())
            .issubset(ABLATION_VARIANTS)
        ),

    "all_measurements_finite":
        (
            ablation_results.empty
            or np.isfinite(ablation_results["value"]).all()
        ),

    "source_provenance_preserved":
        (
            ablation_results.empty
            or ablation_results[
                [
                    "source_path",
                    "source_notebook",
                    "source_filename",
                ]
            ].notna().all().all()
        ),

    "no_duplicate_measurements":
        (
            ablation_results.empty
            or not ablation_results.duplicated(
                [
                    "dataset",
                    "variant",
                    "repetition",
                    "metric",
                    "source_path",
                ]
            ).any()
        ),
}

section_08_validation = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "FAIL",
    }
    for name, passed in section_08_checks.items()
])

section_08_validation.to_csv(
    VALIDATION_DIR / "section_08_validation.csv",
    index=False
)

structural_passed = int(
    (section_08_validation["status"] == "PASS").sum()
)

structural_failed = int(
    (section_08_validation["status"] == "FAIL").sum()
)

empirical_status = (
    "AVAILABLE"
    if len(ablation_results) > 0
    else "MISSING_EMPIRICAL_EVIDENCE"
)

section_08_status = (
    "COMPLETED_WITH_EMPIRICAL_EVIDENCE"
    if structural_failed == 0 and len(ablation_results) > 0
    else (
        "COMPLETED_WITH_MISSING_EVIDENCE"
        if structural_failed == 0
        else "FAILED"
    )
)

section_08_summary = {
    "section": 8,
    "section_name": "Integrate Ablation Results",
    "source_artifacts": int(len(ablation_sources_df)),
    "ablation_measurements": int(len(ablation_results)),
    "unique_datasets": int(
        ablation_results["dataset"].nunique()
    ),
    "unique_variants": int(
        ablation_results["variant"].nunique()
    ),
    "unique_metrics": int(
        ablation_results["metric"].nunique()
    ),
    "metric_groups": sorted(
        ablation_results["metric_group"].dropna().unique().tolist()
    ),
    "empirical_evidence_status": empirical_status,
    "structural_checks_passed": structural_passed,
    "structural_checks_failed": structural_failed,
    "section_status": section_08_status,
}

with open(
    METADATA_DIR / "section_08_summary.json",
    "w"
) as f:
    json.dump(section_08_summary, f, indent=4)

# ------------------------------------------------------------
# COMPLETION SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("NOTEBOOK 20 — SECTION 8: INTEGRATE ABLATION RESULTS")
print("=" * 65)

print("Ablation source artifacts:", len(ablation_sources_df))
print("Ablation measurements:", len(ablation_results))
print("Ablation variants:", ablation_results["variant"].nunique())
print("Unique datasets:", ablation_results["dataset"].nunique())
print("Unique metrics:", ablation_results["metric"].nunique())

print("\nMetric groups:")
print(
    ablation_results["metric_group"].value_counts().to_string()
    if not ablation_results.empty
    else "No empirical metric groups available"
)

print("\nStructural validation passed:", structural_passed)
print("Structural validation failed:", structural_failed)
print("Empirical evidence status:", empirical_status)
print("Section 8 status:", section_08_status)

print("\nSaved:")
print(TABLES_DIR / "integrated_ablation_results.csv")
print(VALIDATION_DIR / "section_08_ablation_sources.csv")
print(VALIDATION_DIR / "section_08_ablation_evidence.csv")
print(VALIDATION_DIR / "section_08_validation.csv")
print(METADATA_DIR / "section_08_summary.json")

assert structural_failed == 0, (
    "Section 8 structural validation failed. "
    "Inspect section_08_validation.csv."
)

NOTEBOOK 20 — SECTION 8: INTEGRATE ABLATION RESULTS
Ablation source artifacts: 9
Ablation measurements: 0
Ablation variants: 0
Unique datasets: 0
Unique metrics: 0

Metric groups:
No empirical metric groups available

Structural validation passed: 8
Structural validation failed: 0
Empirical evidence status: MISSING_EMPIRICAL_EVIDENCE
Section 8 status: COMPLETED_WITH_MISSING_EVIDENCE

Saved:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/integrated_ablation_results.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_08_ablation_sources.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_08_ablation_evidence.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_08_validation.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/metadata/section_08_summary.json


In [23]:
# ============================================================
# SECTION 9 — INTEGRATE TRADE-OFF RESULTS
# ============================================================

TRADEOFF_NUMERIC_COLUMNS = [
    "privacy_score_mean",
    "privacy_score_std",
    "fidelity_score_mean",
    "fidelity_score_std",
    "utility_score_mean",
    "utility_score_std",
    "tradeoff_score_mean",
    "tradeoff_score_std",
    "expected_runs",
    "valid_runs",
    "variants_evaluated",
    "evaluated_solutions",
    "pareto_optimal_solutions",
    "dominated_solutions",
]

TRADEOFF_KEY_COLUMNS = [
    "dataset",
    "variant",
]

tradeoff_frames = []
tradeoff_source_records = []
tradeoff_evidence_records = []
tradeoff_exceptions = []

# ------------------------------------------------------------
# COLLECT TRADE-OFF ARTIFACTS
# ------------------------------------------------------------

for item in artifact_records:

    if str(item.get("artifact_type", "")).lower() != "tradeoff":
        continue

    frame = item.get("normalized_data", item.get("data"))

    source_path = str(
        item.get("source_path", item.get("path", ""))
    )
    source_notebook = str(
        item.get("source_notebook", "")
    )

    if not isinstance(frame, pd.DataFrame) or frame.empty:
        tradeoff_exceptions.append({
            "source_path": source_path,
            "source_notebook": source_notebook,
            "reason": "EMPTY_OR_UNREADABLE_SOURCE",
        })
        continue

    frame = apply_aliases(frame.copy())

    frame = frame.rename(columns={
        column: normalize_column_name(column)
        for column in frame.columns
    })

    frame["source_path"] = source_path
    frame["source_notebook"] = source_notebook

    for column in TRADEOFF_KEY_COLUMNS:
        if column not in frame.columns:
            frame[column] = np.nan

    for column in TRADEOFF_NUMERIC_COLUMNS:
        if column in frame.columns:
            frame[column] = pd.to_numeric(
                frame[column],
                errors="coerce"
            )

    available_score_columns = [
        column for column in [
            "privacy_score_mean",
            "fidelity_score_mean",
            "utility_score_mean",
            "tradeoff_score_mean",
        ]
        if column in frame.columns
    ]

    valid_score_count = 0

    for column in available_score_columns:
        valid_score_count += int(
            (
                frame[column].notna()
                & np.isfinite(frame[column])
            ).sum()
        )

    tradeoff_source_records.append({
        "source_path": source_path,
        "source_notebook": source_notebook,
        "source_rows": len(frame),
        "score_columns_found": len(available_score_columns),
        "valid_score_values": valid_score_count,
        "evidence_status": (
            "AVAILABLE"
            if valid_score_count > 0
            else "MISSING_EMPIRICAL_EVIDENCE"
        ),
    })

    for column in available_score_columns:
        values = frame[column]

        tradeoff_evidence_records.append({
            "source_path": source_path,
            "source_notebook": source_notebook,
            "metric": column,
            "source_rows": len(frame),
            "valid_measurements": int(
                (values.notna() & np.isfinite(values)).sum()
            ),
            "missing_measurements": int(
                (values.isna() | ~np.isfinite(values.fillna(0))).sum()
            ),
            "evidence_status": (
                "AVAILABLE"
                if values.notna().any()
                else "MISSING_EMPIRICAL_EVIDENCE"
            ),
        })

    tradeoff_frames.append(frame)

# ------------------------------------------------------------
# INTEGRATE WITHOUT INVENTING OR RECOMPUTING SCORES
# ------------------------------------------------------------

if tradeoff_frames:
    tradeoff_results = pd.concat(
        tradeoff_frames,
        ignore_index=True,
        sort=False
    )
else:
    tradeoff_results = pd.DataFrame()

tradeoff_sources_df = pd.DataFrame(tradeoff_source_records)
tradeoff_evidence_df = pd.DataFrame(tradeoff_evidence_records)
tradeoff_exceptions_df = pd.DataFrame(tradeoff_exceptions)

# ------------------------------------------------------------
# EVIDENCE STATUS
# ------------------------------------------------------------

score_columns_present = [
    column for column in [
        "privacy_score_mean",
        "fidelity_score_mean",
        "utility_score_mean",
        "tradeoff_score_mean",
    ]
    if column in tradeoff_results.columns
]

valid_score_values = 0

for column in score_columns_present:
    values = pd.to_numeric(
        tradeoff_results[column],
        errors="coerce"
    )
    valid_score_values += int(
        (values.notna() & np.isfinite(values)).sum()
    )

empirical_tradeoff_status = (
    "AVAILABLE"
    if valid_score_values > 0
    else "MISSING_EMPIRICAL_EVIDENCE"
)

# ------------------------------------------------------------
# SAVE OUTPUTS
# ------------------------------------------------------------

tradeoff_results.to_csv(
    TABLES_DIR / "integrated_tradeoff_results.csv",
    index=False
)

tradeoff_sources_df.to_csv(
    VALIDATION_DIR / "section_09_tradeoff_sources.csv",
    index=False
)

tradeoff_evidence_df.to_csv(
    VALIDATION_DIR / "section_09_tradeoff_evidence.csv",
    index=False
)

tradeoff_exceptions_df.to_csv(
    VALIDATION_DIR / "section_09_tradeoff_exceptions.csv",
    index=False
)

# ------------------------------------------------------------
# STRUCTURAL VALIDATION
# ------------------------------------------------------------

section_09_checks = {
    "results_file_exists":
        (TABLES_DIR / "integrated_tradeoff_results.csv").exists(),

    "source_inventory_saved":
        (VALIDATION_DIR / "section_09_tradeoff_sources.csv").exists(),

    "evidence_inventory_saved":
        (VALIDATION_DIR / "section_09_tradeoff_evidence.csv").exists(),

    "exceptions_file_saved":
        (VALIDATION_DIR / "section_09_tradeoff_exceptions.csv").exists(),

    "source_provenance_preserved":
        (
            tradeoff_results.empty
            or tradeoff_results[
                ["source_path", "source_notebook"]
            ].notna().all().all()
        ),

    "numeric_score_columns_valid":
        all(
            pd.api.types.is_numeric_dtype(tradeoff_results[column])
            for column in score_columns_present
        ),

    "no_non_finite_score_values":
        all(
            np.isfinite(
                tradeoff_results[column].dropna()
            ).all()
            for column in score_columns_present
        ),

    "source_row_count_preserved":
        (
            len(tradeoff_results)
            == sum(
                record["source_rows"]
                for record in tradeoff_source_records
            )
        ),
}

section_09_validation = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "FAIL",
    }
    for name, passed in section_09_checks.items()
])

section_09_validation.to_csv(
    VALIDATION_DIR / "section_09_validation.csv",
    index=False
)

structural_passed = int(
    (section_09_validation["status"] == "PASS").sum()
)

structural_failed = int(
    (section_09_validation["status"] == "FAIL").sum()
)

section_09_status = (
    "COMPLETED_WITH_EMPIRICAL_EVIDENCE"
    if structural_failed == 0 and valid_score_values > 0
    else (
        "COMPLETED_WITH_MISSING_EVIDENCE"
        if structural_failed == 0
        else "FAILED"
    )
)

section_09_summary = {
    "section": 9,
    "section_name": "Integrate Trade-off Results",
    "source_artifacts": int(len(tradeoff_sources_df)),
    "tradeoff_records": int(len(tradeoff_results)),
    "score_columns_found": score_columns_present,
    "valid_score_values": int(valid_score_values),
    "empirical_evidence_status": empirical_tradeoff_status,
    "structural_checks_passed": structural_passed,
    "structural_checks_failed": structural_failed,
    "section_status": section_09_status,
}

with open(
    METADATA_DIR / "section_09_summary.json",
    "w"
) as f:
    json.dump(section_09_summary, f, indent=4)

# ------------------------------------------------------------
# COMPLETION SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("NOTEBOOK 20 — SECTION 9: INTEGRATE TRADE-OFF RESULTS")
print("=" * 65)

print("Trade-off source artifacts:", len(tradeoff_sources_df))
print("Trade-off records:", len(tradeoff_results))
print("Score columns found:", score_columns_present)
print("Valid score values:", valid_score_values)
print("Empirical evidence status:", empirical_tradeoff_status)

print("\nStructural validation passed:", structural_passed)
print("Structural validation failed:", structural_failed)
print("Section 9 status:", section_09_status)

print("\nSaved:")
print(TABLES_DIR / "integrated_tradeoff_results.csv")
print(VALIDATION_DIR / "section_09_tradeoff_sources.csv")
print(VALIDATION_DIR / "section_09_tradeoff_evidence.csv")
print(VALIDATION_DIR / "section_09_tradeoff_exceptions.csv")
print(VALIDATION_DIR / "section_09_validation.csv")
print(METADATA_DIR / "section_09_summary.json")

assert structural_failed == 0, (
    "Section 9 structural validation failed. "
    "Inspect section_09_validation.csv."
)

NOTEBOOK 20 — SECTION 9: INTEGRATE TRADE-OFF RESULTS
Trade-off source artifacts: 4
Trade-off records: 22
Score columns found: ['privacy_score_mean', 'fidelity_score_mean', 'utility_score_mean', 'tradeoff_score_mean']
Valid score values: 0
Empirical evidence status: MISSING_EMPIRICAL_EVIDENCE

Structural validation passed: 8
Structural validation failed: 0
Section 9 status: COMPLETED_WITH_MISSING_EVIDENCE

Saved:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/integrated_tradeoff_results.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_09_tradeoff_sources.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_09_tradeoff_evidence.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_09_tradeoff_exceptions.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_09_validation.csv
/content/drive/MyDrive/SP

In [25]:
# ============================================================
# SECTION 10 — INTEGRATE SIGNIFICANCE RESULTS
# ============================================================

SIGNIFICANCE_SOURCE_TYPES = {"significance"}

SIGNIFICANCE_TEST_COLUMNS = [
    "test_name",
    "test_type",
    "test",
    "method",
    "test_statistic",
    "statistic",
    "p_value",
    "adjusted_p_value",
    "corrected_p_value",
    "effect_size",
    "confidence_interval",
    "ci_lower",
    "ci_upper",
    "n_pairs",
    "sample_size",
    "degrees_of_freedom",
    "significant",
    "test_status",
    "execution_status",
]

significance_frames = []
significance_source_records = []
significance_evidence_records = []
significance_exceptions = []

# ------------------------------------------------------------
# LOAD SIGNIFICANCE ARTIFACTS
# ------------------------------------------------------------

for item in artifact_records:

    if str(item.get("artifact_type", "")).lower() not in SIGNIFICANCE_SOURCE_TYPES:
        continue

    frame = item.get("normalized_data", item.get("data"))

    source_path = str(
        item.get("source_path", item.get("path", ""))
    )
    source_notebook = str(
        item.get("source_notebook", "")
    )

    if not isinstance(frame, pd.DataFrame) or frame.empty:
        significance_exceptions.append({
            "source_path": source_path,
            "source_notebook": source_notebook,
            "reason": "EMPTY_OR_UNREADABLE_SOURCE",
        })
        continue

    frame = apply_aliases(frame.copy())

    frame = frame.rename(columns={
        column: normalize_column_name(column)
        for column in frame.columns
    })

    frame["source_path"] = source_path
    frame["source_notebook"] = source_notebook

    significance_frames.append(frame)

    significance_source_records.append({
        "source_path": source_path,
        "source_notebook": source_notebook,
        "source_rows": len(frame),
        "columns": len(frame.columns),
    })

    # --------------------------------------------------------
    # INSPECT ACTUAL STATISTICAL EVIDENCE
    # --------------------------------------------------------

    for column in SIGNIFICANCE_TEST_COLUMNS:

        if column not in frame.columns:
            continue

        values = frame[column]

        if pd.api.types.is_numeric_dtype(values):
            valid_count = int(
                (
                    values.notna()
                    & np.isfinite(
                        pd.to_numeric(values, errors="coerce")
                    )
                ).sum()
            )
        else:
            normalized_values = (
                values.astype("string")
                .str.strip()
                .str.lower()
            )

            valid_count = int(
                (
                    normalized_values.notna()
                    & ~normalized_values.isin([
                        "",
                        "nan",
                        "none",
                        "null",
                        "not_available",
                        "not_run",
                        "not_executed",
                        "missing",
                    ])
                ).sum()
            )

        significance_evidence_records.append({
            "source_path": source_path,
            "source_notebook": source_notebook,
            "field": column,
            "source_rows": len(frame),
            "valid_values": valid_count,
            "missing_values": len(frame) - valid_count,
            "evidence_status": (
                "FIELD_VALUES_AVAILABLE"
                if valid_count > 0
                else "MISSING_EMPIRICAL_EVIDENCE"
            ),
        })

# ------------------------------------------------------------
# CONSOLIDATE SOURCE RECORDS
# ------------------------------------------------------------

if significance_frames:

    significance_results = pd.concat(
        significance_frames,
        ignore_index=True,
        sort=False
    )

else:

    significance_results = pd.DataFrame()

significance_sources_df = pd.DataFrame(
    significance_source_records
)

significance_evidence_df = pd.DataFrame(
    significance_evidence_records
)

significance_exceptions_df = pd.DataFrame(
    significance_exceptions
)

# ------------------------------------------------------------
# DETERMINE WHETHER TESTS WERE ACTUALLY EXECUTED
# ------------------------------------------------------------

def numeric_evidence_count(frame, column):
    if column not in frame.columns:
        return 0

    values = pd.to_numeric(frame[column], errors="coerce")

    return int(
        (values.notna() & np.isfinite(values)).sum()
    )


statistic_columns = [
    "test_statistic",
    "statistic",
]

p_value_columns = [
    "p_value",
    "adjusted_p_value",
    "corrected_p_value",
]

effect_columns = ["effect_size"]

ci_columns = [
    "ci_lower",
    "ci_upper",
]

statistic_count = sum(
    numeric_evidence_count(significance_results, column)
    for column in statistic_columns
)

p_value_count = sum(
    numeric_evidence_count(significance_results, column)
    for column in p_value_columns
)

effect_size_count = sum(
    numeric_evidence_count(significance_results, column)
    for column in effect_columns
)

confidence_interval_count = sum(
    numeric_evidence_count(significance_results, column)
    for column in ci_columns
)

# A valid executed test must contain an actual test statistic
# and an actual p-value. Metadata/status rows alone do not qualify.
executed_test_evidence = (
    statistic_count > 0
    and p_value_count > 0
)

# ------------------------------------------------------------
# SAVE INTEGRATED RESULTS AND EVIDENCE INVENTORY
# ------------------------------------------------------------

significance_results.to_csv(
    TABLES_DIR / "integrated_significance_results.csv",
    index=False
)

significance_sources_df.to_csv(
    VALIDATION_DIR / "section_10_significance_sources.csv",
    index=False
)

significance_evidence_df.to_csv(
    VALIDATION_DIR / "section_10_significance_evidence.csv",
    index=False
)

significance_exceptions_df.to_csv(
    VALIDATION_DIR / "section_10_significance_exceptions.csv",
    index=False
)

# ------------------------------------------------------------
# STRUCTURAL VALIDATION
# ------------------------------------------------------------

section_10_checks = {
    "integrated_results_file_exists":
        (
            TABLES_DIR / "integrated_significance_results.csv"
        ).exists(),

    "source_inventory_saved":
        (
            VALIDATION_DIR / "section_10_significance_sources.csv"
        ).exists(),

    "evidence_inventory_saved":
        (
            VALIDATION_DIR / "section_10_significance_evidence.csv"
        ).exists(),

    "exceptions_inventory_saved":
        (
            VALIDATION_DIR / "section_10_significance_exceptions.csv"
        ).exists(),

    "source_row_count_preserved":
        (
            len(significance_results)
            == sum(
                record["source_rows"]
                for record in significance_source_records
            )
        ),

    "source_provenance_preserved":
        (
            significance_results.empty
            or significance_results[
                ["source_path", "source_notebook"]
            ].notna().all().all()
        ),

    "numeric_test_statistics_valid":
        all(
            pd.api.types.is_numeric_dtype(
                significance_results[column]
            )
            for column in statistic_columns
            if column in significance_results.columns
        ),

    "numeric_p_values_valid":
        all(
            (
                significance_results[column].dropna().between(0, 1)
            ).all()
            for column in p_value_columns
            if column in significance_results.columns
        ),
}

section_10_validation = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "FAIL",
    }
    for name, passed in section_10_checks.items()
])

section_10_validation.to_csv(
    VALIDATION_DIR / "section_10_validation.csv",
    index=False
)

structural_passed = int(
    (section_10_validation["status"] == "PASS").sum()
)

structural_failed = int(
    (section_10_validation["status"] == "FAIL").sum()
)

empirical_status = (
    "EXECUTED_STATISTICAL_EVIDENCE_AVAILABLE"
    if executed_test_evidence
    else "MISSING_EXECUTED_STATISTICAL_EVIDENCE"
)

section_10_status = (
    "COMPLETED_WITH_EMPIRICAL_EVIDENCE"
    if structural_failed == 0 and executed_test_evidence
    else (
        "COMPLETED_WITH_MISSING_EVIDENCE"
        if structural_failed == 0
        else "FAILED"
    )
)

# ------------------------------------------------------------
# SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_10_summary = {
    "section": 10,
    "section_name": "Integrate Significance Results",
    "source_artifacts": int(len(significance_sources_df)),
    "significance_records": int(len(significance_results)),
    "numeric_test_statistics": statistic_count,
    "numeric_p_values": p_value_count,
    "effect_size_values": effect_size_count,
    "confidence_interval_values": confidence_interval_count,
    "executed_test_evidence": bool(executed_test_evidence),
    "empirical_evidence_status": empirical_status,
    "structural_checks_passed": structural_passed,
    "structural_checks_failed": structural_failed,
    "section_status": section_10_status,
}

with open(
    METADATA_DIR / "section_10_summary.json",
    "w"
) as f:
    json.dump(section_10_summary, f, indent=4)

# ------------------------------------------------------------
# COMPLETION SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("NOTEBOOK 20 — SECTION 10: INTEGRATE SIGNIFICANCE RESULTS")
print("=" * 65)

print("Significance source artifacts:", len(significance_sources_df))
print("Imported significance records:", len(significance_results))
print("Numeric test statistics:", statistic_count)
print("Numeric p-values:", p_value_count)
print("Effect-size values:", effect_size_count)
print("Confidence-interval values:", confidence_interval_count)

print("Executed statistical evidence:", executed_test_evidence)
print("Empirical evidence status:", empirical_status)

print("\nStructural validation passed:", structural_passed)
print("Structural validation failed:", structural_failed)
print("Section 10 status:", section_10_status)

print("\nSaved:")
print(TABLES_DIR / "integrated_significance_results.csv")
print(VALIDATION_DIR / "section_10_significance_sources.csv")
print(VALIDATION_DIR / "section_10_significance_evidence.csv")
print(VALIDATION_DIR / "section_10_significance_exceptions.csv")
print(VALIDATION_DIR / "section_10_validation.csv")
print(METADATA_DIR / "section_10_summary.json")

assert structural_failed == 0, (
    "Section 10 structural validation failed. "
    "Inspect section_10_validation.csv."
)

NOTEBOOK 20 — SECTION 10: INTEGRATE SIGNIFICANCE RESULTS
Significance source artifacts: 7
Imported significance records: 309
Numeric test statistics: 0
Numeric p-values: 36
Effect-size values: 0
Confidence-interval values: 0
Executed statistical evidence: False
Empirical evidence status: MISSING_EXECUTED_STATISTICAL_EVIDENCE

Structural validation passed: 8
Structural validation failed: 0
Section 10 status: COMPLETED_WITH_MISSING_EVIDENCE

Saved:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/integrated_significance_results.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_10_significance_sources.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_10_significance_evidence.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_10_significance_exceptions.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/

In [27]:
# ============================================================
# SECTION 11 — INTEGRATE COMPUTATIONAL RESULTS
# ============================================================

COMPUTATIONAL_METRIC_ALIASES = {
    "runtime": "runtime_seconds",
    "runtime_seconds": "runtime_seconds",
    "training_time": "training_time_seconds",
    "training_time_seconds": "training_time_seconds",
    "inference_time": "inference_time_seconds",
    "inference_time_seconds": "inference_time_seconds",
    "generation_time": "generation_time_seconds",
    "generation_time_seconds": "generation_time_seconds",
    "evaluation_time": "evaluation_time_seconds",
    "evaluation_time_seconds": "evaluation_time_seconds",
    "memory_usage": "memory_usage_mb",
    "memory_usage_mb": "memory_usage_mb",
    "peak_memory": "peak_memory_mb",
    "peak_memory_mb": "peak_memory_mb",
    "gpu_memory": "gpu_memory_mb",
    "gpu_memory_mb": "gpu_memory_mb",
    "cpu_usage": "cpu_usage_percent",
    "cpu_usage_percent": "cpu_usage_percent",
    "gpu_utilization": "gpu_utilization_percent",
    "gpu_utilization_percent": "gpu_utilization_percent",
    "epochs": "epochs",
    "batch_size": "batch_size",
    "model_parameters": "model_parameters",
    "parameter_count": "model_parameters",
}

COMPUTATIONAL_SOURCE_FILENAMES = {
    "aligned_experimental_conditions.csv",
    "normalized_metrics.csv",
    "tradeoff_measures.csv",
}

COMPUTATIONAL_SOURCE_NOTEBOOKS = {
    "notebook_15",
    "notebook_16",
    "notebook_17",
    "notebook_18",
    "notebook_19",
}

COMPUTATIONAL_KEY_COLUMNS = [
    "dataset",
    "model",
    "variant",
    "repetition",
]

computational_records = []
computational_source_records = []
computational_evidence_records = []
computational_exceptions = []

# ------------------------------------------------------------
# COLLECT COMPUTATIONAL MEASUREMENTS
# ------------------------------------------------------------

for item in artifact_records:

    filename = str(item.get("filename", "")).lower()
    source_notebook = str(
        item.get("source_notebook", "")
    ).lower()

    source_path = str(
        item.get("path", item.get("source_path", ""))
    )

    if filename not in COMPUTATIONAL_SOURCE_FILENAMES:
        continue

    if source_notebook not in COMPUTATIONAL_SOURCE_NOTEBOOKS:
        continue

    frame = item.get(
        "normalized_data",
        item.get("data")
    )

    if not isinstance(frame, pd.DataFrame) or frame.empty:
        computational_exceptions.append({
            "source_path": source_path,
            "source_notebook": source_notebook,
            "filename": filename,
            "reason": "EMPTY_OR_UNREADABLE_SOURCE",
        })
        continue

    frame = apply_aliases(frame.copy())

    frame = frame.rename(columns={
        column: normalize_column_name(column)
        for column in frame.columns
    })

    for key in COMPUTATIONAL_KEY_COLUMNS:
        if key not in frame.columns:
            frame[key] = np.nan

    frame["dataset"] = (
        frame["dataset"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    frame["model"] = (
        frame["model"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    frame["variant"] = (
        frame["variant"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    frame["repetition"] = pd.to_numeric(
        frame["repetition"],
        errors="coerce"
    )

    metric_columns = []

    for column in frame.columns:

        metric_name = COMPUTATIONAL_METRIC_ALIASES.get(
            normalize_column_name(column)
        )

        if metric_name:
            metric_columns.append((column, metric_name))

    valid_source_measurements = 0

    for column, metric_name in metric_columns:

        values = pd.to_numeric(
            frame[column],
            errors="coerce"
        )

        valid_mask = (
            values.notna()
            & np.isfinite(values)
            & (values >= 0)
        )

        computational_evidence_records.append({
            "source_path": source_path,
            "source_notebook": source_notebook,
            "filename": filename,
            "metric": metric_name,
            "source_rows": len(frame),
            "valid_measurements": int(valid_mask.sum()),
            "missing_or_invalid_measurements": int(
                (~valid_mask).sum()
            ),
            "evidence_status": (
                "AVAILABLE"
                if valid_mask.any()
                else "MISSING_EMPIRICAL_EVIDENCE"
            ),
        })

        if not valid_mask.any():
            continue

        part = frame.loc[
            valid_mask,
            COMPUTATIONAL_KEY_COLUMNS
        ].copy()

        part["metric"] = metric_name
        part["value"] = values.loc[valid_mask].astype(float)
        part["metric_group"] = "computational"

        part["experiment_type"] = np.where(
            part["variant"].notna(),
            "ablation",
            np.where(
                part["model"].notna(),
                "baseline",
                "unclassified"
            )
        )

        part["source_path"] = source_path
        part["source_notebook"] = source_notebook
        part["source_filename"] = filename

        computational_records.extend(
            part.to_dict("records")
        )

        valid_source_measurements += int(valid_mask.sum())

    computational_source_records.append({
        "source_path": source_path,
        "source_notebook": source_notebook,
        "filename": filename,
        "source_rows": len(frame),
        "computational_columns_found": len(metric_columns),
        "valid_measurements": valid_source_measurements,
    })

# ------------------------------------------------------------
# BUILD INTEGRATED RESULTS
# ------------------------------------------------------------

COMPUTATIONAL_RESULT_COLUMNS = [
    "dataset",
    "model",
    "variant",
    "repetition",
    "metric",
    "value",
    "metric_group",
    "experiment_type",
    "source_path",
    "source_notebook",
    "source_filename",
]

computational_results = pd.DataFrame(
    computational_records,
    columns=COMPUTATIONAL_RESULT_COLUMNS
)

computational_sources_df = pd.DataFrame(
    computational_source_records
)

computational_evidence_df = pd.DataFrame(
    computational_evidence_records
)

computational_exceptions_df = pd.DataFrame(
    computational_exceptions
)

# ------------------------------------------------------------
# SAVE ARTIFACTS
# ------------------------------------------------------------

computational_results.to_csv(
    TABLES_DIR / "integrated_computational_results.csv",
    index=False
)

computational_sources_df.to_csv(
    VALIDATION_DIR / "section_11_computational_sources.csv",
    index=False
)

computational_evidence_df.to_csv(
    VALIDATION_DIR / "section_11_computational_evidence.csv",
    index=False
)

computational_exceptions_df.to_csv(
    VALIDATION_DIR / "section_11_computational_exceptions.csv",
    index=False
)

# ------------------------------------------------------------
# STRUCTURAL VALIDATION
# ------------------------------------------------------------

section_11_checks = {
    "integrated_schema_valid":
        list(computational_results.columns)
        == COMPUTATIONAL_RESULT_COLUMNS,

    "results_file_exists":
        (
            TABLES_DIR / "integrated_computational_results.csv"
        ).exists(),

    "source_inventory_saved":
        (
            VALIDATION_DIR / "section_11_computational_sources.csv"
        ).exists(),

    "evidence_inventory_saved":
        (
            VALIDATION_DIR / "section_11_computational_evidence.csv"
        ).exists(),

    "exceptions_inventory_saved":
        (
            VALIDATION_DIR / "section_11_computational_exceptions.csv"
        ).exists(),

    "all_measurements_finite_and_nonnegative":
        (
            computational_results.empty
            or (
                np.isfinite(computational_results["value"]).all()
                and (computational_results["value"] >= 0).all()
            )
        ),

    "source_provenance_preserved":
        (
            computational_results.empty
            or computational_results[
                [
                    "source_path",
                    "source_notebook",
                    "source_filename",
                ]
            ].notna().all().all()
        ),

    "no_duplicate_measurements":
        (
            computational_results.empty
            or not computational_results.duplicated(
                [
                    "dataset",
                    "model",
                    "variant",
                    "repetition",
                    "metric",
                    "source_path",
                ]
            ).any()
        ),
}

section_11_validation = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "FAIL",
    }
    for name, passed in section_11_checks.items()
])

section_11_validation.to_csv(
    VALIDATION_DIR / "section_11_validation.csv",
    index=False
)

structural_passed = int(
    (section_11_validation["status"] == "PASS").sum()
)

structural_failed = int(
    (section_11_validation["status"] == "FAIL").sum()
)

empirical_status = (
    "AVAILABLE"
    if len(computational_results) > 0
    else "MISSING_EMPIRICAL_EVIDENCE"
)

section_11_status = (
    "COMPLETED_WITH_EMPIRICAL_EVIDENCE"
    if structural_failed == 0 and len(computational_results) > 0
    else (
        "COMPLETED_WITH_MISSING_EVIDENCE"
        if structural_failed == 0
        else "FAILED"
    )
)

# ------------------------------------------------------------
# SAVE SUMMARY
# ------------------------------------------------------------

section_11_summary = {
    "section": 11,
    "section_name": "Integrate Computational Results",
    "source_artifacts": int(len(computational_sources_df)),
    "computational_measurements": int(len(computational_results)),
    "unique_metrics": int(
        computational_results["metric"].nunique()
    ),
    "unique_datasets": int(
        computational_results["dataset"].nunique()
    ),
    "unique_models": int(
        computational_results["model"].nunique()
    ),
    "unique_ablation_variants": int(
        computational_results["variant"].nunique()
    ),
    "empirical_evidence_status": empirical_status,
    "structural_checks_passed": structural_passed,
    "structural_checks_failed": structural_failed,
    "section_status": section_11_status,
}

with open(
    METADATA_DIR / "section_11_summary.json",
    "w"
) as f:
    json.dump(section_11_summary, f, indent=4)

# ------------------------------------------------------------
# COMPLETION SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("NOTEBOOK 20 — SECTION 11: INTEGRATE COMPUTATIONAL RESULTS")
print("=" * 65)

print("Computational source artifacts:", len(computational_sources_df))
print("Computational measurements:", len(computational_results))
print("Computational metrics:", computational_results["metric"].nunique())
print("Unique datasets:", computational_results["dataset"].nunique())
print("Unique models:", computational_results["model"].nunique())
print(
    "Unique ablation variants:",
    computational_results["variant"].nunique()
)

print("\nStructural validation passed:", structural_passed)
print("Structural validation failed:", structural_failed)
print("Empirical evidence status:", empirical_status)
print("Section 11 status:", section_11_status)

print("\nSaved:")
print(TABLES_DIR / "integrated_computational_results.csv")
print(VALIDATION_DIR / "section_11_computational_sources.csv")
print(VALIDATION_DIR / "section_11_computational_evidence.csv")
print(VALIDATION_DIR / "section_11_computational_exceptions.csv")
print(VALIDATION_DIR / "section_11_validation.csv")
print(METADATA_DIR / "section_11_summary.json")

assert structural_failed == 0, (
    "Section 11 structural validation failed. "
    "Inspect section_11_validation.csv."
)

NOTEBOOK 20 — SECTION 11: INTEGRATE COMPUTATIONAL RESULTS
Computational source artifacts: 3
Computational measurements: 0
Computational metrics: 0
Unique datasets: 0
Unique models: 0
Unique ablation variants: 0

Structural validation passed: 8
Structural validation failed: 0
Empirical evidence status: MISSING_EMPIRICAL_EVIDENCE
Section 11 status: COMPLETED_WITH_MISSING_EVIDENCE

Saved:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/integrated_computational_results.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_11_computational_sources.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_11_computational_evidence.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_11_computational_exceptions.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_11_validation.csv
/content/drive/MyDrive/SPP_GAN_R

In [29]:
# ============================================================
# SECTION 12 — MODEL-LEVEL COMPARISON
# ============================================================

MODEL_COMPARISON_COLUMNS = [
    "dataset",
    "comparison_type",
    "comparison_entity",
    "model",
    "variant",
    "metric",
    "metric_group",
    "n",
    "mean",
    "std",
    "median",
    "minimum",
    "maximum",
    "repetitions_observed",
]

COMPARISON_INPUTS = {
    "fidelity": fidelity_results,
    "utility": utility_results,
    "empirical_privacy": privacy_results,
    "computational": computational_results,
}

comparison_frames = []
comparison_exceptions = []

# ------------------------------------------------------------
# NORMALIZE AND COMBINE AVAILABLE MEASUREMENTS
# ------------------------------------------------------------

for metric_group, source_frame in COMPARISON_INPUTS.items():

    if not isinstance(source_frame, pd.DataFrame):
        comparison_exceptions.append({
            "metric_group": metric_group,
            "reason": "SOURCE_NOT_A_DATAFRAME",
        })
        continue

    if source_frame.empty:
        continue

    frame = source_frame.copy()

    required_columns = [
        "dataset",
        "model",
        "variant",
        "repetition",
        "metric",
        "value",
    ]

    for column in required_columns:
        if column not in frame.columns:
            frame[column] = np.nan

    frame["dataset"] = (
        frame["dataset"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    frame["model"] = (
        frame["model"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    frame["variant"] = (
        frame["variant"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    frame["repetition"] = pd.to_numeric(
        frame["repetition"],
        errors="coerce"
    )

    frame["value"] = pd.to_numeric(
        frame["value"],
        errors="coerce"
    )

    frame["metric_group"] = metric_group

    valid_mask = (
        frame["value"].notna()
        & np.isfinite(frame["value"])
        & frame["dataset"].notna()
        & frame["metric"].notna()
    )

    invalid_count = int((~valid_mask).sum())

    if invalid_count:
        comparison_exceptions.append({
            "metric_group": metric_group,
            "reason": "INVALID_OR_INCOMPLETE_MEASUREMENT_ROWS",
            "invalid_rows": invalid_count,
        })

    frame = frame.loc[valid_mask].copy()

    if frame.empty:
        continue

    # Ablation variants and baseline models remain distinct.
    frame["comparison_type"] = np.where(
        frame["variant"].notna(),
        "ablation",
        np.where(
            frame["model"].notna(),
            "baseline",
            "unclassified"
        )
    )

    frame["comparison_entity"] = np.where(
        frame["variant"].notna(),
        frame["variant"],
        frame["model"]
    )

    frame = frame[
        frame["comparison_entity"].notna()
    ].copy()

    if not frame.empty:
        comparison_frames.append(frame)

# ------------------------------------------------------------
# MODEL-LEVEL AGGREGATION
# ------------------------------------------------------------

if comparison_frames:

    all_metric_results = pd.concat(
        comparison_frames,
        ignore_index=True,
        sort=False
    )

    GROUP_COLUMNS = [
        "dataset",
        "comparison_type",
        "comparison_entity",
        "model",
        "variant",
        "metric",
        "metric_group",
    ]

    model_comparison = (
        all_metric_results
        .groupby(
            GROUP_COLUMNS,
            dropna=False
        )
        .agg(
            n=("value", "count"),
            mean=("value", "mean"),
            std=("value", "std"),
            median=("value", "median"),
            minimum=("value", "min"),
            maximum=("value", "max"),
            repetitions_observed=(
                "repetition",
                "nunique"
            ),
        )
        .reset_index()
    )

    model_comparison = model_comparison[
        MODEL_COMPARISON_COLUMNS
    ]

else:

    all_metric_results = pd.DataFrame()

    model_comparison = pd.DataFrame(
        columns=MODEL_COMPARISON_COLUMNS
    )

# ------------------------------------------------------------
# SAVE COMPARISON OUTPUTS
# ------------------------------------------------------------

model_comparison.to_csv(
    TABLES_DIR / "model_level_comparison.csv",
    index=False
)

comparison_exceptions_df = pd.DataFrame(
    comparison_exceptions
)

comparison_exceptions_df.to_csv(
    VALIDATION_DIR / "section_12_comparison_exceptions.csv",
    index=False
)

# ------------------------------------------------------------
# EVIDENCE COVERAGE
# ------------------------------------------------------------

comparison_evidence_status = (
    "AVAILABLE"
    if not model_comparison.empty
    else "MISSING_EMPIRICAL_EVIDENCE"
)

baseline_comparison_count = (
    int(
        (
            model_comparison["comparison_type"] == "baseline"
        ).sum()
    )
    if not model_comparison.empty else 0
)

ablation_comparison_count = (
    int(
        (
            model_comparison["comparison_type"] == "ablation"
        ).sum()
    )
    if not model_comparison.empty else 0
)

# ------------------------------------------------------------
# STRUCTURAL VALIDATION
# ------------------------------------------------------------

section_12_checks = {
    "comparison_schema_valid":
        list(model_comparison.columns)
        == MODEL_COMPARISON_COLUMNS,

    "comparison_file_exists":
        (TABLES_DIR / "model_level_comparison.csv").exists(),

    "exceptions_file_exists":
        (VALIDATION_DIR / "section_12_comparison_exceptions.csv").exists(),

    "all_aggregated_counts_positive":
        (
            model_comparison.empty
            or (model_comparison["n"] > 0).all()
        ),

    "all_aggregated_means_finite":
        (
            model_comparison.empty
            or np.isfinite(model_comparison["mean"]).all()
        ),

    "baseline_and_ablation_types_valid":
        (
            model_comparison.empty
            or set(
                model_comparison["comparison_type"].dropna().unique()
            ).issubset({"baseline", "ablation"})
        ),

    "ablation_variants_preserved":
        (
            model_comparison.empty
            or (
                model_comparison.loc[
                    model_comparison["comparison_type"] == "ablation",
                    "variant"
                ].notna().all()
            )
        ),

    "no_duplicate_aggregation_keys":
        (
            model_comparison.empty
            or not model_comparison.duplicated(
                GROUP_COLUMNS
            ).any()
        ),
}

section_12_validation = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "FAIL",
    }
    for name, passed in section_12_checks.items()
])

section_12_validation.to_csv(
    VALIDATION_DIR / "section_12_validation.csv",
    index=False
)

structural_passed = int(
    (section_12_validation["status"] == "PASS").sum()
)

structural_failed = int(
    (section_12_validation["status"] == "FAIL").sum()
)

section_12_status = (
    "COMPLETED_WITH_EMPIRICAL_EVIDENCE"
    if structural_failed == 0 and not model_comparison.empty
    else (
        "COMPLETED_WITH_MISSING_EVIDENCE"
        if structural_failed == 0
        else "FAILED"
    )
)

# ------------------------------------------------------------
# SAVE SUMMARY
# ------------------------------------------------------------

section_12_summary = {
    "section": 12,
    "section_name": "Model-Level Comparison",
    "integrated_measurements": int(len(all_metric_results)),
    "comparison_rows": int(len(model_comparison)),
    "baseline_comparison_rows": baseline_comparison_count,
    "ablation_comparison_rows": ablation_comparison_count,
    "unique_datasets": int(
        model_comparison["dataset"].nunique()
    ),
    "unique_models": int(
        model_comparison["model"].nunique()
    ),
    "unique_variants": int(
        model_comparison["variant"].nunique()
    ),
    "unique_metrics": int(
        model_comparison["metric"].nunique()
    ),
    "empirical_evidence_status": comparison_evidence_status,
    "structural_checks_passed": structural_passed,
    "structural_checks_failed": structural_failed,
    "section_status": section_12_status,
}

with open(
    METADATA_DIR / "section_12_summary.json",
    "w"
) as f:
    json.dump(section_12_summary, f, indent=4)

# ------------------------------------------------------------
# COMPLETION SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("NOTEBOOK 20 — SECTION 12: MODEL-LEVEL COMPARISON")
print("=" * 65)

print("Integrated measurements:", len(all_metric_results))
print("Model comparison rows:", len(model_comparison))
print("Baseline comparison rows:", baseline_comparison_count)
print("Ablation comparison rows:", ablation_comparison_count)

print("Models observed:", model_comparison["model"].nunique())
print("Ablation variants observed:", model_comparison["variant"].nunique())
print("Datasets observed:", model_comparison["dataset"].nunique())
print("Metrics observed:", model_comparison["metric"].nunique())

print("\nEmpirical evidence status:", comparison_evidence_status)
print("Structural validation passed:", structural_passed)
print("Structural validation failed:", structural_failed)
print("Section 12 status:", section_12_status)

print("\nSaved:")
print(TABLES_DIR / "model_level_comparison.csv")
print(VALIDATION_DIR / "section_12_comparison_exceptions.csv")
print(VALIDATION_DIR / "section_12_validation.csv")
print(METADATA_DIR / "section_12_summary.json")

assert structural_failed == 0, (
    "Section 12 structural validation failed. "
    "Inspect section_12_validation.csv."
)

NOTEBOOK 20 — SECTION 12: MODEL-LEVEL COMPARISON
Integrated measurements: 0
Model comparison rows: 0
Baseline comparison rows: 0
Ablation comparison rows: 0
Models observed: 0
Ablation variants observed: 0
Datasets observed: 0
Metrics observed: 0

Empirical evidence status: MISSING_EMPIRICAL_EVIDENCE
Structural validation passed: 8
Structural validation failed: 0
Section 12 status: COMPLETED_WITH_MISSING_EVIDENCE

Saved:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/model_level_comparison.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_12_comparison_exceptions.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_12_validation.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/metadata/section_12_summary.json


In [30]:

# ============================================================
# SECTION 13 — DATASET-LEVEL COMPARISON
# ============================================================

if not all_metric_results.empty:

    dataset_comparison = (
        all_metric_results
        .groupby(
            ["dataset", "metric", "metric_group"],
            dropna=False
        )["value"]
        .agg(
            n="count",
            mean="mean",
            std="std",
            median="median",
            minimum="min",
            maximum="max"
        )
        .reset_index()
    )

else:

    dataset_comparison = pd.DataFrame()

dataset_comparison.to_csv(
    TABLES_DIR / "dataset_level_comparison.csv",
    index=False
)

print("Dataset comparison rows:", len(dataset_comparison))


Dataset comparison rows: 0


In [31]:

# ============================================================
# SECTION 14 — METRIC-LEVEL COMPARISON
# ============================================================

if not all_metric_results.empty:

    metric_comparison = (
        all_metric_results
        .groupby(
            ["metric_group", "metric"],
            dropna=False
        )["value"]
        .agg(
            n="count",
            mean="mean",
            std="std",
            median="median",
            minimum="min",
            maximum="max"
        )
        .reset_index()
    )

else:

    metric_comparison = pd.DataFrame()

metric_comparison.to_csv(
    TABLES_DIR / "metric_level_comparison.csv",
    index=False
)

print("Metric comparison rows:", len(metric_comparison))


Metric comparison rows: 0


In [33]:
# ============================================================
# SECTION 15 — OVERALL PERFORMANCE RANKING
# ============================================================

# ------------------------------------------------------------
# EXPLICIT METRIC DIRECTIONS
# ------------------------------------------------------------

DIRECTION_MAP = {
    metric: "higher"
    for metric in HIGHER_IS_BETTER
}

DIRECTION_MAP.update({
    metric: "lower"
    for metric in LOWER_IS_BETTER
})

# Exclude metrics whose direction has not been confirmed.
UNCONFIRMED_DIRECTIONS = set(
    globals().get("UNCONFIRMED_METRIC_DIRECTIONS", [])
)

for metric in UNCONFIRMED_DIRECTIONS:
    DIRECTION_MAP.pop(metric, None)

RANKING_KEY_COLUMNS = [
    "dataset",
    "comparison_type",
    "comparison_entity",
    "model",
    "variant",
]

ranking_source = all_metric_results.copy()

# ------------------------------------------------------------
# NORMALIZE RANKING INPUT
# ------------------------------------------------------------

if not ranking_source.empty:

    for column in [
        "dataset",
        "model",
        "variant",
        "metric",
        "metric_group",
        "comparison_type",
        "comparison_entity",
    ]:
        if column not in ranking_source.columns:
            ranking_source[column] = np.nan

    ranking_source["dataset"] = (
        ranking_source["dataset"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    ranking_source["model"] = (
        ranking_source["model"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    ranking_source["variant"] = (
        ranking_source["variant"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    ranking_source["value"] = pd.to_numeric(
        ranking_source["value"],
        errors="coerce"
    )

    ranking_source["direction"] = (
        ranking_source["metric"].map(DIRECTION_MAP)
    )

    # Identify baseline models and ablation variants.
    ranking_source["comparison_type"] = np.where(
        ranking_source["variant"].notna(),
        "ablation",
        np.where(
            ranking_source["model"].notna(),
            "baseline",
            "unclassified"
        )
    )

    ranking_source["comparison_entity"] = np.where(
        ranking_source["variant"].notna(),
        ranking_source["variant"],
        ranking_source["model"]
    )

    ranking_source = ranking_source.dropna(
        subset=[
            "dataset",
            "metric",
            "value",
            "direction",
            "comparison_entity",
        ]
    )

    ranking_source = ranking_source[
        np.isfinite(ranking_source["value"])
    ].copy()

else:

    ranking_source = pd.DataFrame()

# ------------------------------------------------------------
# METRIC-SPECIFIC RANKING
# ------------------------------------------------------------

RANKING_SUMMARY_COLUMNS = [
    "dataset",
    "comparison_type",
    "comparison_entity",
    "model",
    "variant",
    "metric",
    "metric_group",
    "direction",
    "mean_value",
    "rank",
]

OVERALL_RANKING_COLUMNS = [
    "dataset",
    "comparison_type",
    "comparison_entity",
    "model",
    "variant",
    "metrics_ranked",
    "mean_rank",
    "median_rank",
    "overall_rank",
]

if not ranking_source.empty:

    ranking_summary = (
        ranking_source
        .groupby(
            RANKING_KEY_COLUMNS
            + ["metric", "metric_group", "direction"],
            dropna=False
        )["value"]
        .mean()
        .reset_index(name="mean_value")
    )

    ranking_summary["rank"] = np.nan

    # Rank only within the same dataset and metric.
    for (dataset, metric), indices in (
        ranking_summary.groupby(
            ["dataset", "metric"]
        ).groups.items()
    ):

        subset = ranking_summary.loc[indices]

        directions = subset["direction"].dropna().unique()

        if len(directions) != 1:
            continue

        ascending = directions[0] == "lower"

        ranking_summary.loc[indices, "rank"] = (
            subset["mean_value"].rank(
                ascending=ascending,
                method="average"
            )
        )

    # --------------------------------------------------------
    # OVERALL RANKING
    # --------------------------------------------------------

    valid_ranks = ranking_summary.dropna(
        subset=["rank"]
    ).copy()

    if not valid_ranks.empty:

        overall_ranking = (
            valid_ranks
            .groupby(
                RANKING_KEY_COLUMNS,
                dropna=False
            )["rank"]
            .agg(
                metrics_ranked="count",
                mean_rank="mean",
                median_rank="median"
            )
            .reset_index()
        )

        overall_ranking["overall_rank"] = (
            overall_ranking
            .groupby(
                ["dataset", "comparison_type"]
            )["mean_rank"]
            .rank(
                method="average",
                ascending=True
            )
        )

        overall_ranking = overall_ranking.sort_values(
            [
                "dataset",
                "comparison_type",
                "overall_rank",
            ]
        )

    else:

        overall_ranking = pd.DataFrame(
            columns=OVERALL_RANKING_COLUMNS
        )

else:

    ranking_summary = pd.DataFrame(
        columns=RANKING_SUMMARY_COLUMNS
    )

    overall_ranking = pd.DataFrame(
        columns=OVERALL_RANKING_COLUMNS
    )

# ------------------------------------------------------------
# SAVE RANKING OUTPUTS
# ------------------------------------------------------------

ranking_summary.to_csv(
    TABLES_DIR / "metric_specific_model_ranks.csv",
    index=False
)

overall_ranking.to_csv(
    TABLES_DIR / "overall_performance_ranking.csv",
    index=False
)

# ------------------------------------------------------------
# EVIDENCE AND DIRECTION COVERAGE
# ------------------------------------------------------------

unconfirmed_metrics_present = sorted(
    set(
        all_metric_results["metric"].dropna().astype(str)
    ).intersection(UNCONFIRMED_DIRECTIONS)
) if not all_metric_results.empty else []

rankable_metrics = (
    sorted(ranking_source["metric"].dropna().unique().tolist())
    if not ranking_source.empty
    else []
)

ranking_evidence_status = (
    "AVAILABLE"
    if len(overall_ranking) > 0
    else "MISSING_EMPIRICAL_EVIDENCE"
)

# ------------------------------------------------------------
# STRUCTURAL VALIDATION
# ------------------------------------------------------------

section_15_checks = {
    "metric_rank_schema_valid":
        list(ranking_summary.columns)
        == RANKING_SUMMARY_COLUMNS,

    "overall_rank_schema_valid":
        list(overall_ranking.columns)
        == OVERALL_RANKING_COLUMNS,

    "metric_ranking_file_exists":
        (TABLES_DIR / "metric_specific_model_ranks.csv").exists(),

    "overall_ranking_file_exists":
        (TABLES_DIR / "overall_performance_ranking.csv").exists(),

    "all_rank_values_finite":
        (
            ranking_summary.empty
            or np.isfinite(
                ranking_summary["rank"].dropna()
            ).all()
        ),

    "overall_ranks_positive":
        (
            overall_ranking.empty
            or (overall_ranking["overall_rank"] > 0).all()
        ),

    "only_confirmed_metric_directions":
        (
            ranking_summary.empty
            or ranking_summary["metric"].isin(
                DIRECTION_MAP
            ).all()
        ),

    "baseline_and_ablation_types_valid":
        (
            overall_ranking.empty
            or set(
                overall_ranking["comparison_type"].dropna().unique()
            ).issubset({"baseline", "ablation"})
        ),
}

section_15_validation = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "FAIL",
    }
    for name, passed in section_15_checks.items()
])

section_15_validation.to_csv(
    VALIDATION_DIR / "section_15_validation.csv",
    index=False
)

structural_passed = int(
    (section_15_validation["status"] == "PASS").sum()
)

structural_failed = int(
    (section_15_validation["status"] == "FAIL").sum()
)

section_15_status = (
    "COMPLETED_WITH_EMPIRICAL_EVIDENCE"
    if structural_failed == 0 and len(overall_ranking) > 0
    else (
        "COMPLETED_WITH_MISSING_EVIDENCE"
        if structural_failed == 0
        else "FAILED"
    )
)

# ------------------------------------------------------------
# SAVE SECTION SUMMARY
# ------------------------------------------------------------

section_15_summary = {
    "section": 15,
    "section_name": "Overall Performance Ranking",
    "input_measurements": int(len(all_metric_results)),
    "rankable_measurements": int(len(ranking_source)),
    "ranked_metric_records": int(len(ranking_summary)),
    "overall_ranking_records": int(len(overall_ranking)),
    "rankable_metrics": rankable_metrics,
    "unconfirmed_direction_metrics": unconfirmed_metrics_present,
    "empirical_evidence_status": ranking_evidence_status,
    "structural_checks_passed": structural_passed,
    "structural_checks_failed": structural_failed,
    "section_status": section_15_status,
}

with open(
    METADATA_DIR / "section_15_summary.json",
    "w"
) as f:
    json.dump(section_15_summary, f, indent=4)

# ------------------------------------------------------------
# COMPLETION SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("NOTEBOOK 20 — SECTION 15: OVERALL PERFORMANCE RANKING")
print("=" * 65)

print("Input measurements:", len(all_metric_results))
print("Rankable measurements:", len(ranking_source))
print("Ranked metric records:", len(ranking_summary))
print("Overall ranking records:", len(overall_ranking))
print("Rankable metrics:", rankable_metrics)
print("Unconfirmed direction metrics:", unconfirmed_metrics_present)

print("\nEmpirical evidence status:", ranking_evidence_status)
print("Structural validation passed:", structural_passed)
print("Structural validation failed:", structural_failed)
print("Section 15 status:", section_15_status)

print("\nSaved:")
print(TABLES_DIR / "metric_specific_model_ranks.csv")
print(TABLES_DIR / "overall_performance_ranking.csv")
print(VALIDATION_DIR / "section_15_validation.csv")
print(METADATA_DIR / "section_15_summary.json")

assert structural_failed == 0, (
    "Section 15 structural validation failed. "
    "Inspect section_15_validation.csv."
)

NOTEBOOK 20 — SECTION 15: OVERALL PERFORMANCE RANKING
Input measurements: 0
Rankable measurements: 0
Ranked metric records: 0
Overall ranking records: 0
Rankable metrics: []
Unconfirmed direction metrics: []

Empirical evidence status: MISSING_EMPIRICAL_EVIDENCE
Structural validation passed: 8
Structural validation failed: 0
Section 15 status: COMPLETED_WITH_MISSING_EVIDENCE

Saved:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/metric_specific_model_ranks.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/overall_performance_ranking.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_15_validation.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/metadata/section_15_summary.json


In [35]:
# ============================================================
# SECTION 16 — PRIVACY–UTILITY RANKING
# ============================================================

PRIVACY_UTILITY_KEYS = [
    "dataset",
    "comparison_type",
    "comparison_entity",
    "model",
    "variant",
]

# ------------------------------------------------------------
# HELPER: AGGREGATE ONE EXACT METRIC
# ------------------------------------------------------------

def aggregate_pu_metric(frame, metric, output_name):

    output_columns = PRIVACY_UTILITY_KEYS + [output_name]

    if not isinstance(frame, pd.DataFrame) or frame.empty:
        return pd.DataFrame(columns=output_columns)

    required = ["dataset", "metric", "value"]

    if not set(required).issubset(frame.columns):
        return pd.DataFrame(columns=output_columns)

    subset = frame[
        frame["metric"].astype(str).str.lower() == metric.lower()
    ].copy()

    if subset.empty:
        return pd.DataFrame(columns=output_columns)

    for column in ["model", "variant"]:
        if column not in subset.columns:
            subset[column] = np.nan

    subset["dataset"] = (
        subset["dataset"].astype("string").str.strip().str.lower()
    )

    subset["model"] = (
        subset["model"].astype("string").str.strip().str.lower()
    )

    subset["variant"] = (
        subset["variant"].astype("string").str.strip().str.lower()
    )

    subset["value"] = pd.to_numeric(
        subset["value"], errors="coerce"
    )

    subset = subset[
        subset["value"].notna()
        & np.isfinite(subset["value"])
        & subset["dataset"].notna()
    ].copy()

    if subset.empty:
        return pd.DataFrame(columns=output_columns)

    subset["comparison_type"] = np.where(
        subset["variant"].notna(),
        "ablation",
        np.where(subset["model"].notna(), "baseline", "unclassified")
    )

    subset["comparison_entity"] = np.where(
        subset["variant"].notna(),
        subset["variant"],
        subset["model"]
    )

    subset = subset[
        subset["comparison_entity"].notna()
    ].copy()

    if subset.empty:
        return pd.DataFrame(columns=output_columns)

    result = (
        subset
        .groupby(
            PRIVACY_UTILITY_KEYS,
            dropna=False
        )["value"]
        .mean()
        .reset_index(name=output_name)
    )

    return result


# ------------------------------------------------------------
# 1. UTILITY COMPONENT
# ------------------------------------------------------------

# Use only an explicitly defined aggregate utility score.
# Do not substitute Macro-F1 or another metric automatically.

utility_component = aggregate_pu_metric(
    utility_results,
    "utility_score",
    "utility_measure"
)

# ------------------------------------------------------------
# 2. EMPIRICAL PRIVACY COMPONENT
# ------------------------------------------------------------

EMPIRICAL_RISK_METRICS = [
    "nearest_neighbor_disclosure",
    "reidentification_risk",
    "membership_inference_risk",
    "membership_inference_auc",
    "membership_inference_advantage",
    "attribute_inference_risk",
    "singling_out_risk",
]

empirical_risk_frames = []

for metric in EMPIRICAL_RISK_METRICS:

    component = aggregate_pu_metric(
        privacy_results,
        metric,
        "empirical_risk"
    )

    if not component.empty:
        component["risk_metric"] = metric
        empirical_risk_frames.append(component)

if empirical_risk_frames:

    empirical_risk_long = pd.concat(
        empirical_risk_frames,
        ignore_index=True
    )

    empirical_risk = (
        empirical_risk_long
        .groupby(
            PRIVACY_UTILITY_KEYS,
            dropna=False
        )["empirical_risk"]
        .mean()
        .reset_index()
    )

else:

    empirical_risk = pd.DataFrame(
        columns=PRIVACY_UTILITY_KEYS + ["empirical_risk"]
    )

# ------------------------------------------------------------
# 3. FORMAL DP COMPONENT
# ------------------------------------------------------------

DP_COMPONENT_COLUMNS = (
    PRIVACY_UTILITY_KEYS
    + ["achieved_epsilon", "achieved_delta"]
)

if (
    isinstance(dp_results, pd.DataFrame)
    and not dp_results.empty
    and "verified_dp_evidence" in dp_results.columns
):

    verified_dp = dp_results[
        dp_results["verified_dp_evidence"].fillna(False)
    ].copy()

    if not verified_dp.empty:

        for column in ["model", "variant"]:
            if column not in verified_dp.columns:
                verified_dp[column] = np.nan

        verified_dp["dataset"] = (
            verified_dp["dataset"]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        verified_dp["model"] = (
            verified_dp["model"]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        verified_dp["variant"] = (
            verified_dp["variant"]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        verified_dp["achieved_epsilon"] = pd.to_numeric(
            verified_dp["achieved_epsilon"],
            errors="coerce"
        )

        verified_dp["achieved_delta"] = pd.to_numeric(
            verified_dp["achieved_delta"],
            errors="coerce"
        )

        verified_dp = verified_dp[
            verified_dp["achieved_epsilon"].notna()
            & verified_dp["achieved_delta"].notna()
            & np.isfinite(verified_dp["achieved_epsilon"])
            & np.isfinite(verified_dp["achieved_delta"])
            & (verified_dp["achieved_epsilon"] > 0)
            & (verified_dp["achieved_delta"] > 0)
            & (verified_dp["achieved_delta"] < 1)
        ].copy()

        verified_dp["comparison_type"] = np.where(
            verified_dp["variant"].notna(),
            "ablation",
            np.where(
                verified_dp["model"].notna(),
                "baseline",
                "unclassified"
            )
        )

        verified_dp["comparison_entity"] = np.where(
            verified_dp["variant"].notna(),
            verified_dp["variant"],
            verified_dp["model"]
        )

        verified_dp = verified_dp[
            verified_dp["comparison_entity"].notna()
        ].copy()

        formal_dp_component = (
            verified_dp
            .groupby(
                PRIVACY_UTILITY_KEYS,
                dropna=False
            )
            .agg(
                achieved_epsilon=("achieved_epsilon", "mean"),
                achieved_delta=("achieved_delta", "mean")
            )
            .reset_index()
        )

    else:

        formal_dp_component = pd.DataFrame(
            columns=DP_COMPONENT_COLUMNS
        )

else:

    formal_dp_component = pd.DataFrame(
        columns=DP_COMPONENT_COLUMNS
    )

# ------------------------------------------------------------
# 4. BUILD SEPARATE COMPARISON TABLES
# ------------------------------------------------------------

def merge_available_components(components):

    available = [
        component
        for component in components
        if isinstance(component, pd.DataFrame)
        and not component.empty
    ]

    if not available:
        return pd.DataFrame(
            columns=PRIVACY_UTILITY_KEYS
        )

    result = available[0].copy()

    for component in available[1:]:

        result = result.merge(
            component,
            on=PRIVACY_UTILITY_KEYS,
            how="outer",
            validate="one_to_one"
        )

    return result


# Utility and empirical disclosure risk comparison.
empirical_pu_comparison = merge_available_components([
    utility_component,
    empirical_risk,
])

# Formal DP epsilon/delta remains a separate table.
formal_dp_pu_comparison = formal_dp_component.copy()

# ------------------------------------------------------------
# 5. RANK ONLY WHEN COMPARABLE EVIDENCE EXISTS
# ------------------------------------------------------------

def add_pu_ranks(frame, rank_directions):

    result = frame.copy()

    if result.empty:
        result["dimensions_available"] = pd.Series(dtype=int)
        result["mean_rank"] = pd.Series(dtype=float)
        result["ranking_status"] = pd.Series(dtype=str)
        return result

    rank_columns = []

    for metric, ascending in rank_directions.items():

        if metric not in result.columns:
            continue

        rank_column = metric + "_rank"
        result[rank_column] = np.nan

        for dataset, indices in result.groupby(
            "dataset"
        ).groups.items():

            values = result.loc[indices, metric]

            # Require at least two distinct observed values.
            if values.nunique(dropna=True) < 2:
                continue

            result.loc[indices, rank_column] = values.rank(
                ascending=ascending,
                method="average"
            )

        rank_columns.append(rank_column)

    if rank_columns:

        result["dimensions_available"] = (
            result[rank_columns].notna().sum(axis=1)
        )

        result["mean_rank"] = (
            result[rank_columns].mean(axis=1)
        )

    else:

        result["dimensions_available"] = 0
        result["mean_rank"] = np.nan

    result["ranking_status"] = np.where(
        result["dimensions_available"] >= 2,
        "PARTIAL_COMPARATIVE_RANKING",
        "INSUFFICIENT_COMPARABLE_EVIDENCE"
    )

    return result


# Higher utility is better; lower empirical risk is better.
empirical_pu_ranking = add_pu_ranks(
    empirical_pu_comparison,
    {
        "utility_measure": False,
        "empirical_risk": True,
    }
)

# Lower epsilon is not automatically a complete privacy ranking:
# epsilon is meaningful only with its corresponding delta and
# verified accountant/privacy boundary. Keep formal DP separate.
formal_dp_pu_ranking = formal_dp_pu_comparison.copy()

# ------------------------------------------------------------
# 6. SAVE OUTPUTS
# ------------------------------------------------------------

empirical_pu_comparison.to_csv(
    TABLES_DIR / "privacy_utility_empirical_comparison.csv",
    index=False
)

empirical_pu_ranking.to_csv(
    TABLES_DIR / "privacy_utility_empirical_ranking.csv",
    index=False
)

formal_dp_pu_comparison.to_csv(
    TABLES_DIR / "privacy_utility_formal_dp_comparison.csv",
    index=False
)

# ------------------------------------------------------------
# 7. STRUCTURAL VALIDATION
# ------------------------------------------------------------

section_16_checks = {
    "empirical_comparison_file_exists":
        (
            TABLES_DIR / "privacy_utility_empirical_comparison.csv"
        ).exists(),

    "empirical_ranking_file_exists":
        (
            TABLES_DIR / "privacy_utility_empirical_ranking.csv"
        ).exists(),

    "formal_dp_comparison_file_exists":
        (
            TABLES_DIR / "privacy_utility_formal_dp_comparison.csv"
        ).exists(),

    "empirical_ranking_has_no_infinite_ranks":
        (
            empirical_pu_ranking.empty
            or np.isfinite(
                empirical_pu_ranking["mean_rank"].dropna()
            ).all()
        ),

    "formal_dp_rows_are_verified":
        (
            formal_dp_pu_comparison.empty
            or (
                isinstance(dp_results, pd.DataFrame)
                and not dp_results.empty
                and dp_results["verified_dp_evidence"].fillna(False).any()
            )
        ),

    "empirical_and_formal_dp_are_separate":
        (
            "achieved_epsilon" not in empirical_pu_comparison.columns
            and "empirical_risk" not in formal_dp_pu_comparison.columns
        ),
}

section_16_validation = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "FAIL",
    }
    for name, passed in section_16_checks.items()
])

section_16_validation.to_csv(
    VALIDATION_DIR / "section_16_validation.csv",
    index=False
)

structural_passed = int(
    (section_16_validation["status"] == "PASS").sum()
)

structural_failed = int(
    (section_16_validation["status"] == "FAIL").sum()
)

empirical_evidence_available = (
    not empirical_pu_comparison.empty
)

formal_dp_evidence_available = (
    not formal_dp_pu_comparison.empty
)

section_16_status = (
    "COMPLETED_WITH_EMPIRICAL_EVIDENCE"
    if structural_failed == 0 and empirical_evidence_available
    else (
        "COMPLETED_WITH_MISSING_EVIDENCE"
        if structural_failed == 0
        else "FAILED"
    )
)

section_16_summary = {
    "section": 16,
    "section_name": "Privacy–Utility Ranking",
    "utility_records": int(len(utility_component)),
    "empirical_privacy_records": int(len(empirical_risk)),
    "formal_dp_records": int(len(formal_dp_pu_comparison)),
    "empirical_comparison_records": int(len(empirical_pu_comparison)),
    "empirical_ranking_records": int(len(empirical_pu_ranking)),
    "formal_dp_evidence_available": bool(formal_dp_evidence_available),
    "structural_checks_passed": structural_passed,
    "structural_checks_failed": structural_failed,
    "section_status": section_16_status,
}

with open(
    METADATA_DIR / "section_16_summary.json",
    "w"
) as f:
    json.dump(section_16_summary, f, indent=4)

# ------------------------------------------------------------
# 8. COMPLETION SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("NOTEBOOK 20 — SECTION 16: PRIVACY–UTILITY RANKING")
print("=" * 65)

print("Utility records:", len(utility_component))
print("Empirical privacy records:", len(empirical_risk))
print("Formal DP records:", len(formal_dp_pu_comparison))
print("Empirical comparison records:", len(empirical_pu_comparison))
print("Empirical ranking records:", len(empirical_pu_ranking))

print("Formal DP evidence available:", formal_dp_evidence_available)

print("\nStructural validation passed:", structural_passed)
print("Structural validation failed:", structural_failed)
print("Section 16 status:", section_16_status)

print("\nSaved:")
print(TABLES_DIR / "privacy_utility_empirical_comparison.csv")
print(TABLES_DIR / "privacy_utility_empirical_ranking.csv")
print(TABLES_DIR / "privacy_utility_formal_dp_comparison.csv")
print(VALIDATION_DIR / "section_16_validation.csv")
print(METADATA_DIR / "section_16_summary.json")

assert structural_failed == 0, (
    "Section 16 structural validation failed. "
    "Inspect section_16_validation.csv."
)

NOTEBOOK 20 — SECTION 16: PRIVACY–UTILITY RANKING
Utility records: 0
Empirical privacy records: 0
Formal DP records: 0
Empirical comparison records: 0
Empirical ranking records: 0
Formal DP evidence available: False

Structural validation passed: 6
Structural validation failed: 0
Section 16 status: COMPLETED_WITH_MISSING_EVIDENCE

Saved:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/privacy_utility_empirical_comparison.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/privacy_utility_empirical_ranking.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/privacy_utility_formal_dp_comparison.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_16_validation.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/metadata/section_16_summary.json


In [37]:
# ============================================================
# SECTION 17 — OVERALL EVIDENCE SUMMARY
# ============================================================

# ------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------

def count_valid_numeric(frame, columns):
    if not isinstance(frame, pd.DataFrame) or frame.empty:
        return 0

    count = 0

    for column in columns:
        if column not in frame.columns:
            continue

        values = pd.to_numeric(
            frame[column],
            errors="coerce"
        )

        count += int(
            (values.notna() & np.isfinite(values)).sum()
        )

    return count


def count_source_artifacts(frame):
    if not isinstance(frame, pd.DataFrame) or frame.empty:
        return 0

    if "source_path" in frame.columns:
        return int(frame["source_path"].nunique())

    if "source_path" not in frame.columns and "source_filename" in frame.columns:
        return int(frame["source_filename"].nunique())

    return 0


def evidence_record(name, frame, valid_measurements=None,
                    evidence_note=None):

    if not isinstance(frame, pd.DataFrame):
        frame = pd.DataFrame()

    records = len(frame)

    if valid_measurements is None:
        valid_measurements = records

    if records == 0:
        evidence_status = "MISSING"
    elif valid_measurements == 0:
        evidence_status = "RECORDS_PRESENT_BUT_NO_VALID_MEASUREMENTS"
    else:
        evidence_status = "EMPIRICAL_MEASUREMENTS_AVAILABLE"

    return {
        "evidence_category": name,
        "records": records,
        "valid_measurements": int(valid_measurements),
        "datasets": (
            frame["dataset"].nunique()
            if "dataset" in frame.columns else 0
        ),
        "models": (
            frame["model"].nunique()
            if "model" in frame.columns else 0
        ),
        "variants": (
            frame["variant"].nunique()
            if "variant" in frame.columns else 0
        ),
        "source_artifacts": count_source_artifacts(frame),
        "evidence_status": evidence_status,
        "evidence_note": evidence_note or "",
    }


# ------------------------------------------------------------
# 1. FIDELITY EVIDENCE
# ------------------------------------------------------------

fidelity_valid = count_valid_numeric(
    fidelity_results,
    ["value"]
)

# ------------------------------------------------------------
# 2. ML UTILITY EVIDENCE
# ------------------------------------------------------------

utility_valid = count_valid_numeric(
    utility_results,
    ["value"]
)

# ------------------------------------------------------------
# 3. EMPIRICAL PRIVACY EVIDENCE
# ------------------------------------------------------------

privacy_valid = count_valid_numeric(
    privacy_results,
    ["value"]
)

# ------------------------------------------------------------
# 4. FORMAL DIFFERENTIAL PRIVACY EVIDENCE
# ------------------------------------------------------------

dp_valid = 0

if (
    isinstance(dp_results, pd.DataFrame)
    and not dp_results.empty
    and "verified_dp_evidence" in dp_results.columns
):
    dp_valid = int(
        dp_results["verified_dp_evidence"]
        .fillna(False)
        .astype(bool)
        .sum()
    )

# ------------------------------------------------------------
# 5. ABLATION EVIDENCE
# ------------------------------------------------------------

ablation_valid = count_valid_numeric(
    ablation_results,
    ["value"]
)

# ------------------------------------------------------------
# 6. TRADE-OFF EVIDENCE
# ------------------------------------------------------------

tradeoff_score_columns = [
    "privacy_score_mean",
    "fidelity_score_mean",
    "utility_score_mean",
    "tradeoff_score_mean",
]

tradeoff_valid = count_valid_numeric(
    tradeoff_results,
    tradeoff_score_columns
)

# ------------------------------------------------------------
# 7. SIGNIFICANCE EVIDENCE
# ------------------------------------------------------------

# Imported p-values alone do not prove that statistical tests
# were executed. Require actual test statistics and p-values.

significance_statistic_count = count_valid_numeric(
    significance_results,
    ["test_statistic", "statistic"]
)

significance_pvalue_count = count_valid_numeric(
    significance_results,
    ["p_value", "adjusted_p_value", "corrected_p_value"]
)

significance_effect_count = count_valid_numeric(
    significance_results,
    ["effect_size"]
)

significance_ci_count = count_valid_numeric(
    significance_results,
    ["ci_lower", "ci_upper"]
)

significance_valid = (
    min(
        significance_statistic_count,
        significance_pvalue_count
    )
    if (
        significance_statistic_count > 0
        and significance_pvalue_count > 0
    )
    else 0
)

# ------------------------------------------------------------
# 8. COMPUTATIONAL EVIDENCE
# ------------------------------------------------------------

computational_valid = count_valid_numeric(
    computational_results,
    ["value"]
)

# ------------------------------------------------------------
# 9. BUILD MASTER EVIDENCE SUMMARY
# ------------------------------------------------------------

evidence_summary = pd.DataFrame([

    evidence_record(
        "Fidelity",
        fidelity_results,
        fidelity_valid,
        "Requires valid empirical fidelity measurements."
    ),

    evidence_record(
        "ML utility",
        utility_results,
        utility_valid,
        "Requires valid downstream utility/TSTR measurements."
    ),

    evidence_record(
        "Empirical privacy",
        privacy_results,
        privacy_valid,
        "Requires actual disclosure or inference attack measurements."
    ),

    evidence_record(
        "Formal DP",
        dp_results,
        dp_valid,
        "Requires verified formal DP evidence, achieved epsilon, delta and accountant evidence."
    ),

    evidence_record(
        "Ablation",
        ablation_results,
        ablation_valid,
        "Requires valid measurements for the registered ablation variants."
    ),

    evidence_record(
        "Trade-off",
        tradeoff_results,
        tradeoff_valid,
        "Rows without valid component scores do not establish empirical trade-off evidence."
    ),

    evidence_record(
        "Significance",
        significance_results,
        significance_valid,
        "Imported p-values without test statistics do not establish executed statistical tests."
    ),

    evidence_record(
        "Computational",
        computational_results,
        computational_valid,
        "Requires actual runtime, memory or other computational measurements."
    ),

])

# ------------------------------------------------------------
# 10. ADD OVERALL EVIDENCE CLASSIFICATION
# ------------------------------------------------------------

evidence_summary["evidence_available"] = (
    evidence_summary["valid_measurements"] > 0
)

evidence_summary["structural_records_present"] = (
    evidence_summary["records"] > 0
)

evidence_summary["missing_empirical_evidence"] = (
    evidence_summary["valid_measurements"] == 0
)

# ------------------------------------------------------------
# 11. SAVE MASTER EVIDENCE SUMMARY
# ------------------------------------------------------------

evidence_summary.to_csv(
    TABLES_DIR / "overall_evidence_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 12. VALIDATION
# ------------------------------------------------------------

section_17_checks = {
    "all_evidence_categories_present":
        len(evidence_summary) == 8,

    "no_negative_record_counts":
        (evidence_summary["records"] >= 0).all(),

    "no_negative_measurement_counts":
        (evidence_summary["valid_measurements"] >= 0).all(),

    "valid_measurements_do_not_exceed_records":
        (
            evidence_summary["valid_measurements"]
            <= evidence_summary["records"]
        ).all(),

    "missing_evidence_classification_correct":
        (
            evidence_summary.loc[
                evidence_summary["valid_measurements"] == 0,
                "evidence_status"
            ].isin([
                "MISSING",
                "RECORDS_PRESENT_BUT_NO_VALID_MEASUREMENTS"
            ]).all()
        ),

    "tradeoff_not_misclassified":
        (
            tradeoff_valid > 0
            or evidence_summary.loc[
                evidence_summary["evidence_category"] == "Trade-off",
                "evidence_available"
            ].eq(False).all()
        ),

    "significance_not_misclassified":
        (
            significance_valid > 0
            or evidence_summary.loc[
                evidence_summary["evidence_category"] == "Significance",
                "evidence_available"
            ].eq(False).all()
        ),

    "summary_file_exists":
        (TABLES_DIR / "overall_evidence_summary.csv").exists(),
}

section_17_validation = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "FAIL"
    }
    for name, passed in section_17_checks.items()
])

section_17_validation.to_csv(
    VALIDATION_DIR / "section_17_validation.csv",
    index=False
)

structural_passed = int(
    (section_17_validation["status"] == "PASS").sum()
)

structural_failed = int(
    (section_17_validation["status"] == "FAIL").sum()
)

# ------------------------------------------------------------
# 13. SAVE SECTION SUMMARY
# ------------------------------------------------------------

available_categories = int(
    evidence_summary["evidence_available"].sum()
)

missing_categories = int(
    evidence_summary["missing_empirical_evidence"].sum()
)

section_17_status = (
    "COMPLETED_WITH_EMPIRICAL_EVIDENCE"
    if structural_failed == 0 and available_categories > 0
    else (
        "COMPLETED_WITH_MISSING_EVIDENCE"
        if structural_failed == 0
        else "FAILED"
    )
)

section_17_summary = {
    "section": 17,
    "section_name": "Overall Evidence Summary",
    "evidence_categories": int(len(evidence_summary)),
    "categories_with_empirical_evidence": available_categories,
    "categories_missing_empirical_evidence": missing_categories,
    "total_structural_records": int(
        evidence_summary["records"].sum()
    ),
    "total_valid_measurements": int(
        evidence_summary["valid_measurements"].sum()
    ),
    "structural_checks_passed": structural_passed,
    "structural_checks_failed": structural_failed,
    "section_status": section_17_status,
}

with open(
    METADATA_DIR / "section_17_summary.json",
    "w"
) as f:
    json.dump(section_17_summary, f, indent=4)

# ------------------------------------------------------------
# 14. COMPLETION SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("NOTEBOOK 20 — SECTION 17: OVERALL EVIDENCE SUMMARY")
print("=" * 65)

print(evidence_summary.to_string(index=False))

print("\nEvidence categories:", len(evidence_summary))
print("Categories with empirical evidence:", available_categories)
print("Categories missing empirical evidence:", missing_categories)
print(
    "Total structural records:",
    int(evidence_summary["records"].sum())
)
print(
    "Total valid measurements:",
    int(evidence_summary["valid_measurements"].sum())
)

print("\nStructural validation passed:", structural_passed)
print("Structural validation failed:", structural_failed)
print("Section 17 status:", section_17_status)

print("\nSaved:")
print(TABLES_DIR / "overall_evidence_summary.csv")
print(VALIDATION_DIR / "section_17_validation.csv")
print(METADATA_DIR / "section_17_summary.json")

assert structural_failed == 0, (
    "Section 17 structural validation failed. "
    "Inspect section_17_validation.csv."
)

NOTEBOOK 20 — SECTION 17: OVERALL EVIDENCE SUMMARY
evidence_category  records  valid_measurements  datasets  models  variants  source_artifacts                           evidence_status                                                                          evidence_note  evidence_available  structural_records_present  missing_empirical_evidence
         Fidelity        0                   0         0       0         0                 0                                   MISSING                                        Requires valid empirical fidelity measurements.               False                       False                        True
       ML utility        0                   0         0       0         0                 0                                   MISSING                                   Requires valid downstream utility/TSTR measurements.               False                       False                        True
Empirical privacy        0                   0         

In [40]:
# ============================================================
# NOTEBOOK 20 — SECTION 18
# FAILURE / EXCEPTION SUMMARY
# ============================================================

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 80)
print("NOTEBOOK 20 — SECTION 18: FAILURE / EXCEPTION SUMMARY")
print("=" * 80)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

TABLES_DIR = Path(TABLES_DIR)
VALIDATION_DIR = Path(VALIDATION_DIR)
METADATA_DIR = Path(METADATA_DIR)

for directory in [
    TABLES_DIR,
    VALIDATION_DIR,
    METADATA_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

STATUS_COLUMNS = [
    "status",
    "validation_status",
    "execution_status",
    "evidence_status"
]

FAILURE_PATTERN = re.compile(
    r"^(?:"
    r"FAIL(?:ED|URE)?|"
    r"ERROR(?:ED)?|"
    r"EXCEPTION|"
    r"BLOCKED|"
    r"INCOMPLETE"
    r")(?:$|[_\s:;.,(\/-])",
    flags=re.IGNORECASE
)

MISSING_EVIDENCE_PATTERN = re.compile(
    r"^(?:"
    r"MISSING|"
    r"NO_EMPIRICAL|"
    r"NO_VALID|"
    r"NO_EXECUTED|"
    r"COMPLETED_WITH_MISSING_EVIDENCE|"
    r"RECORDS_PRESENT_BUT_NO_VALID_MEASUREMENTS"
    r")(?:$|[_\s:;.,(\/-])",
    flags=re.IGNORECASE
)

NEGATIVE_STATUS_PREFIXES = (
    "NOT_",
    "NON_",
    "WITHOUT_"
)

# ------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------

def get_artifact_frame(item):

    frame = item.get("normalized_data")

    if not isinstance(frame, pd.DataFrame):
        frame = item.get("data")

    if not isinstance(frame, pd.DataFrame):
        return pd.DataFrame()

    return frame.copy()


def get_artifact_source(item):

    source_path = item.get(
        "source_path",
        item.get("path", "")
    )

    source_notebook = item.get(
        "source_notebook",
        ""
    )

    return (
        str(source_path),
        str(source_notebook)
    )


def normalize_status(value):

    if pd.isna(value):
        return ""

    return str(value).strip().upper()


def classify_status(value):

    status = normalize_status(value)

    if not status:
        return None

    if status.startswith(
        NEGATIVE_STATUS_PREFIXES
    ):
        return None

    if FAILURE_PATTERN.match(status):
        return "EXECUTION_OR_VALIDATION_EXCEPTION"

    if MISSING_EVIDENCE_PATTERN.match(status):
        return "MISSING_EMPIRICAL_EVIDENCE"

    return None


def safe_json(value):

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        return float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, Path):
        return str(value)

    return str(value)


def save_csv(frame, path):

    frame.to_csv(
        path,
        index=False
    )

    return path.exists()


# ------------------------------------------------------------
# 3. COLLECT EXPLICIT FAILURE ARTIFACTS
# ------------------------------------------------------------

explicit_failure_frames = []
explicit_failure_inventory = []

for artifact_index, item in enumerate(
    artifact_records
):

    if str(
        item.get("artifact_type", "")
    ).strip().lower() != "failure":
        continue

    frame = get_artifact_frame(item)

    source_path, source_notebook = (
        get_artifact_source(item)
    )

    explicit_failure_inventory.append({
        "artifact_index": artifact_index,
        "source_path": source_path,
        "source_notebook": source_notebook,
        "source_rows": len(frame)
    })

    if frame.empty:
        continue

    frame = apply_aliases(frame)

    frame = frame.reset_index(drop=True)

    frame["source_path"] = source_path
    frame["source_notebook"] = source_notebook

    frame["source_artifact_index"] = (
        artifact_index
    )

    frame["source_row_index"] = np.arange(
        len(frame)
    )

    frame["source_record_id"] = [
        f"artifact_{artifact_index}_row_{row}"
        for row in range(len(frame))
    ]

    explicit_failure_frames.append(frame)


failure_summary = (
    pd.concat(
        explicit_failure_frames,
        ignore_index=True,
        sort=False
    )
    if explicit_failure_frames
    else pd.DataFrame(
        columns=[
            "source_path",
            "source_notebook",
            "source_artifact_index",
            "source_row_index",
            "source_record_id"
        ]
    )
)

explicit_failure_inventory_df = (
    pd.DataFrame(
        explicit_failure_inventory,
        columns=[
            "artifact_index",
            "source_path",
            "source_notebook",
            "source_rows"
        ]
    )
)

# ------------------------------------------------------------
# 4. DETECT STATUS EXCEPTIONS
# ------------------------------------------------------------

exception_detection_records = []

scanned_artifact_records = []

for artifact_index, item in enumerate(
    artifact_records
):

    frame = get_artifact_frame(item)

    source_path, source_notebook = (
        get_artifact_source(item)
    )

    if frame.empty:
        scanned_artifact_records.append({
            "artifact_index": artifact_index,
            "source_path": source_path,
            "source_notebook": source_notebook,
            "source_rows": len(frame),
            "status_columns_found": 0,
            "detected_status_events": 0
        })

        continue

    frame = frame.reset_index(drop=True)

    normalized_column_lookup = {
        str(column).strip().lower(): column
        for column in frame.columns
    }

    available_status_columns = [
        normalized_column_lookup[column]
        for column in STATUS_COLUMNS
        if column in normalized_column_lookup
    ]

    artifact_detection_count = 0

    for row_index in range(len(frame)):

        row = frame.iloc[row_index]

        source_record_id = (
            f"artifact_{artifact_index}_row_{row_index}"
        )

        for status_column in available_status_columns:

            status_value = normalize_status(
                row[status_column]
            )

            exception_category = (
                classify_status(status_value)
            )

            if exception_category is None:
                continue

            exception_detection_records.append({
                "source_record_id": source_record_id,
                "source_artifact_index": artifact_index,
                "source_row_index": row_index,
                "source_path": source_path,
                "source_notebook": source_notebook,
                "detected_status_column": str(
                    status_column
                ),
                "detected_status_value": status_value,
                "exception_category": exception_category
            })

            artifact_detection_count += 1

    scanned_artifact_records.append({
        "artifact_index": artifact_index,
        "source_path": source_path,
        "source_notebook": source_notebook,
        "source_rows": len(frame),
        "status_columns_found": len(
            available_status_columns
        ),
        "detected_status_events": (
            artifact_detection_count
        )
    })


EVENT_COLUMNS = [
    "source_record_id",
    "source_artifact_index",
    "source_row_index",
    "source_path",
    "source_notebook",
    "detected_status_column",
    "detected_status_value",
    "exception_category"
]

exception_events = pd.DataFrame(
    exception_detection_records,
    columns=EVENT_COLUMNS
)

scanned_artifact_inventory = pd.DataFrame(
    scanned_artifact_records
)

# ------------------------------------------------------------
# 5. CONSOLIDATE BY STABLE SOURCE RECORD ID
# ------------------------------------------------------------

CONSOLIDATED_COLUMNS = [
    "source_record_id",
    "source_artifact_index",
    "source_row_index",
    "source_path",
    "source_notebook",
    "detected_status_columns",
    "detected_status_values",
    "exception_categories",
    "status_event_count",
    "execution_or_validation_exception",
    "missing_empirical_evidence"
]

if not exception_events.empty:

    grouped = exception_events.groupby(
        "source_record_id",
        sort=False,
        dropna=False
    )

    consolidated_records = []

    for source_record_id, group in grouped:

        first = group.iloc[0]

        categories = sorted(
            set(
                group[
                    "exception_category"
                ].dropna().astype(str)
            )
        )

        status_columns = sorted(
            set(
                group[
                    "detected_status_column"
                ].dropna().astype(str)
            )
        )

        status_values = sorted(
            set(
                group[
                    "detected_status_value"
                ].dropna().astype(str)
            )
        )

        consolidated_records.append({
            "source_record_id": source_record_id,
            "source_artifact_index": int(
                first["source_artifact_index"]
            ),
            "source_row_index": int(
                first["source_row_index"]
            ),
            "source_path": first[
                "source_path"
            ],
            "source_notebook": first[
                "source_notebook"
            ],
            "detected_status_columns": (
                "; ".join(status_columns)
            ),
            "detected_status_values": (
                "; ".join(status_values)
            ),
            "exception_categories": (
                "; ".join(categories)
            ),
            "status_event_count": len(group),
            "execution_or_validation_exception": (
                "EXECUTION_OR_VALIDATION_EXCEPTION"
                in categories
            ),
            "missing_empirical_evidence": (
                "MISSING_EMPIRICAL_EVIDENCE"
                in categories
            )
        })

    detected_exceptions = pd.DataFrame(
        consolidated_records,
        columns=CONSOLIDATED_COLUMNS
    )

else:

    detected_exceptions = pd.DataFrame(
        columns=CONSOLIDATED_COLUMNS
    )


# ------------------------------------------------------------
# 6. EXCEPTION CATEGORY SUMMARY
# ------------------------------------------------------------

if not exception_events.empty:

    exception_category_summary = (
        exception_events
        .groupby(
            "exception_category",
            dropna=False
        )
        .agg(
            status_events=(
                "source_record_id",
                "size"
            ),
            unique_source_records=(
                "source_record_id",
                "nunique"
            ),
            source_artifacts=(
                "source_path",
                "nunique"
            )
        )
        .reset_index()
    )

else:

    exception_category_summary = pd.DataFrame(
        columns=[
            "exception_category",
            "status_events",
            "unique_source_records",
            "source_artifacts"
        ]
    )


# ------------------------------------------------------------
# 7. SOURCE-LEVEL EXCEPTION SUMMARY
# ------------------------------------------------------------

if not detected_exceptions.empty:

    source_exception_summary = (
        detected_exceptions
        .groupby(
            [
                "source_notebook",
                "source_path"
            ],
            dropna=False
        )
        .agg(
            unique_exception_records=(
                "source_record_id",
                "nunique"
            ),
            status_events=(
                "status_event_count",
                "sum"
            ),
            execution_or_validation_records=(
                "execution_or_validation_exception",
                "sum"
            ),
            missing_evidence_records=(
                "missing_empirical_evidence",
                "sum"
            )
        )
        .reset_index()
    )

else:

    source_exception_summary = pd.DataFrame(
        columns=[
            "source_notebook",
            "source_path",
            "unique_exception_records",
            "status_events",
            "execution_or_validation_records",
            "missing_evidence_records"
        ]
    )


# ------------------------------------------------------------
# 8. SAVE SECTION 18 ARTIFACTS
# ------------------------------------------------------------

output_paths = {
    "explicit_failures":
        TABLES_DIR / "failure_exception_records.csv",

    "explicit_failure_inventory":
        TABLES_DIR / "explicit_failure_artifact_inventory.csv",

    "exception_events":
        TABLES_DIR / "detected_exception_status_events.csv",

    "detected_exceptions":
        TABLES_DIR / "detected_incomplete_or_failed_records.csv",

    "exception_category_summary":
        TABLES_DIR / "exception_category_summary.csv",

    "source_exception_summary":
        TABLES_DIR / "source_exception_summary.csv",

    "scanned_artifact_inventory":
        TABLES_DIR / "section_18_scanned_artifact_inventory.csv"
}

save_csv(
    failure_summary,
    output_paths["explicit_failures"]
)

save_csv(
    explicit_failure_inventory_df,
    output_paths["explicit_failure_inventory"]
)

save_csv(
    exception_events,
    output_paths["exception_events"]
)

save_csv(
    detected_exceptions,
    output_paths["detected_exceptions"]
)

save_csv(
    exception_category_summary,
    output_paths["exception_category_summary"]
)

save_csv(
    source_exception_summary,
    output_paths["source_exception_summary"]
)

save_csv(
    scanned_artifact_inventory,
    output_paths["scanned_artifact_inventory"]
)


# ------------------------------------------------------------
# 9. STRUCTURAL VALIDATION
# ------------------------------------------------------------

section_18_checks = []

def section_18_check(
    check_name,
    condition
):

    section_18_checks.append({
        "check": check_name,
        "passed": bool(condition)
    })


section_18_check(
    "Artifact registry available",
    isinstance(artifact_records, list)
)

section_18_check(
    "Explicit failure table valid",
    isinstance(failure_summary, pd.DataFrame)
)

section_18_check(
    "Exception events table valid",
    isinstance(exception_events, pd.DataFrame)
)

section_18_check(
    "Consolidated exception table valid",
    isinstance(detected_exceptions, pd.DataFrame)
)

section_18_check(
    "Stable source record IDs are unique",
    detected_exceptions[
        "source_record_id"
    ].is_unique
)

section_18_check(
    "Consolidated records match unique event IDs",
    len(detected_exceptions)
    == exception_events[
        "source_record_id"
    ].nunique()
)

section_18_check(
    "Event counts reconcile",
    int(
        detected_exceptions[
            "status_event_count"
        ].sum()
    )
    == len(exception_events)
)

section_18_check(
    "Every exception event has provenance",
    exception_events.empty
    or (
        exception_events[
            "source_record_id"
        ].notna().all()
        and exception_events[
            "source_path"
        ].notna().all()
    )
)

section_18_check(
    "Every consolidated exception has provenance",
    detected_exceptions.empty
    or (
        detected_exceptions[
            "source_record_id"
        ].notna().all()
        and detected_exceptions[
            "source_path"
        ].notna().all()
    )
)

section_18_check(
    "Category summary reconciles with events",
    int(
        exception_category_summary[
            "status_events"
        ].sum()
    )
    == len(exception_events)
)

section_18_check(
    "Source summary reconciles with records",
    int(
        source_exception_summary[
            "unique_exception_records"
        ].sum()
    )
    == len(detected_exceptions)
)

section_18_check(
    "All output files saved",
    all(
        path.exists()
        for path in output_paths.values()
    )
)

section_18_validation = pd.DataFrame(
    section_18_checks
)

validation_path = (
    VALIDATION_DIR /
    "section_18_validation.csv"
)

section_18_validation.to_csv(
    validation_path,
    index=False
)


# ------------------------------------------------------------
# 10. COMPLETION METADATA
# ------------------------------------------------------------

structural_passed = int(
    section_18_validation[
        "passed"
    ].sum()
)

structural_failed = int(
    (~section_18_validation[
        "passed"
    ]).sum()
)

explicit_failure_count = len(
    failure_summary
)

status_event_count = len(
    exception_events
)

unique_exception_count = len(
    detected_exceptions
)

execution_exception_count = int(
    detected_exceptions[
        "execution_or_validation_exception"
    ].sum()
)

missing_evidence_count = int(
    detected_exceptions[
        "missing_empirical_evidence"
    ].sum()
)

if structural_failed > 0:

    section_18_status = (
        "COMPLETED_WITH_VALIDATION_ERRORS"
    )

elif (
    explicit_failure_count > 0
    or execution_exception_count > 0
):

    section_18_status = (
        "COMPLETED_WITH_RECORDED_EXCEPTIONS"
    )

elif missing_evidence_count > 0:

    section_18_status = (
        "COMPLETED_WITH_MISSING_EVIDENCE"
    )

else:

    section_18_status = "COMPLETED"


section_18_summary = {
    "notebook": 20,
    "section": 18,
    "section_name": "Failure / Exception Summary",
    "source_artifacts_scanned": len(
        artifact_records
    ),
    "explicit_failure_records": (
        explicit_failure_count
    ),
    "detected_status_events": (
        status_event_count
    ),
    "unique_detected_exception_records": (
        unique_exception_count
    ),
    "execution_or_validation_exception_records": (
        execution_exception_count
    ),
    "missing_empirical_evidence_records": (
        missing_evidence_count
    ),
    "structural_validation_passed": (
        structural_passed
    ),
    "structural_validation_failed": (
        structural_failed
    ),
    "status": section_18_status,
    "output_files": {
        name: str(path)
        for name, path in output_paths.items()
    },
    "validation_file": str(
        validation_path
    )
}

summary_path = (
    METADATA_DIR /
    "section_18_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_18_summary,
        file,
        indent=2,
        default=safe_json
    )


# ------------------------------------------------------------
# 11. FINAL REPORT
# ------------------------------------------------------------

print()
print("Source artifacts scanned:", len(artifact_records))
print("Explicit failure records:", explicit_failure_count)
print("Detected status events:", status_event_count)
print("Unique detected exception records:", unique_exception_count)

print(
    "Execution / validation exception records:",
    execution_exception_count
)

print(
    "Missing empirical evidence records:",
    missing_evidence_count
)

print()
print("Exception category summary:")

if exception_category_summary.empty:

    print("No matching exception statuses found.")

else:

    print(
        exception_category_summary.to_string(
            index=False
        )
    )

print()
print("Structural validation passed:", structural_passed)
print("Structural validation failed:", structural_failed)
print("Section 18 status:", section_18_status)

print()
print("Saved:")

for path in output_paths.values():
    print(path)

print(validation_path)
print(summary_path)

assert structural_failed == 0, (
    "Section 18 structural validation failed."
)

NOTEBOOK 20 — SECTION 18: FAILURE / EXCEPTION SUMMARY

Source artifacts scanned: 50
Explicit failure records: 0
Detected status events: 465
Unique detected exception records: 465
Execution / validation exception records: 180
Missing empirical evidence records: 285

Exception category summary:
               exception_category  status_events  unique_source_records  source_artifacts
EXECUTION_OR_VALIDATION_EXCEPTION            180                    180                 2
       MISSING_EMPIRICAL_EVIDENCE            285                    285                 8

Structural validation passed: 12
Structural validation failed: 0
Section 18 status: COMPLETED_WITH_RECORDED_EXCEPTIONS

Saved:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/failure_exception_records.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/explicit_failure_artifact_inventory.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/detecte

In [43]:
# ============================================================
# NOTEBOOK 20 — SECTION 19
# ROBUSTNESS SUMMARY
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 80)
print("NOTEBOOK 20 — SECTION 19: ROBUSTNESS SUMMARY")
print("=" * 80)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

TABLES_DIR = Path(TABLES_DIR)
VALIDATION_DIR = Path(VALIDATION_DIR)
METADATA_DIR = Path(METADATA_DIR)

for directory in [
    TABLES_DIR,
    VALIDATION_DIR,
    METADATA_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

REQUIRED_COLUMNS = [
    "dataset",
    "model",
    "metric",
    "metric_group",
    "value"
]

REPETITION_CANDIDATES = [
    "repetition",
    "repetition_id",
    "run_id",
    "seed"
]

GROUP_COLUMNS = [
    "dataset",
    "model",
    "metric",
    "metric_group"
]

# ------------------------------------------------------------
# 2. LOAD AND INSPECT INPUT
# ------------------------------------------------------------

if isinstance(all_metric_results, pd.DataFrame):

    robustness_input = all_metric_results.copy()

else:

    robustness_input = pd.DataFrame()

input_rows = len(robustness_input)

available_columns = list(
    robustness_input.columns
)

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in available_columns
]

input_is_empty = robustness_input.empty

input_schema_valid = (
    len(missing_columns) == 0
)

print("Input records:", input_rows)
print("Available input columns:", available_columns)
print("Missing required columns:", missing_columns)

# ------------------------------------------------------------
# 3. IDENTIFY REPETITION COLUMN
# ------------------------------------------------------------

repetition_column = next(
    (
        column
        for column in REPETITION_CANDIDATES
        if column in robustness_input.columns
    ),
    None
)

# ------------------------------------------------------------
# 4. PREPARE EMPTY OUTPUT SCHEMAS
# ------------------------------------------------------------

robustness_columns = (
    GROUP_COLUMNS
    + [
        "valid_measurements",
        "repetitions_observed",
        "mean",
        "std",
        "minimum",
        "maximum",
        "range",
        "coefficient_of_variation",
        "robustness_status"
    ]
)

robustness_summary = pd.DataFrame(
    columns=robustness_columns
)

valid_metric_results = pd.DataFrame(
    columns=available_columns
)

invalid_metric_results = pd.DataFrame(
    columns=available_columns
)

# ------------------------------------------------------------
# 5. PROCESS AVAILABLE EMPIRICAL MEASUREMENTS
# ------------------------------------------------------------

if (
    not input_is_empty
    and input_schema_valid
):

    robustness_input["value"] = pd.to_numeric(
        robustness_input["value"],
        errors="coerce"
    )

    finite_mask = np.isfinite(
        robustness_input["value"].to_numpy(
            dtype=float
        )
    )

    valid_metric_results = (
        robustness_input.loc[
            finite_mask
        ].copy()
    )

    invalid_metric_results = (
        robustness_input.loc[
            ~finite_mask
        ].copy()
    )

    if repetition_column is not None:

        valid_metric_results["_repetition_id"] = (
            valid_metric_results[
                repetition_column
            ].astype("string")
        )

    # --------------------------------------------------------
    # 6. AGGREGATE ROBUSTNESS STATISTICS
    # --------------------------------------------------------

    robustness_records = []

    if not valid_metric_results.empty:

        group_columns = GROUP_COLUMNS.copy()

        if "variant" in valid_metric_results.columns:
            group_columns.insert(
                2,
                "variant"
            )

        for group_key, group in (
            valid_metric_results.groupby(
                group_columns,
                dropna=False,
                sort=True
            )
        ):

            if not isinstance(group_key, tuple):
                group_key = (group_key,)

            record = dict(
                zip(group_columns, group_key)
            )

            values = group["value"].astype(float)

            valid_count = len(values)

            if repetition_column is not None:

                repetitions = (
                    group["_repetition_id"]
                    .nunique(dropna=True)
                )

            else:

                repetitions = np.nan

            mean_value = values.mean()

            std_value = (
                values.std(ddof=1)
                if valid_count >= 2
                else np.nan
            )

            minimum_value = values.min()
            maximum_value = values.max()

            value_range = (
                maximum_value - minimum_value
            )

            if (
                pd.notna(std_value)
                and pd.notna(mean_value)
                and not np.isclose(
                    mean_value,
                    0
                )
            ):

                cv = (
                    abs(std_value)
                    / abs(mean_value)
                )

            else:

                cv = np.nan

            if repetition_column is None:

                status = (
                    "REPETITION_IDENTIFIER_UNAVAILABLE"
                )

            elif repetitions >= 5:

                status = (
                    "FIVE_OR_MORE_DISTINCT_REPETITIONS"
                )

            elif repetitions >= 2:

                status = (
                    "PARTIAL_REPEATED_RUN_EVIDENCE"
                )

            else:

                status = "SINGLE_RUN_ONLY"

            record.update({
                "valid_measurements": valid_count,
                "repetitions_observed": repetitions,
                "mean": mean_value,
                "std": std_value,
                "minimum": minimum_value,
                "maximum": maximum_value,
                "range": value_range,
                "coefficient_of_variation": cv,
                "robustness_status": status
            })

            robustness_records.append(record)

    robustness_summary = pd.DataFrame(
        robustness_records
    )

    if robustness_summary.empty:

        robustness_summary = pd.DataFrame(
            columns=robustness_columns
        )

# ------------------------------------------------------------
# 7. DETERMINE EMPIRICAL EVIDENCE STATUS
# ------------------------------------------------------------

valid_measurement_count = len(
    valid_metric_results
)

robustness_record_count = len(
    robustness_summary
)

if robustness_record_count > 0:

    evidence_status = (
        "EMPIRICAL_ROBUSTNESS_MEASUREMENTS_AVAILABLE"
    )

elif input_is_empty:

    evidence_status = (
        "MISSING_UPSTREAM_EMPIRICAL_MEASUREMENTS"
    )

elif not input_schema_valid:

    evidence_status = (
        "INPUT_SCHEMA_INCOMPLETE"
    )

else:

    evidence_status = (
        "NO_VALID_FINITE_METRIC_MEASUREMENTS"
    )

robustness_evidence_summary = pd.DataFrame([{
    "input_records": input_rows,
    "valid_measurements": valid_measurement_count,
    "invalid_or_nonfinite_measurements": len(
        invalid_metric_results
    ),
    "robustness_records": robustness_record_count,
    "repetition_identifier": (
        repetition_column
        if repetition_column is not None
        else "NOT_AVAILABLE"
    ),
    "input_schema_valid": input_schema_valid,
    "evidence_status": evidence_status
}])

# ------------------------------------------------------------
# 8. SAVE OUTPUT ARTIFACTS
# ------------------------------------------------------------

output_paths = {
    "robustness_summary":
        TABLES_DIR / "robustness_summary.csv",

    "valid_input_records":
        TABLES_DIR / "robustness_valid_input_records.csv",

    "invalid_input_records":
        TABLES_DIR / "robustness_invalid_input_records.csv",

    "evidence_summary":
        TABLES_DIR / "robustness_evidence_summary.csv"
}

robustness_summary.to_csv(
    output_paths["robustness_summary"],
    index=False
)

valid_metric_results.to_csv(
    output_paths["valid_input_records"],
    index=False
)

invalid_metric_results.to_csv(
    output_paths["invalid_input_records"],
    index=False
)

robustness_evidence_summary.to_csv(
    output_paths["evidence_summary"],
    index=False
)

# ------------------------------------------------------------
# 9. STRUCTURAL VALIDATION
# ------------------------------------------------------------

section_19_checks = []

def section_19_check(name, condition):

    section_19_checks.append({
        "check": name,
        "passed": bool(condition)
    })


section_19_check(
    "Input is a DataFrame",
    isinstance(robustness_input, pd.DataFrame)
)

section_19_check(
    "Missing input columns correctly identified",
    missing_columns == [
        column
        for column in REQUIRED_COLUMNS
        if column not in available_columns
    ]
)

section_19_check(
    "Robustness summary has expected output schema",
    set(robustness_columns).issubset(
        robustness_summary.columns
    )
)

section_19_check(
    "Valid measurements contain only finite values",
    valid_metric_results.empty
    or np.isfinite(
        valid_metric_results["value"].to_numpy(
            dtype=float
        )
    ).all()
)

section_19_check(
    "Valid and invalid input records reconcile",
    (
        len(valid_metric_results)
        + len(invalid_metric_results)
        == (
            input_rows
            if input_schema_valid and not input_is_empty
            else 0
        )
    )
)

section_19_check(
    "No robustness records without valid measurements",
    (
        robustness_summary.empty
        or robustness_summary[
            "valid_measurements"
        ].gt(0).all()
    )
)

section_19_check(
    "Variant dimension preserved when available",
    (
        "variant" not in available_columns
        or "variant" in robustness_summary.columns
        or robustness_summary.empty
    )
)

section_19_check(
    "Evidence status correctly identifies missing input",
    (
        not input_is_empty
        or evidence_status
        == "MISSING_UPSTREAM_EMPIRICAL_MEASUREMENTS"
    )
)

section_19_check(
    "Evidence summary contains one record",
    len(robustness_evidence_summary) == 1
)

section_19_check(
    "All robustness output files saved",
    all(
        path.exists()
        for path in output_paths.values()
    )
)

section_19_validation = pd.DataFrame(
    section_19_checks
)

validation_path = (
    VALIDATION_DIR /
    "section_19_validation.csv"
)

section_19_validation.to_csv(
    validation_path,
    index=False
)

# ------------------------------------------------------------
# 10. COMPLETION METADATA
# ------------------------------------------------------------

structural_passed = int(
    section_19_validation["passed"].sum()
)

structural_failed = int(
    (~section_19_validation["passed"]).sum()
)

if structural_failed > 0:

    section_19_status = (
        "COMPLETED_WITH_VALIDATION_ERRORS"
    )

elif robustness_record_count > 0:

    section_19_status = (
        "COMPLETED_WITH_EMPIRICAL_EVIDENCE"
    )

else:

    section_19_status = (
        "COMPLETED_WITH_MISSING_EVIDENCE"
    )

section_19_metadata = {
    "notebook": 20,
    "section": 19,
    "section_name": "Robustness Summary",
    "input_records": input_rows,
    "available_columns": available_columns,
    "missing_required_columns": missing_columns,
    "valid_measurements": valid_measurement_count,
    "invalid_measurements": len(
        invalid_metric_results
    ),
    "robustness_records": robustness_record_count,
    "repetition_identifier": repetition_column,
    "evidence_status": evidence_status,
    "structural_validation_passed": structural_passed,
    "structural_validation_failed": structural_failed,
    "status": section_19_status,
    "output_files": {
        name: str(path)
        for name, path in output_paths.items()
    }
}

metadata_path = (
    METADATA_DIR /
    "section_19_summary.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_19_metadata,
        file,
        indent=2,
        default=str
    )

# ------------------------------------------------------------
# 11. FINAL REPORT
# ------------------------------------------------------------

print()
print("Input records:", input_rows)
print("Valid measurements:", valid_measurement_count)
print(
    "Invalid/non-finite measurements:",
    len(invalid_metric_results)
)
print("Robustness records:", robustness_record_count)

print(
    "Repetition identifier:",
    repetition_column
    if repetition_column is not None
    else "NOT AVAILABLE"
)

print()
print("Robustness evidence summary:")

print(
    robustness_evidence_summary.to_string(
        index=False
    )
)

print()
print("Structural validation passed:", structural_passed)
print("Structural validation failed:", structural_failed)
print("Evidence status:", evidence_status)
print("Section 19 status:", section_19_status)

print()
print("Saved:")

for path in output_paths.values():
    print(path)

print(validation_path)
print(metadata_path)

assert structural_failed == 0, (
    "Section 19 structural validation failed. "
    "Review section_19_validation.csv."
)

NOTEBOOK 20 — SECTION 19: ROBUSTNESS SUMMARY
Input records: 0
Available input columns: []
Missing required columns: ['dataset', 'model', 'metric', 'metric_group', 'value']

Input records: 0
Valid measurements: 0
Invalid/non-finite measurements: 0
Robustness records: 0
Repetition identifier: NOT AVAILABLE

Robustness evidence summary:
 input_records  valid_measurements  invalid_or_nonfinite_measurements  robustness_records repetition_identifier  input_schema_valid                         evidence_status
             0                   0                                  0                   0         NOT_AVAILABLE               False MISSING_UPSTREAM_EMPIRICAL_MEASUREMENTS

Structural validation passed: 10
Structural validation failed: 0
Evidence status: MISSING_UPSTREAM_EMPIRICAL_MEASUREMENTS
Section 19 status: COMPLETED_WITH_MISSING_EVIDENCE

Saved:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/robustness_summary.csv
/content/drive/MyDrive/SPP_GAN_Researc

In [45]:
# ============================================================
# NOTEBOOK 20 — SECTION 20
# SAVE MASTER RESULTS
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 80)
print("NOTEBOOK 20 — SECTION 20: SAVE MASTER RESULTS")
print("=" * 80)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

RESULTS_DIR = Path(RESULTS_DIR)
TABLES_DIR = Path(TABLES_DIR)
VALIDATION_DIR = Path(VALIDATION_DIR)
METADATA_DIR = Path(METADATA_DIR)

for directory in [
    RESULTS_DIR,
    TABLES_DIR,
    VALIDATION_DIR,
    METADATA_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

MASTER_COLUMNS = [
    "dataset",
    "model",
    "repetition",
    "metric",
    "value"
]

EXPERIMENT_COLUMNS = [
    "dataset",
    "model",
    "repetition"
]

# ------------------------------------------------------------
# 2. LOAD MASTER INPUT
# ------------------------------------------------------------

if isinstance(all_metric_results, pd.DataFrame):

    master_input = all_metric_results.copy()

else:

    master_input = pd.DataFrame()

input_rows = len(master_input)

available_columns = list(
    master_input.columns
)

missing_columns = [
    column
    for column in MASTER_COLUMNS
    if column not in available_columns
]

print("Input measurement rows:", input_rows)
print("Available columns:", available_columns)
print("Missing required columns:", missing_columns)

# ------------------------------------------------------------
# 3. PREPARE MASTER RESULTS
# ------------------------------------------------------------

if (
    not master_input.empty
    and not missing_columns
):

    master_results = master_input.copy()

    master_results["repetition"] = pd.to_numeric(
        master_results["repetition"],
        errors="coerce"
    )

    master_results["value"] = pd.to_numeric(
        master_results["value"],
        errors="coerce"
    )

    # Remove invalid or non-finite metric values.
    finite_value_mask = np.isfinite(
        master_results["value"].to_numpy(
            dtype=float
        )
    )

    master_results = master_results.loc[
        finite_value_mask
    ].copy()

    master_results = master_results.dropna(
        subset=MASTER_COLUMNS
    )

    # Preserve available source and experimental metadata.
    sort_columns = [
        column
        for column in [
            "dataset",
            "model",
            "variant",
            "repetition",
            "metric"
        ]
        if column in master_results.columns
    ]

    master_results = (
        master_results
        .sort_values(sort_columns)
        .reset_index(drop=True)
    )

else:

    master_results = pd.DataFrame(
        columns=available_columns
        if available_columns
        else MASTER_COLUMNS
    )

# ------------------------------------------------------------
# 4. CREATE EXPERIMENT-LEVEL INDEX
# ------------------------------------------------------------

if not master_results.empty:

    experiment_group_columns = (
        EXPERIMENT_COLUMNS.copy()
    )

    # Keep baseline and ablation variants distinguishable.
    if "variant" in master_results.columns:
        experiment_group_columns.insert(
            2,
            "variant"
        )

    experiment_index = (
        master_results[
            experiment_group_columns
        ]
        .drop_duplicates()
        .sort_values(
            experiment_group_columns
        )
        .reset_index(drop=True)
    )

else:

    experiment_index = pd.DataFrame(
        columns=EXPERIMENT_COLUMNS
    )

# ------------------------------------------------------------
# 5. CREATE SOURCE ARTIFACT INVENTORY
# ------------------------------------------------------------

if isinstance(artifact_inventory, pd.DataFrame):

    final_source_inventory = (
        artifact_inventory.copy()
    )

else:

    final_source_inventory = pd.DataFrame()

# Do not reconstruct or invent source artifact information.
# Preserve the inventory already created by the source-loading
# section.

# ------------------------------------------------------------
# 6. SAVE MASTER RESULTS
# ------------------------------------------------------------

master_results_path = (
    RESULTS_DIR /
    "notebook_20_master_results.csv"
)

experiment_index_path = (
    TABLES_DIR /
    "master_experiment_index.csv"
)

source_inventory_path = (
    TABLES_DIR /
    "final_source_inventory.csv"
)

master_results.to_csv(
    master_results_path,
    index=False
)

experiment_index.to_csv(
    experiment_index_path,
    index=False
)

final_source_inventory.to_csv(
    source_inventory_path,
    index=False
)

# ------------------------------------------------------------
# 7. VALIDATE MASTER RESULTS
# ------------------------------------------------------------

section_20_checks = []

def section_20_check(name, condition):

    section_20_checks.append({
        "check": name,
        "passed": bool(condition)
    })


section_20_check(
    "Master input is a DataFrame",
    isinstance(master_input, pd.DataFrame)
)

section_20_check(
    "Master results are a DataFrame",
    isinstance(master_results, pd.DataFrame)
)

section_20_check(
    "Experiment index has required columns",
    set(EXPERIMENT_COLUMNS).issubset(
        experiment_index.columns
    )
)

section_20_check(
    "Master results contain no non-finite values",
    (
        "value" not in master_results.columns
        or master_results.empty
        or np.isfinite(
            pd.to_numeric(
                master_results["value"],
                errors="coerce"
            ).to_numpy(dtype=float)
        ).all()
    )
)

section_20_check(
    "Master results contain no missing experiment keys",
    (
        master_results.empty
        or not master_results[
            EXPERIMENT_COLUMNS
        ].isna().any().any()
    )
)

section_20_check(
    "Experiment index matches master results",
    (
        len(experiment_index) == 0
        if master_results.empty
        else len(experiment_index)
        == len(
            master_results[
                [
                    column
                    for column in EXPERIMENT_COLUMNS
                    + ["variant"]
                    if column in master_results.columns
                ]
            ].drop_duplicates()
        )
    )
)

section_20_check(
    "Master result file saved",
    master_results_path.exists()
)

section_20_check(
    "Experiment index file saved",
    experiment_index_path.exists()
)

section_20_check(
    "Source inventory file saved",
    source_inventory_path.exists()
)

section_20_validation = pd.DataFrame(
    section_20_checks
)

validation_path = (
    VALIDATION_DIR /
    "section_20_validation.csv"
)

section_20_validation.to_csv(
    validation_path,
    index=False
)

# ------------------------------------------------------------
# 8. DETERMINE EVIDENCE STATUS
# ------------------------------------------------------------

if not master_results.empty:

    evidence_status = (
        "MASTER_EMPIRICAL_MEASUREMENTS_AVAILABLE"
    )

elif input_rows == 0:

    evidence_status = (
        "MISSING_UPSTREAM_EMPIRICAL_MEASUREMENTS"
    )

elif missing_columns:

    evidence_status = (
        "INPUT_SCHEMA_INCOMPLETE"
    )

else:

    evidence_status = (
        "NO_VALID_MASTER_MEASUREMENTS"
    )

# ------------------------------------------------------------
# 9. COMPLETION SUMMARY
# ------------------------------------------------------------

structural_passed = int(
    section_20_validation["passed"].sum()
)

structural_failed = int(
    (~section_20_validation["passed"]).sum()
)

if structural_failed > 0:

    section_20_status = (
        "COMPLETED_WITH_VALIDATION_ERRORS"
    )

elif master_results.empty:

    section_20_status = (
        "COMPLETED_WITH_MISSING_EVIDENCE"
    )

else:

    section_20_status = "COMPLETED"

section_20_summary = {
    "notebook": 20,
    "section": 20,
    "section_name": "Save Master Results",
    "input_measurement_rows": input_rows,
    "missing_required_columns": missing_columns,
    "master_measurement_rows": len(master_results),
    "unique_experiment_keys": len(experiment_index),
    "source_inventory_records": len(
        final_source_inventory
    ),
    "evidence_status": evidence_status,
    "structural_validation_passed": structural_passed,
    "structural_validation_failed": structural_failed,
    "status": section_20_status,
    "output_files": {
        "master_results": str(master_results_path),
        "experiment_index": str(experiment_index_path),
        "source_inventory": str(source_inventory_path),
        "validation": str(validation_path)
    }
}

metadata_path = (
    METADATA_DIR /
    "section_20_summary.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_20_summary,
        file,
        indent=2,
        default=str
    )

# ------------------------------------------------------------
# 10. FINAL REPORT
# ------------------------------------------------------------

print()
print("Master measurement rows:", len(master_results))
print("Unique experiment keys:", len(experiment_index))
print(
    "Source inventory records:",
    len(final_source_inventory)
)

print()
print("Evidence status:", evidence_status)
print("Structural validation passed:", structural_passed)
print("Structural validation failed:", structural_failed)
print("Section 20 status:", section_20_status)

print()
print("Saved:")
print(master_results_path)
print(experiment_index_path)
print(source_inventory_path)
print(validation_path)
print(metadata_path)

assert structural_failed == 0, (
    "Section 20 validation failed. "
    "Review section_20_validation.csv."
)

NOTEBOOK 20 — SECTION 20: SAVE MASTER RESULTS
Input measurement rows: 0
Available columns: []
Missing required columns: ['dataset', 'model', 'repetition', 'metric', 'value']

Master measurement rows: 0
Unique experiment keys: 0
Source inventory records: 50

Evidence status: MISSING_UPSTREAM_EMPIRICAL_MEASUREMENTS
Structural validation passed: 9
Structural validation failed: 0
Section 20 status: COMPLETED_WITH_MISSING_EVIDENCE

Saved:
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/results/notebook_20_master_results.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/master_experiment_index.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/final_source_inventory.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/validation/section_20_validation.csv
/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/metadata/section_20_summary.json


In [47]:
# ============================================================
# NOTEBOOK 20 — SECTION 21
# FINAL EVIDENCE VERIFICATION
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 80)
print("NOTEBOOK 20 — SECTION 21: FINAL EVIDENCE VERIFICATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(PROJECT_ROOT)

verification_output_dir = Path(VALIDATION_DIR)
verification_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

master_results_path = Path(master_results_path)

experiment_index_path = (
    TABLES_DIR / "master_experiment_index.csv"
)

source_inventory_path = (
    TABLES_DIR / "final_source_inventory.csv"
)

evidence_summary_path = (
    TABLES_DIR / "overall_evidence_summary.csv"
)

verification_path = (
    verification_output_dir
    / "section_21_final_evidence_verification.csv"
)

verification_metadata_path = (
    METADATA_DIR
    / "section_21_summary.json"
)

# ------------------------------------------------------------
# 2. VERIFICATION HELPER
# ------------------------------------------------------------

verification_records = []

def add_verification(
    name,
    condition,
    detail="",
    applicable=True
):

    verification_records.append({
        "check": name,
        "passed": bool(condition),
        "applicable": bool(applicable),
        "detail": str(detail)
    })


# ------------------------------------------------------------
# 3. PROJECT AND DIRECTORY VERIFICATION
# ------------------------------------------------------------

required_directories = [
    RESULTS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    METADATA_DIR,
    VALIDATION_DIR,
    MANIFESTS_DIR
]

add_verification(
    "project_root_exists",
    PROJECT_ROOT.is_dir(),
    str(PROJECT_ROOT)
)

missing_directories = [
    str(path)
    for path in required_directories
    if not Path(path).is_dir()
]

add_verification(
    "notebook_output_directories_exist",
    len(missing_directories) == 0,
    (
        "All required directories exist."
        if not missing_directories
        else "; ".join(missing_directories)
    )
)

# ------------------------------------------------------------
# 4. MASTER ARTIFACT VERIFICATION
# ------------------------------------------------------------

add_verification(
    "master_results_saved",
    master_results_path.is_file(),
    str(master_results_path)
)

add_verification(
    "experiment_index_saved",
    experiment_index_path.is_file(),
    str(experiment_index_path)
)

add_verification(
    "source_inventory_saved",
    source_inventory_path.is_file(),
    str(source_inventory_path)
)

# ------------------------------------------------------------
# 5. LOAD SAVED ARTIFACTS
# ------------------------------------------------------------

try:

    saved_master_results = pd.read_csv(
        master_results_path
    )

    master_read_success = True

except Exception as exc:

    saved_master_results = pd.DataFrame()

    master_read_success = False

    master_read_error = str(exc)


try:

    saved_experiment_index = pd.read_csv(
        experiment_index_path
    )

    experiment_index_read_success = True

except Exception as exc:

    saved_experiment_index = pd.DataFrame()

    experiment_index_read_success = False

    experiment_index_read_error = str(exc)


try:

    saved_source_inventory = pd.read_csv(
        source_inventory_path
    )

    source_inventory_read_success = True

except Exception as exc:

    saved_source_inventory = pd.DataFrame()

    source_inventory_read_success = False

    source_inventory_read_error = str(exc)


add_verification(
    "master_results_readable",
    master_read_success,
    (
        "Master results loaded successfully."
        if master_read_success
        else master_read_error
    )
)

add_verification(
    "experiment_index_readable",
    experiment_index_read_success,
    (
        "Experiment index loaded successfully."
        if experiment_index_read_success
        else experiment_index_read_error
    )
)

add_verification(
    "source_inventory_readable",
    source_inventory_read_success,
    (
        "Source inventory loaded successfully."
        if source_inventory_read_success
        else source_inventory_read_error
    )
)

# ------------------------------------------------------------
# 6. MASTER MEASUREMENT VALIDATION
# ------------------------------------------------------------

MASTER_REQUIRED_COLUMNS = [
    "dataset",
    "model",
    "repetition",
    "metric",
    "value"
]

master_has_schema = (
    set(MASTER_REQUIRED_COLUMNS).issubset(
        saved_master_results.columns
    )
)

add_verification(
    "master_results_schema_valid",
    master_has_schema,
    (
        "Required measurement columns are present."
        if master_has_schema
        else (
            "Missing columns: "
            + ", ".join(
                column
                for column in MASTER_REQUIRED_COLUMNS
                if column not in saved_master_results.columns
            )
        )
    )
)

if master_read_success and master_has_schema:

    if not saved_master_results.empty:

        numeric_values = pd.to_numeric(
            saved_master_results["value"],
            errors="coerce"
        )

        finite_values = np.isfinite(
            numeric_values.to_numpy(dtype=float)
        ).all()

        add_verification(
            "master_metric_values_finite",
            finite_values,
            "All saved metric values must be finite."
        )

        keys_present = (
            saved_master_results[
                MASTER_REQUIRED_COLUMNS
            ].notna().all().all()
        )

        add_verification(
            "master_keys_present",
            keys_present,
            "All required experiment and metric keys must exist."
        )

        if "source_path" in saved_master_results.columns:

            source_paths_preserved = (
                saved_master_results[
                    "source_path"
                ].notna().all()
            )

        else:

            source_paths_preserved = False

        add_verification(
            "source_paths_preserved",
            source_paths_preserved,
            "Every master measurement must retain source provenance."
        )

    else:

        add_verification(
            "master_metric_values_finite",
            True,
            "Not applicable: no master measurements exist.",
            applicable=False
        )

        add_verification(
            "master_keys_present",
            True,
            "Not applicable: no master measurements exist.",
            applicable=False
        )

        add_verification(
            "source_paths_preserved",
            True,
            "Not applicable: no master measurements exist.",
            applicable=False
        )

else:

    add_verification(
        "master_metric_values_finite",
        False,
        "Cannot verify metric values because the master schema is invalid."
    )

    add_verification(
        "master_keys_present",
        False,
        "Cannot verify experiment keys because the master schema is invalid."
    )

    add_verification(
        "source_paths_preserved",
        False,
        "Cannot verify provenance because the master schema is invalid."
    )

# ------------------------------------------------------------
# 7. EXPERIMENT INDEX VALIDATION
# ------------------------------------------------------------

INDEX_REQUIRED_COLUMNS = [
    "dataset",
    "model",
    "repetition"
]

index_schema_valid = (
    set(INDEX_REQUIRED_COLUMNS).issubset(
        saved_experiment_index.columns
    )
)

add_verification(
    "experiment_index_schema_valid",
    index_schema_valid,
    (
        "Required experiment index columns are present."
        if index_schema_valid
        else "Required experiment index columns are missing."
    )
)

if (
    master_read_success
    and experiment_index_read_success
    and master_has_schema
    and index_schema_valid
):

    if not saved_master_results.empty:

        expected_index = (
            saved_master_results[
                INDEX_REQUIRED_COLUMNS
            ]
            .drop_duplicates()
        )

        if "variant" in saved_master_results.columns:

            expected_index = (
                saved_master_results[
                    INDEX_REQUIRED_COLUMNS + ["variant"]
                ]
                .drop_duplicates()
            )

        index_matches = (
            len(expected_index)
            == len(saved_experiment_index)
        )

    else:

        index_matches = (
            saved_experiment_index.empty
        )

    add_verification(
        "experiment_index_matches_master",
        index_matches,
        "Saved experiment index reconciles with master measurements."
    )

else:

    add_verification(
        "experiment_index_matches_master",
        False,
        "Unable to reconcile master results and experiment index."
    )

# ------------------------------------------------------------
# 8. SOURCE INVENTORY VALIDATION
# ------------------------------------------------------------

inventory_count_matches = (
    len(saved_source_inventory)
    == len(artifact_inventory)
    if (
        isinstance(artifact_inventory, pd.DataFrame)
        and source_inventory_read_success
    )
    else False
)

add_verification(
    "source_inventory_record_count_matches",
    inventory_count_matches,
    (
        f"Saved records: {len(saved_source_inventory)}; "
        f"Current inventory records: "
        f"{len(artifact_inventory) if isinstance(artifact_inventory, pd.DataFrame) else 'UNAVAILABLE'}"
    )
)

# ------------------------------------------------------------
# 9. FORMAL DP EVIDENCE VERIFICATION
# ------------------------------------------------------------

DP_REQUIRED_COLUMNS = [
    "verified_dp_evidence",
    "achieved_epsilon",
    "achieved_delta"
]

if isinstance(dp_results, pd.DataFrame):

    if dp_results.empty:

        dp_separation_valid = True

        dp_detail = (
            "No formal DP records exist. "
            "No formal privacy guarantee is claimed."
        )

    else:

        dp_schema_valid = (
            set(DP_REQUIRED_COLUMNS).issubset(
                dp_results.columns
            )
        )

        if dp_schema_valid:

            verified_mask = (
                dp_results[
                    "verified_dp_evidence"
                ]
                .fillna(False)
                .astype(bool)
            )

            verified_rows = dp_results.loc[
                verified_mask
            ]

            dp_values_valid = True

            if not verified_rows.empty:

                epsilon = pd.to_numeric(
                    verified_rows["achieved_epsilon"],
                    errors="coerce"
                )

                delta = pd.to_numeric(
                    verified_rows["achieved_delta"],
                    errors="coerce"
                )

                dp_values_valid = bool(
                    np.isfinite(epsilon).all()
                    and np.isfinite(delta).all()
                    and (epsilon > 0).all()
                    and (delta > 0).all()
                    and (delta < 1).all()
                )

            dp_separation_valid = dp_values_valid

            dp_detail = (
                f"DP schema valid; verified records: "
                f"{len(verified_rows)}; "
                f"numeric privacy parameters valid: "
                f"{dp_values_valid}."
            )

        else:

            dp_separation_valid = False

            dp_detail = (
                "Formal DP records exist but required "
                "verification columns are missing."
            )

else:

    dp_separation_valid = False

    dp_detail = "DP results are not a valid DataFrame."


add_verification(
    "formal_dp_evidence_separated",
    dp_separation_valid,
    dp_detail
)

# ------------------------------------------------------------
# 10. SIGNIFICANCE EVIDENCE VERIFICATION
# ------------------------------------------------------------

if isinstance(significance_results, pd.DataFrame):

    if significance_results.empty:

        significance_separation_valid = True

        significance_detail = (
            "No significance records exist. "
            "No statistical significance is claimed."
        )

    else:

        normalized_columns = {
            str(column).strip().lower()
            for column in significance_results.columns
        }

        statistic_columns = {
            "test_statistic",
            "statistic"
        }

        p_value_columns = {
            "p_value",
            "adjusted_p_value",
            "corrected_p_value"
        }

        has_statistic = bool(
            normalized_columns.intersection(
                statistic_columns
            )
        )

        has_p_value = bool(
            normalized_columns.intersection(
                p_value_columns
            )
        )

        significance_separation_valid = (
            has_statistic and has_p_value
        )

        significance_detail = (
            f"Test statistic available: {has_statistic}; "
            f"p-value available: {has_p_value}. "
            "Column presence alone does not prove executed tests."
        )

else:

    significance_separation_valid = False

    significance_detail = (
        "Significance results are not a valid DataFrame."
    )


add_verification(
    "significance_evidence_not_inferred",
    significance_separation_valid,
    significance_detail
)

# ------------------------------------------------------------
# 11. MISSING EVIDENCE VERIFICATION
# ------------------------------------------------------------

if evidence_summary_path.is_file():

    try:

        saved_evidence_summary = pd.read_csv(
            evidence_summary_path
        )

        evidence_summary_readable = True

        required_evidence_columns = {
            "evidence_category",
            "valid_measurements",
            "evidence_status",
            "evidence_available"
        }

        evidence_schema_valid = (
            required_evidence_columns.issubset(
                saved_evidence_summary.columns
            )
        )

        if evidence_schema_valid:

            missing_evidence_rows = (
                saved_evidence_summary[
                    "valid_measurements"
                ]
                .fillna(0)
                .eq(0)
            )

            missing_evidence_consistent = (
                (
                    saved_evidence_summary.loc[
                        missing_evidence_rows,
                        "evidence_available"
                    ]
                    .astype(bool)
                    .eq(False)
                ).all()
            )

        else:

            missing_evidence_consistent = False

    except Exception:

        evidence_summary_readable = False
        evidence_schema_valid = False
        missing_evidence_consistent = False

else:

    evidence_summary_readable = False
    evidence_schema_valid = False
    missing_evidence_consistent = False


add_verification(
    "missing_evidence_not_imputed",
    (
        evidence_summary_readable
        and evidence_schema_valid
        and missing_evidence_consistent
    ),
    (
        "Saved evidence summary preserves zero-measurement "
        "categories as unavailable."
        if missing_evidence_consistent
        else "Evidence summary is missing, invalid, or inconsistent."
    )
)

# ------------------------------------------------------------
# 12. SAVE VERIFICATION RESULTS
# ------------------------------------------------------------

verification_df = pd.DataFrame(
    verification_records
)

verification_df.to_csv(
    verification_path,
    index=False
)

verification_passed = int(
    verification_df["passed"].sum()
)

verification_failed = int(
    (~verification_df["passed"]).sum()
)

verification_not_applicable = int(
    (~verification_df["applicable"]).sum()
)

verification_applicable = int(
    verification_df["applicable"].sum()
)

verification_status = (
    "PASS"
    if verification_failed == 0
    else "FAIL"
)

# ------------------------------------------------------------
# 13. SAVE VERIFICATION METADATA
# ------------------------------------------------------------

section_21_summary = {
    "notebook": 20,
    "section": 21,
    "section_name": "Final Evidence Verification",
    "total_checks": len(verification_df),
    "passed": verification_passed,
    "failed": verification_failed,
    "applicable_checks": verification_applicable,
    "not_applicable_checks": verification_not_applicable,
    "master_measurement_rows": len(
        saved_master_results
    ),
    "source_inventory_records": len(
        saved_source_inventory
    ),
    "verification_status": verification_status,
    "verification_file": str(
        verification_path
    )
}

with open(
    verification_metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        section_21_summary,
        file,
        indent=2,
        default=str
    )

# ------------------------------------------------------------
# 14. FINAL REPORT
# ------------------------------------------------------------

print()
print("Verification checks:", len(verification_df))
print("Passed:", verification_passed)
print("Failed:", verification_failed)
print("Not applicable:", verification_not_applicable)
print("Applicable checks:", verification_applicable)

print()
print("Detailed verification:")

print(
    verification_df[
        [
            "check",
            "passed",
            "applicable",
            "detail"
        ]
    ].to_string(index=False)
)

print()
print("Verification status:", verification_status)

print()
print("Saved:")
print(verification_path)
print(verification_metadata_path)

assert verification_failed == 0, (
    "Final evidence verification failed. "
    "Review section_21_final_evidence_verification.csv."
)

NOTEBOOK 20 — SECTION 21: FINAL EVIDENCE VERIFICATION

Verification checks: 18
Passed: 18
Failed: 0
Not applicable: 3
Applicable checks: 15

Detailed verification:
                                check  passed  applicable                                                                                                        detail
                  project_root_exists    True        True                                                                       /content/drive/MyDrive/SPP_GAN_Research
    notebook_output_directories_exist    True        True                                                                               All required directories exist.
                 master_results_saved    True        True  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/results/notebook_20_master_results.csv
               experiment_index_saved    True        True      /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_20/tables/master_experiment_index.

In [49]:
# ============================================================
# NOTEBOOK 20 — SECTION 22
# COMPLETION SUMMARY
# ============================================================

import os
import gc
import json
import hashlib
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone

print("=" * 80)
print("NOTEBOOK 20 — SECTION 22: COMPLETION SUMMARY")
print("=" * 80)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

# Methodology-defined experiment design:
# 3 datasets x 4 ablation variants x 5 repetitions = 60.

EXPECTED_EXPERIMENTS = 60

NOTEBOOK_ROOT = Path(NOTEBOOK_ROOT)
MANIFESTS_DIR = Path(MANIFESTS_DIR)
METADATA_DIR = Path(METADATA_DIR)

MANIFESTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

completion_path = (
    METADATA_DIR /
    "notebook_20_completion_summary.json"
)

integrity_path = (
    MANIFESTS_DIR /
    "notebook_20_integrity_manifest.csv"
)

# ------------------------------------------------------------
# 2. EXECUTION COMPLETENESS
# ------------------------------------------------------------

required_outputs = [
    master_results_path,

    TABLES_DIR / "master_experiment_index.csv",
    TABLES_DIR / "final_source_inventory.csv",

    TABLES_DIR / "model_level_comparison.csv",
    TABLES_DIR / "dataset_level_comparison.csv",
    TABLES_DIR / "metric_level_comparison.csv",

    TABLES_DIR / "overall_performance_ranking.csv",
    TABLES_DIR / "privacy_utility_ranking.csv",

    TABLES_DIR / "overall_evidence_summary.csv",

    TABLES_DIR / "failure_exception_records.csv",
    TABLES_DIR / "robustness_summary.csv",

    TABLES_DIR / "detected_incomplete_or_failed_records.csv",
    TABLES_DIR / "exception_category_summary.csv",

    TABLES_DIR / "robustness_evidence_summary.csv",

    VALIDATION_DIR / "section_17_validation.csv",
    VALIDATION_DIR / "section_18_validation.csv",
    VALIDATION_DIR / "section_19_validation.csv",
    VALIDATION_DIR / "section_20_validation.csv",

    verification_path
]

missing_required_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).is_file()
]

all_required_outputs_saved = (
    len(missing_required_outputs) == 0
)

# Verify Section 21 report directly.
verification_file_valid = False

if Path(verification_path).is_file():

    try:

        saved_verification = pd.read_csv(
            verification_path
        )

        verification_file_valid = (
            not saved_verification.empty
            and {"check", "passed"}.issubset(
                saved_verification.columns
            )
            and saved_verification[
                "passed"
            ].astype(bool).all()
        )

    except Exception:

        verification_file_valid = False

execution_complete = bool(
    all_required_outputs_saved
    and verification_file_valid
    and verification_failed == 0
)

# ------------------------------------------------------------
# 3. RESEARCH EVIDENCE COMPLETENESS
# ------------------------------------------------------------

evidence_categories = [
    "Fidelity",
    "ML utility",
    "Empirical privacy",
    "Formal DP",
    "Ablation",
    "Trade-off",
    "Significance",
    "Computational"
]

evidence_summary_path = (
    TABLES_DIR /
    "overall_evidence_summary.csv"
)

if evidence_summary_path.is_file():

    evidence_summary = pd.read_csv(
        evidence_summary_path
    )

else:

    evidence_summary = pd.DataFrame()

if not evidence_summary.empty:

    if "evidence_available" in evidence_summary.columns:

        available_categories = int(
            evidence_summary[
                "evidence_available"
            ]
            .fillna(False)
            .astype(bool)
            .sum()
        )

    elif "valid_measurements" in evidence_summary.columns:

        available_categories = int(
            (
                pd.to_numeric(
                    evidence_summary[
                        "valid_measurements"
                    ],
                    errors="coerce"
                ).fillna(0) > 0
            ).sum()
        )

    else:

        available_categories = 0

else:

    available_categories = 0

missing_categories = max(
    0,
    len(evidence_categories) - available_categories
)

research_evidence_complete = bool(
    available_categories == len(evidence_categories)
)

# ------------------------------------------------------------
# 4. FINAL STATUS
# ------------------------------------------------------------

if not execution_complete:

    FINAL_STATUS = "INCOMPLETE"

elif not research_evidence_complete:

    FINAL_STATUS = (
        "COMPLETED_WITH_MISSING_EVIDENCE"
    )

else:

    FINAL_STATUS = "COMPLETED"

# ------------------------------------------------------------
# 5. SAFE SUMMARY COUNTS
# ------------------------------------------------------------

def safe_length(value):

    if isinstance(value, pd.DataFrame):
        return len(value)

    if isinstance(value, (list, tuple, dict)):
        return len(value)

    return 0


def safe_unique_values(frame, column):

    if (
        isinstance(frame, pd.DataFrame)
        and column in frame.columns
    ):

        return sorted(
            frame[column]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

    return []


source_notebooks = safe_unique_values(
    artifact_inventory,
    "source_notebook"
)

# ------------------------------------------------------------
# 6. COMPLETION SUMMARY
# ------------------------------------------------------------

completion_summary = {

    "notebook": NOTEBOOK_ID,

    "title": NOTEBOOK_NAME,

    "status": FINAL_STATUS,

    "execution_complete": execution_complete,

    "research_evidence_complete": (
        research_evidence_complete
    ),

    "source_artifacts_discovered": len(
        artifact_records
    ),

    "source_notebooks": source_notebooks,

    "master_measurement_rows": len(
        master_results
    ),

    "unique_experiment_keys": len(
        experiment_index
    ),

    "expected_ablation_experiments": (
        EXPECTED_EXPERIMENTS
    ),

    "fidelity_measurements": safe_length(
        fidelity_results
    ),

    "utility_measurements": safe_length(
        utility_results
    ),

    "empirical_privacy_measurements": safe_length(
        privacy_results
    ),

    "formal_dp_records": safe_length(
        dp_results
    ),

    "verified_formal_dp_records": int(
        verified_dp_count
    ),

    "ablation_measurements": safe_length(
        ablation_results
    ),

    "tradeoff_records": safe_length(
        tradeoff_results
    ),

    "significance_records": safe_length(
        significance_results
    ),

    "computational_measurements": safe_length(
        computational_results
    ),

    "failure_records": safe_length(
        failure_summary
    ),

    "detected_incomplete_records": safe_length(
        detected_exceptions
    ),

    "robustness_records": safe_length(
        robustness_summary
    ),

    "evidence_categories_total": len(
        evidence_categories
    ),

    "evidence_categories_available": (
        available_categories
    ),

    "evidence_categories_missing": (
        missing_categories
    ),

    "verification_passed": int(
        verification_passed
    ),

    "verification_failed": int(
        verification_failed
    ),

    "verification_file_valid": (
        verification_file_valid
    ),

    "required_outputs_saved": (
        all_required_outputs_saved
    ),

    "missing_required_outputs": (
        missing_required_outputs
    ),

    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

# ------------------------------------------------------------
# 7. SAVE INITIAL COMPLETION SUMMARY
# ------------------------------------------------------------

with open(
    completion_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        completion_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 8. SHA-256 INTEGRITY FUNCTION
# ------------------------------------------------------------

def calculate_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()

# ------------------------------------------------------------
# 9. GENERATE ARTIFACT INTEGRITY MANIFEST
# ------------------------------------------------------------

integrity_records = []

excluded_paths = {
    integrity_path.resolve(),
    completion_path.resolve()
}

for path in sorted(
    NOTEBOOK_ROOT.rglob("*")
):

    if not path.is_file():
        continue

    resolved_path = path.resolve()

    if resolved_path in excluded_paths:
        continue

    try:

        integrity_records.append({

            "relative_path": str(
                path.relative_to(
                    NOTEBOOK_ROOT
                )
            ),

            "size_bytes": path.stat().st_size,

            "sha256": calculate_sha256(path)

        })

    except Exception as exc:

        print(
            "WARNING: Could not hash:",
            path,
            str(exc)
        )

integrity_df = pd.DataFrame(
    integrity_records,
    columns=[
        "relative_path",
        "size_bytes",
        "sha256"
    ]
)

integrity_df.to_csv(
    integrity_path,
    index=False
)

# ------------------------------------------------------------
# 10. VERIFY INTEGRITY MANIFEST
# ------------------------------------------------------------

integrity_manifest_saved = (
    integrity_path.is_file()
)

integrity_manifest_valid = (
    integrity_manifest_saved
    and len(integrity_df) > 0
    and integrity_df[
        "sha256"
    ].notna().all()
    and integrity_df[
        "sha256"
    ].str.fullmatch(r"[a-f0-9]{64}").all()
)

integrity_manifest_hash = (
    calculate_sha256(integrity_path)
    if integrity_manifest_saved
    else None
)

completion_summary["integrity_manifest"] = str(
    integrity_path
)

completion_summary["integrity_manifest_files"] = (
    len(integrity_df)
)

completion_summary["integrity_manifest_saved"] = (
    integrity_manifest_saved
)

completion_summary["integrity_manifest_valid"] = (
    integrity_manifest_valid
)

completion_summary["integrity_manifest_sha256"] = (
    integrity_manifest_hash
)

# ------------------------------------------------------------
# 11. UPDATE COMPLETION STATUS
# ------------------------------------------------------------

completion_summary["integrity_manifest_valid"] = (
    integrity_manifest_valid
)

if not integrity_manifest_valid:

    FINAL_STATUS = "INCOMPLETE"

completion_summary["status"] = FINAL_STATUS

completion_summary["execution_complete"] = bool(
    execution_complete
    and integrity_manifest_valid
)

# ------------------------------------------------------------
# 12. SAVE FINAL COMPLETION SUMMARY
# ------------------------------------------------------------

with open(
    completion_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        completion_summary,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# 13. FINAL REPORT
# ------------------------------------------------------------

gc.collect()

print()
print("=" * 80)
print("NOTEBOOK 20 — FINAL COMPARATIVE ANALYSIS")
print("=" * 80)

print("Final status:", FINAL_STATUS)

print(
    "Execution complete:",
    completion_summary["execution_complete"]
)

print(
    "Research evidence complete:",
    research_evidence_complete
)

print("-" * 80)

print(
    "Source artifacts:",
    len(artifact_records)
)

print(
    "Master measurement rows:",
    len(master_results)
)

print(
    "Unique experiment keys:",
    len(experiment_index)
)

print(
    "Expected ablation experiments:",
    EXPECTED_EXPERIMENTS
)

print("-" * 80)

print(
    "Fidelity measurements:",
    safe_length(fidelity_results)
)

print(
    "Utility measurements:",
    safe_length(utility_results)
)

print(
    "Empirical privacy measurements:",
    safe_length(privacy_results)
)

print(
    "Verified formal DP records:",
    verified_dp_count
)

print(
    "Ablation measurements:",
    safe_length(ablation_results)
)

print(
    "Trade-off records:",
    safe_length(tradeoff_results)
)

print(
    "Significance records:",
    safe_length(significance_results)
)

print(
    "Computational measurements:",
    safe_length(computational_results)
)

print("-" * 80)

print(
    "Evidence categories available:",
    available_categories
)

print(
    "Evidence categories missing:",
    missing_categories
)

print(
    "Verification passed:",
    verification_passed
)

print(
    "Verification failed:",
    verification_failed
)

print(
    "Required outputs saved:",
    all_required_outputs_saved
)

print(
    "Integrity manifest files:",
    len(integrity_df)
)

print(
    "Integrity manifest valid:",
    integrity_manifest_valid
)

print("-" * 80)

print("Master results:", master_results_path)

print("Completion summary:", completion_path)

print("Integrity manifest:", integrity_path)

print("Verification report:", verification_path)

print("=" * 80)

assert FINAL_STATUS != "INCOMPLETE", (
    "Notebook 20 execution or artifact integrity verification failed. "
    "Review the completion summary and missing outputs."
)

NOTEBOOK 20 — SECTION 22: COMPLETION SUMMARY

NOTEBOOK 20 — FINAL COMPARATIVE ANALYSIS
Final status: COMPLETED_WITH_MISSING_EVIDENCE
Execution complete: True
Research evidence complete: False
--------------------------------------------------------------------------------
Source artifacts: 50
Master measurement rows: 0
Unique experiment keys: 0
Expected ablation experiments: 60
--------------------------------------------------------------------------------
Fidelity measurements: 0
Utility measurements: 0
Empirical privacy measurements: 0
Verified formal DP records: 0
Ablation measurements: 0
Trade-off records: 22
Significance records: 309
Computational measurements: 0
--------------------------------------------------------------------------------
Evidence categories available: 0
Evidence categories missing: 8
Verification passed: 18
Verification failed: 0
Required outputs saved: True
Integrity manifest files: 99
Integrity manifest valid: True
-----------------------------------------